# PSID-SHELF Topic Splitter

**Splits the harmonized PSID-SHELF dataset into 30 topic-specific CSV files.**

This notebook is the bridge between the SHELF .dta file you download from OpenICPSR and the topic CSVs that the [psid-shelf-app](https://psid-shelf-app.netlify.app) expects.

---

## What this notebook does

It reads `PSIDSHELF_1968_2021_WIDE.dta` (the wide-format harmonized file from the SHELF Beta Release V2) and produces 30 CSV files — one per topic — each containing the same 84,121 rows but only the columns belonging to that topic, plus three ID columns (`ID`, `LINEAGE`, `PNUM`) for cross-topic linking.

The topic-to-variable mapping is the official one used by the PSID-SHELF Variable Catalog and the live psid-shelf-app — no judgment calls.

## What you need before running

1. A SHELF dataset download from OpenICPSR: <https://doi.org/10.3886/E194322>
   You'll need a free account. Download the V2.0 zip (~270 MB).
2. After downloading, unzip twice — first the outer ICPSR wrapper, then the inner `PSIDSHELF_1968_2021_WIDE_2.4_GB.zip`. You'll end up with `PSIDSHELF_1968_2021_WIDE.dta` (~2.4 GB).
3. Python with pandas. That's it.

## Output

30 CSV files written to your output directory, one per topic. After this runs, point a local copy of `index.html` at the output directory and you have a fully working SHELF Variable Finder.


---

## Configuration

Edit the two paths below. Everything else runs as-is.

In [ ]:
# ============================================================
# CONFIGURATION — Edit these two paths, then run everything
# ============================================================

# Path to the unzipped SHELF .dta file
DTA_FILE = "PSIDSHELF_1968_2021_WIDE.dta"

# Directory where the 30 topic CSVs will be written.
# Will be created if it doesn't exist.
OUTPUT_DIR = "shelf_csvs"


---

## Step 1: Imports and Sanity Check

In [ ]:
import pandas as pd
import os
import sys

if not os.path.exists(DTA_FILE):
    print(f"\u26d4 File not found: {DTA_FILE}")
    print(f"   Make sure you've downloaded SHELF from OpenICPSR (https://doi.org/10.3886/E194322)")
    print(f"   and unzipped it twice to get the .dta file.")
    sys.exit(1)

size_gb = os.path.getsize(DTA_FILE) / (1024**3)
print(f"\u2713 Found: {DTA_FILE} ({size_gb:.2f} GB)")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\u2713 Output directory ready: {OUTPUT_DIR}/")


---

## Step 2: The Topic Mapping

This dictionary maps each of the 30 topics to its column list. The mapping comes directly from the PSID-SHELF dataset structure — every non-ID variable in SHELF appears in exactly one topic.

The three ID columns (`ID`, `LINEAGE`, `PNUM`) appear in every topic so each CSV is self-sufficient and can be re-joined to others as needed.

In [ ]:
TOPIC_COLUMNS = {
  "covid": [
    "ID",
    "LINEAGE",
    "PNUM",
    "COVID_VACC_2021",
    "COVID_POSI_2021",
    "COVID_MEDI_2021",
    "COVID_TEST_2021",
    "COVID_DIAG_2021",
    "COVID_SYMP_2021",
    "COVID_HOSP_ANY_2021",
    "COVID_HOSP_NUM_2021",
    "COVID_HOSP_TREAT_ICU_2021",
    "COVID_HOSP_TREAT_OXY_2021",
    "COVID_HOSP_TREAT_VEN_2021",
    "COVID_HOSP_TREAT_OTH_2021",
    "COVID_LING_ANY_2021",
    "COVID_LING_TYPE_2021",
    "COVID_LING_SEV_2021"
  ],
  "demographics": [
    "ID",
    "LINEAGE",
    "PNUM",
    "DEMO_SEX",
    "DEMO_BIRTH_YEAR",
    "DEMO_BIRTH_MONTH",
    "DEMO_DEATH_REP",
    "DEMO_DEATH_YEAR",
    "DEMO_AGE_GEN_1968",
    "DEMO_AGE_GEN_1969",
    "DEMO_AGE_GEN_1970",
    "DEMO_AGE_GEN_1971",
    "DEMO_AGE_GEN_1972",
    "DEMO_AGE_GEN_1973",
    "DEMO_AGE_GEN_1974",
    "DEMO_AGE_GEN_1975",
    "DEMO_AGE_GEN_1976",
    "DEMO_AGE_GEN_1977",
    "DEMO_AGE_GEN_1978",
    "DEMO_AGE_GEN_1979",
    "DEMO_AGE_GEN_1980",
    "DEMO_AGE_GEN_1981",
    "DEMO_AGE_GEN_1982",
    "DEMO_AGE_GEN_1983",
    "DEMO_AGE_GEN_1984",
    "DEMO_AGE_GEN_1985",
    "DEMO_AGE_GEN_1986",
    "DEMO_AGE_GEN_1987",
    "DEMO_AGE_GEN_1988",
    "DEMO_AGE_GEN_1989",
    "DEMO_AGE_GEN_1990",
    "DEMO_AGE_GEN_1991",
    "DEMO_AGE_GEN_1992",
    "DEMO_AGE_GEN_1993",
    "DEMO_AGE_GEN_1994",
    "DEMO_AGE_GEN_1995",
    "DEMO_AGE_GEN_1996",
    "DEMO_AGE_GEN_1997",
    "DEMO_AGE_GEN_1999",
    "DEMO_AGE_GEN_2001",
    "DEMO_AGE_GEN_2003",
    "DEMO_AGE_GEN_2005",
    "DEMO_AGE_GEN_2007",
    "DEMO_AGE_GEN_2009",
    "DEMO_AGE_GEN_2011",
    "DEMO_AGE_GEN_2013",
    "DEMO_AGE_GEN_2015",
    "DEMO_AGE_GEN_2017",
    "DEMO_AGE_GEN_2019",
    "DEMO_AGE_GEN_2021",
    "DEMO_AGE_REP_1968",
    "DEMO_AGE_REP_1969",
    "DEMO_AGE_REP_1970",
    "DEMO_AGE_REP_1971",
    "DEMO_AGE_REP_1972",
    "DEMO_AGE_REP_1973",
    "DEMO_AGE_REP_1974",
    "DEMO_AGE_REP_1975",
    "DEMO_AGE_REP_1976",
    "DEMO_AGE_REP_1977",
    "DEMO_AGE_REP_1978",
    "DEMO_AGE_REP_1979",
    "DEMO_AGE_REP_1980",
    "DEMO_AGE_REP_1981",
    "DEMO_AGE_REP_1982",
    "DEMO_AGE_REP_1983",
    "DEMO_AGE_REP_1984",
    "DEMO_AGE_REP_1985",
    "DEMO_AGE_REP_1986",
    "DEMO_AGE_REP_1987",
    "DEMO_AGE_REP_1988",
    "DEMO_AGE_REP_1989",
    "DEMO_AGE_REP_1990",
    "DEMO_AGE_REP_1991",
    "DEMO_AGE_REP_1992",
    "DEMO_AGE_REP_1993",
    "DEMO_AGE_REP_1994",
    "DEMO_AGE_REP_1995",
    "DEMO_AGE_REP_1996",
    "DEMO_AGE_REP_1997",
    "DEMO_AGE_REP_1999",
    "DEMO_AGE_REP_2001",
    "DEMO_AGE_REP_2003",
    "DEMO_AGE_REP_2005",
    "DEMO_AGE_REP_2007",
    "DEMO_AGE_REP_2009",
    "DEMO_AGE_REP_2011",
    "DEMO_AGE_REP_2013",
    "DEMO_AGE_REP_2015",
    "DEMO_AGE_REP_2017",
    "DEMO_AGE_REP_2019",
    "DEMO_AGE_REP_2021"
  ],
  "depression": [
    "ID",
    "LINEAGE",
    "PNUM",
    "DEP_SCORE_TOT_2001",
    "DEP_SCORE_TOT_2003",
    "DEP_SCORE_TOT_2007",
    "DEP_SCORE_TOT_2009",
    "DEP_SCORE_TOT_2011",
    "DEP_SCORE_TOT_2013",
    "DEP_SCORE_TOT_2015",
    "DEP_SCORE_TOT_2017",
    "DEP_SCORE_TOT_2019",
    "DEP_SCORE_TOT_2021",
    "DEP_SCORE_CUT_2001",
    "DEP_SCORE_CUT_2003",
    "DEP_SCORE_CUT_2007",
    "DEP_SCORE_CUT_2009",
    "DEP_SCORE_CUT_2011",
    "DEP_SCORE_CUT_2013",
    "DEP_SCORE_CUT_2015",
    "DEP_SCORE_CUT_2017",
    "DEP_SCORE_CUT_2019",
    "DEP_SCORE_CUT_2021",
    "DEP_DEGR_2001",
    "DEP_DEGR_2003",
    "DEP_DEGR_2007",
    "DEP_DEGR_2009",
    "DEP_DEGR_2011",
    "DEP_DEGR_2013",
    "DEP_DEGR_2015",
    "DEP_DEGR_2017",
    "DEP_DEGR_2019",
    "DEP_DEGR_2021",
    "DEP_Q1_FREQ_2001",
    "DEP_Q1_FREQ_2003",
    "DEP_Q1_FREQ_2007",
    "DEP_Q1_FREQ_2009",
    "DEP_Q1_FREQ_2011",
    "DEP_Q1_FREQ_2013",
    "DEP_Q1_FREQ_2015",
    "DEP_Q1_FREQ_2017",
    "DEP_Q1_FREQ_2019",
    "DEP_Q1_FREQ_2021",
    "DEP_Q2_FREQ_2001",
    "DEP_Q2_FREQ_2003",
    "DEP_Q2_FREQ_2007",
    "DEP_Q2_FREQ_2009",
    "DEP_Q2_FREQ_2011",
    "DEP_Q2_FREQ_2013",
    "DEP_Q2_FREQ_2015",
    "DEP_Q2_FREQ_2017",
    "DEP_Q2_FREQ_2019",
    "DEP_Q2_FREQ_2021",
    "DEP_Q3_FREQ_2001",
    "DEP_Q3_FREQ_2003",
    "DEP_Q3_FREQ_2007",
    "DEP_Q3_FREQ_2009",
    "DEP_Q3_FREQ_2011",
    "DEP_Q3_FREQ_2013",
    "DEP_Q3_FREQ_2015",
    "DEP_Q3_FREQ_2017",
    "DEP_Q3_FREQ_2019",
    "DEP_Q3_FREQ_2021",
    "DEP_Q4_FREQ_2001",
    "DEP_Q4_FREQ_2003",
    "DEP_Q4_FREQ_2007",
    "DEP_Q4_FREQ_2009",
    "DEP_Q4_FREQ_2011",
    "DEP_Q4_FREQ_2013",
    "DEP_Q4_FREQ_2015",
    "DEP_Q4_FREQ_2017",
    "DEP_Q4_FREQ_2019",
    "DEP_Q4_FREQ_2021",
    "DEP_Q5_FREQ_2001",
    "DEP_Q5_FREQ_2003",
    "DEP_Q5_FREQ_2007",
    "DEP_Q5_FREQ_2009",
    "DEP_Q5_FREQ_2011",
    "DEP_Q5_FREQ_2013",
    "DEP_Q5_FREQ_2015",
    "DEP_Q5_FREQ_2017",
    "DEP_Q5_FREQ_2019",
    "DEP_Q5_FREQ_2021",
    "DEP_Q6_FREQ_2001",
    "DEP_Q6_FREQ_2003",
    "DEP_Q6_FREQ_2007",
    "DEP_Q6_FREQ_2009",
    "DEP_Q6_FREQ_2011",
    "DEP_Q6_FREQ_2013",
    "DEP_Q6_FREQ_2015",
    "DEP_Q6_FREQ_2017",
    "DEP_Q6_FREQ_2019",
    "DEP_Q6_FREQ_2021"
  ],
  "earnings_nominal": [
    "ID",
    "LINEAGE",
    "PNUM",
    "EARN_TOT_ND_1968",
    "EARN_TOT_ND_1969",
    "EARN_TOT_ND_1970",
    "EARN_TOT_ND_1971",
    "EARN_TOT_ND_1972",
    "EARN_TOT_ND_1973",
    "EARN_TOT_ND_1974",
    "EARN_TOT_ND_1975",
    "EARN_TOT_ND_1976",
    "EARN_TOT_ND_1977",
    "EARN_TOT_ND_1978",
    "EARN_TOT_ND_1979",
    "EARN_TOT_ND_1980",
    "EARN_TOT_ND_1981",
    "EARN_TOT_ND_1982",
    "EARN_TOT_ND_1983",
    "EARN_TOT_ND_1984",
    "EARN_TOT_ND_1985",
    "EARN_TOT_ND_1986",
    "EARN_TOT_ND_1987",
    "EARN_TOT_ND_1988",
    "EARN_TOT_ND_1989",
    "EARN_TOT_ND_1990",
    "EARN_TOT_ND_1991",
    "EARN_TOT_ND_1992",
    "EARN_TOT_ND_1993",
    "EARN_TOT_ND_1994",
    "EARN_TOT_ND_1995",
    "EARN_TOT_ND_1996",
    "EARN_TOT_ND_1997",
    "EARN_TOT_ND_1999",
    "EARN_TOT_ND_2001",
    "EARN_TOT_ND_2003",
    "EARN_TOT_ND_2005",
    "EARN_TOT_ND_2007",
    "EARN_TOT_ND_2009",
    "EARN_TOT_ND_2011",
    "EARN_TOT_ND_2013",
    "EARN_TOT_ND_2015",
    "EARN_TOT_ND_2017",
    "EARN_TOT_ND_2019",
    "EARN_TOT_ND_2021",
    "EARN_TOT_NDF_1968",
    "EARN_TOT_NDF_1969",
    "EARN_TOT_NDF_1970",
    "EARN_TOT_NDF_1971",
    "EARN_TOT_NDF_1972",
    "EARN_TOT_NDF_1973",
    "EARN_TOT_NDF_1974",
    "EARN_TOT_NDF_1975",
    "EARN_TOT_NDF_1976",
    "EARN_TOT_NDF_1977",
    "EARN_TOT_NDF_1978",
    "EARN_TOT_NDF_1979",
    "EARN_TOT_NDF_1980",
    "EARN_TOT_NDF_1981",
    "EARN_TOT_NDF_1982",
    "EARN_TOT_NDF_1983",
    "EARN_TOT_NDF_1984",
    "EARN_TOT_NDF_1985",
    "EARN_TOT_NDF_1986",
    "EARN_TOT_NDF_1987",
    "EARN_TOT_NDF_1988",
    "EARN_TOT_NDF_1989",
    "EARN_TOT_NDF_1990",
    "EARN_TOT_NDF_1991",
    "EARN_TOT_NDF_1992",
    "EARN_TOT_NDF_1993",
    "EARN_TOT_NDF_1994",
    "EARN_TOT_NDF_1995",
    "EARN_TOT_NDF_1996",
    "EARN_TOT_NDF_1997",
    "EARN_TOT_NDF_1999",
    "EARN_TOT_NDF_2001",
    "EARN_TOT_NDF_2003",
    "EARN_TOT_NDF_2005",
    "EARN_TOT_NDF_2007",
    "EARN_TOT_NDF_2009",
    "EARN_TOT_NDF_2011",
    "EARN_TOT_NDF_2013",
    "EARN_TOT_NDF_2015",
    "EARN_TOT_NDF_2017",
    "EARN_TOT_NDF_2019",
    "EARN_TOT_NDF_2021",
    "EARN_TOT_ND_RP_1968",
    "EARN_TOT_ND_RP_1969",
    "EARN_TOT_ND_RP_1970",
    "EARN_TOT_ND_RP_1971",
    "EARN_TOT_ND_RP_1972",
    "EARN_TOT_ND_RP_1973",
    "EARN_TOT_ND_RP_1974",
    "EARN_TOT_ND_RP_1975",
    "EARN_TOT_ND_RP_1976",
    "EARN_TOT_ND_RP_1977",
    "EARN_TOT_ND_RP_1978",
    "EARN_TOT_ND_RP_1979",
    "EARN_TOT_ND_RP_1980",
    "EARN_TOT_ND_RP_1981",
    "EARN_TOT_ND_RP_1982",
    "EARN_TOT_ND_RP_1983",
    "EARN_TOT_ND_RP_1984",
    "EARN_TOT_ND_RP_1985",
    "EARN_TOT_ND_RP_1986",
    "EARN_TOT_ND_RP_1987",
    "EARN_TOT_ND_RP_1988",
    "EARN_TOT_ND_RP_1989",
    "EARN_TOT_ND_RP_1990",
    "EARN_TOT_ND_RP_1991",
    "EARN_TOT_ND_RP_1992",
    "EARN_TOT_ND_RP_1993",
    "EARN_TOT_ND_RP_1994",
    "EARN_TOT_ND_RP_1995",
    "EARN_TOT_ND_RP_1996",
    "EARN_TOT_ND_RP_1997",
    "EARN_TOT_ND_RP_1999",
    "EARN_TOT_ND_RP_2001",
    "EARN_TOT_ND_RP_2003",
    "EARN_TOT_ND_RP_2005",
    "EARN_TOT_ND_RP_2007",
    "EARN_TOT_ND_RP_2009",
    "EARN_TOT_ND_RP_2011",
    "EARN_TOT_ND_RP_2013",
    "EARN_TOT_ND_RP_2015",
    "EARN_TOT_ND_RP_2017",
    "EARN_TOT_ND_RP_2019",
    "EARN_TOT_ND_RP_2021",
    "EARN_TOT_ND_SP_1968",
    "EARN_TOT_ND_SP_1969",
    "EARN_TOT_ND_SP_1970",
    "EARN_TOT_ND_SP_1971",
    "EARN_TOT_ND_SP_1972",
    "EARN_TOT_ND_SP_1973",
    "EARN_TOT_ND_SP_1974",
    "EARN_TOT_ND_SP_1975",
    "EARN_TOT_ND_SP_1976",
    "EARN_TOT_ND_SP_1977",
    "EARN_TOT_ND_SP_1978",
    "EARN_TOT_ND_SP_1979",
    "EARN_TOT_ND_SP_1980",
    "EARN_TOT_ND_SP_1981",
    "EARN_TOT_ND_SP_1982",
    "EARN_TOT_ND_SP_1983",
    "EARN_TOT_ND_SP_1984",
    "EARN_TOT_ND_SP_1985",
    "EARN_TOT_ND_SP_1986",
    "EARN_TOT_ND_SP_1987",
    "EARN_TOT_ND_SP_1988",
    "EARN_TOT_ND_SP_1989",
    "EARN_TOT_ND_SP_1990",
    "EARN_TOT_ND_SP_1991",
    "EARN_TOT_ND_SP_1992",
    "EARN_TOT_ND_SP_1993",
    "EARN_TOT_ND_SP_1994",
    "EARN_TOT_ND_SP_1995",
    "EARN_TOT_ND_SP_1996",
    "EARN_TOT_ND_SP_1997",
    "EARN_TOT_ND_SP_1999",
    "EARN_TOT_ND_SP_2001",
    "EARN_TOT_ND_SP_2003",
    "EARN_TOT_ND_SP_2005",
    "EARN_TOT_ND_SP_2007",
    "EARN_TOT_ND_SP_2009",
    "EARN_TOT_ND_SP_2011",
    "EARN_TOT_ND_SP_2013",
    "EARN_TOT_ND_SP_2015",
    "EARN_TOT_ND_SP_2017",
    "EARN_TOT_ND_SP_2019",
    "EARN_TOT_ND_SP_2021",
    "EARN_TOT_NDF_RP_1968",
    "EARN_TOT_NDF_RP_1969",
    "EARN_TOT_NDF_RP_1970",
    "EARN_TOT_NDF_RP_1971",
    "EARN_TOT_NDF_RP_1972",
    "EARN_TOT_NDF_RP_1973",
    "EARN_TOT_NDF_RP_1974",
    "EARN_TOT_NDF_RP_1975",
    "EARN_TOT_NDF_RP_1976",
    "EARN_TOT_NDF_RP_1977",
    "EARN_TOT_NDF_RP_1978",
    "EARN_TOT_NDF_RP_1979",
    "EARN_TOT_NDF_RP_1980",
    "EARN_TOT_NDF_RP_1981",
    "EARN_TOT_NDF_RP_1982",
    "EARN_TOT_NDF_RP_1983",
    "EARN_TOT_NDF_RP_1984",
    "EARN_TOT_NDF_RP_1985",
    "EARN_TOT_NDF_RP_1986",
    "EARN_TOT_NDF_RP_1987",
    "EARN_TOT_NDF_RP_1988",
    "EARN_TOT_NDF_RP_1989",
    "EARN_TOT_NDF_RP_1990",
    "EARN_TOT_NDF_RP_1991",
    "EARN_TOT_NDF_RP_1992",
    "EARN_TOT_NDF_RP_1993",
    "EARN_TOT_NDF_RP_1994",
    "EARN_TOT_NDF_RP_1995",
    "EARN_TOT_NDF_RP_1996",
    "EARN_TOT_NDF_RP_1997",
    "EARN_TOT_NDF_RP_1999",
    "EARN_TOT_NDF_RP_2001",
    "EARN_TOT_NDF_RP_2003",
    "EARN_TOT_NDF_RP_2005",
    "EARN_TOT_NDF_RP_2007",
    "EARN_TOT_NDF_RP_2009",
    "EARN_TOT_NDF_RP_2011",
    "EARN_TOT_NDF_RP_2013",
    "EARN_TOT_NDF_RP_2015",
    "EARN_TOT_NDF_RP_2017",
    "EARN_TOT_NDF_RP_2019",
    "EARN_TOT_NDF_RP_2021",
    "EARN_TOT_NDF_SP_1968",
    "EARN_TOT_NDF_SP_1969",
    "EARN_TOT_NDF_SP_1970",
    "EARN_TOT_NDF_SP_1971",
    "EARN_TOT_NDF_SP_1972",
    "EARN_TOT_NDF_SP_1973",
    "EARN_TOT_NDF_SP_1974",
    "EARN_TOT_NDF_SP_1975",
    "EARN_TOT_NDF_SP_1976",
    "EARN_TOT_NDF_SP_1977",
    "EARN_TOT_NDF_SP_1978",
    "EARN_TOT_NDF_SP_1979",
    "EARN_TOT_NDF_SP_1980",
    "EARN_TOT_NDF_SP_1981",
    "EARN_TOT_NDF_SP_1982",
    "EARN_TOT_NDF_SP_1983",
    "EARN_TOT_NDF_SP_1984",
    "EARN_TOT_NDF_SP_1985",
    "EARN_TOT_NDF_SP_1986",
    "EARN_TOT_NDF_SP_1987",
    "EARN_TOT_NDF_SP_1988",
    "EARN_TOT_NDF_SP_1989",
    "EARN_TOT_NDF_SP_1990",
    "EARN_TOT_NDF_SP_1991",
    "EARN_TOT_NDF_SP_1992",
    "EARN_TOT_NDF_SP_1993",
    "EARN_TOT_NDF_SP_1994",
    "EARN_TOT_NDF_SP_1995",
    "EARN_TOT_NDF_SP_1996",
    "EARN_TOT_NDF_SP_1997",
    "EARN_TOT_NDF_SP_1999",
    "EARN_TOT_NDF_SP_2001",
    "EARN_TOT_NDF_SP_2003",
    "EARN_TOT_NDF_SP_2005",
    "EARN_TOT_NDF_SP_2007",
    "EARN_TOT_NDF_SP_2009",
    "EARN_TOT_NDF_SP_2011",
    "EARN_TOT_NDF_SP_2013",
    "EARN_TOT_NDF_SP_2015",
    "EARN_TOT_NDF_SP_2017",
    "EARN_TOT_NDF_SP_2019",
    "EARN_TOT_NDF_SP_2021"
  ],
  "earnings_real": [
    "ID",
    "LINEAGE",
    "PNUM",
    "EARN_TOT_RD_1968",
    "EARN_TOT_RD_1969",
    "EARN_TOT_RD_1970",
    "EARN_TOT_RD_1971",
    "EARN_TOT_RD_1972",
    "EARN_TOT_RD_1973",
    "EARN_TOT_RD_1974",
    "EARN_TOT_RD_1975",
    "EARN_TOT_RD_1976",
    "EARN_TOT_RD_1977",
    "EARN_TOT_RD_1978",
    "EARN_TOT_RD_1979",
    "EARN_TOT_RD_1980",
    "EARN_TOT_RD_1981",
    "EARN_TOT_RD_1982",
    "EARN_TOT_RD_1983",
    "EARN_TOT_RD_1984",
    "EARN_TOT_RD_1985",
    "EARN_TOT_RD_1986",
    "EARN_TOT_RD_1987",
    "EARN_TOT_RD_1988",
    "EARN_TOT_RD_1989",
    "EARN_TOT_RD_1990",
    "EARN_TOT_RD_1991",
    "EARN_TOT_RD_1992",
    "EARN_TOT_RD_1993",
    "EARN_TOT_RD_1994",
    "EARN_TOT_RD_1995",
    "EARN_TOT_RD_1996",
    "EARN_TOT_RD_1997",
    "EARN_TOT_RD_1999",
    "EARN_TOT_RD_2001",
    "EARN_TOT_RD_2003",
    "EARN_TOT_RD_2005",
    "EARN_TOT_RD_2007",
    "EARN_TOT_RD_2009",
    "EARN_TOT_RD_2011",
    "EARN_TOT_RD_2013",
    "EARN_TOT_RD_2015",
    "EARN_TOT_RD_2017",
    "EARN_TOT_RD_2019",
    "EARN_TOT_RD_2021",
    "EARN_TOT_RDF_1968",
    "EARN_TOT_RDF_1969",
    "EARN_TOT_RDF_1970",
    "EARN_TOT_RDF_1971",
    "EARN_TOT_RDF_1972",
    "EARN_TOT_RDF_1973",
    "EARN_TOT_RDF_1974",
    "EARN_TOT_RDF_1975",
    "EARN_TOT_RDF_1976",
    "EARN_TOT_RDF_1977",
    "EARN_TOT_RDF_1978",
    "EARN_TOT_RDF_1979",
    "EARN_TOT_RDF_1980",
    "EARN_TOT_RDF_1981",
    "EARN_TOT_RDF_1982",
    "EARN_TOT_RDF_1983",
    "EARN_TOT_RDF_1984",
    "EARN_TOT_RDF_1985",
    "EARN_TOT_RDF_1986",
    "EARN_TOT_RDF_1987",
    "EARN_TOT_RDF_1988",
    "EARN_TOT_RDF_1989",
    "EARN_TOT_RDF_1990",
    "EARN_TOT_RDF_1991",
    "EARN_TOT_RDF_1992",
    "EARN_TOT_RDF_1993",
    "EARN_TOT_RDF_1994",
    "EARN_TOT_RDF_1995",
    "EARN_TOT_RDF_1996",
    "EARN_TOT_RDF_1997",
    "EARN_TOT_RDF_1999",
    "EARN_TOT_RDF_2001",
    "EARN_TOT_RDF_2003",
    "EARN_TOT_RDF_2005",
    "EARN_TOT_RDF_2007",
    "EARN_TOT_RDF_2009",
    "EARN_TOT_RDF_2011",
    "EARN_TOT_RDF_2013",
    "EARN_TOT_RDF_2015",
    "EARN_TOT_RDF_2017",
    "EARN_TOT_RDF_2019",
    "EARN_TOT_RDF_2021",
    "EARN_TOT_RD_RP_1968",
    "EARN_TOT_RD_RP_1969",
    "EARN_TOT_RD_RP_1970",
    "EARN_TOT_RD_RP_1971",
    "EARN_TOT_RD_RP_1972",
    "EARN_TOT_RD_RP_1973",
    "EARN_TOT_RD_RP_1974",
    "EARN_TOT_RD_RP_1975",
    "EARN_TOT_RD_RP_1976",
    "EARN_TOT_RD_RP_1977",
    "EARN_TOT_RD_RP_1978",
    "EARN_TOT_RD_RP_1979",
    "EARN_TOT_RD_RP_1980",
    "EARN_TOT_RD_RP_1981",
    "EARN_TOT_RD_RP_1982",
    "EARN_TOT_RD_RP_1983",
    "EARN_TOT_RD_RP_1984",
    "EARN_TOT_RD_RP_1985",
    "EARN_TOT_RD_RP_1986",
    "EARN_TOT_RD_RP_1987",
    "EARN_TOT_RD_RP_1988",
    "EARN_TOT_RD_RP_1989",
    "EARN_TOT_RD_RP_1990",
    "EARN_TOT_RD_RP_1991",
    "EARN_TOT_RD_RP_1992",
    "EARN_TOT_RD_RP_1993",
    "EARN_TOT_RD_RP_1994",
    "EARN_TOT_RD_RP_1995",
    "EARN_TOT_RD_RP_1996",
    "EARN_TOT_RD_RP_1997",
    "EARN_TOT_RD_RP_1999",
    "EARN_TOT_RD_RP_2001",
    "EARN_TOT_RD_RP_2003",
    "EARN_TOT_RD_RP_2005",
    "EARN_TOT_RD_RP_2007",
    "EARN_TOT_RD_RP_2009",
    "EARN_TOT_RD_RP_2011",
    "EARN_TOT_RD_RP_2013",
    "EARN_TOT_RD_RP_2015",
    "EARN_TOT_RD_RP_2017",
    "EARN_TOT_RD_RP_2019",
    "EARN_TOT_RD_RP_2021",
    "EARN_TOT_RD_SP_1968",
    "EARN_TOT_RD_SP_1969",
    "EARN_TOT_RD_SP_1970",
    "EARN_TOT_RD_SP_1971",
    "EARN_TOT_RD_SP_1972",
    "EARN_TOT_RD_SP_1973",
    "EARN_TOT_RD_SP_1974",
    "EARN_TOT_RD_SP_1975",
    "EARN_TOT_RD_SP_1976",
    "EARN_TOT_RD_SP_1977",
    "EARN_TOT_RD_SP_1978",
    "EARN_TOT_RD_SP_1979",
    "EARN_TOT_RD_SP_1980",
    "EARN_TOT_RD_SP_1981",
    "EARN_TOT_RD_SP_1982",
    "EARN_TOT_RD_SP_1983",
    "EARN_TOT_RD_SP_1984",
    "EARN_TOT_RD_SP_1985",
    "EARN_TOT_RD_SP_1986",
    "EARN_TOT_RD_SP_1987",
    "EARN_TOT_RD_SP_1988",
    "EARN_TOT_RD_SP_1989",
    "EARN_TOT_RD_SP_1990",
    "EARN_TOT_RD_SP_1991",
    "EARN_TOT_RD_SP_1992",
    "EARN_TOT_RD_SP_1993",
    "EARN_TOT_RD_SP_1994",
    "EARN_TOT_RD_SP_1995",
    "EARN_TOT_RD_SP_1996",
    "EARN_TOT_RD_SP_1997",
    "EARN_TOT_RD_SP_1999",
    "EARN_TOT_RD_SP_2001",
    "EARN_TOT_RD_SP_2003",
    "EARN_TOT_RD_SP_2005",
    "EARN_TOT_RD_SP_2007",
    "EARN_TOT_RD_SP_2009",
    "EARN_TOT_RD_SP_2011",
    "EARN_TOT_RD_SP_2013",
    "EARN_TOT_RD_SP_2015",
    "EARN_TOT_RD_SP_2017",
    "EARN_TOT_RD_SP_2019",
    "EARN_TOT_RD_SP_2021",
    "EARN_TOT_RDF_RP_1968",
    "EARN_TOT_RDF_RP_1969",
    "EARN_TOT_RDF_RP_1970",
    "EARN_TOT_RDF_RP_1971",
    "EARN_TOT_RDF_RP_1972",
    "EARN_TOT_RDF_RP_1973",
    "EARN_TOT_RDF_RP_1974",
    "EARN_TOT_RDF_RP_1975",
    "EARN_TOT_RDF_RP_1976",
    "EARN_TOT_RDF_RP_1977",
    "EARN_TOT_RDF_RP_1978",
    "EARN_TOT_RDF_RP_1979",
    "EARN_TOT_RDF_RP_1980",
    "EARN_TOT_RDF_RP_1981",
    "EARN_TOT_RDF_RP_1982",
    "EARN_TOT_RDF_RP_1983",
    "EARN_TOT_RDF_RP_1984",
    "EARN_TOT_RDF_RP_1985",
    "EARN_TOT_RDF_RP_1986",
    "EARN_TOT_RDF_RP_1987",
    "EARN_TOT_RDF_RP_1988",
    "EARN_TOT_RDF_RP_1989",
    "EARN_TOT_RDF_RP_1990",
    "EARN_TOT_RDF_RP_1991",
    "EARN_TOT_RDF_RP_1992",
    "EARN_TOT_RDF_RP_1993",
    "EARN_TOT_RDF_RP_1994",
    "EARN_TOT_RDF_RP_1995",
    "EARN_TOT_RDF_RP_1996",
    "EARN_TOT_RDF_RP_1997",
    "EARN_TOT_RDF_RP_1999",
    "EARN_TOT_RDF_RP_2001",
    "EARN_TOT_RDF_RP_2003",
    "EARN_TOT_RDF_RP_2005",
    "EARN_TOT_RDF_RP_2007",
    "EARN_TOT_RDF_RP_2009",
    "EARN_TOT_RDF_RP_2011",
    "EARN_TOT_RDF_RP_2013",
    "EARN_TOT_RDF_RP_2015",
    "EARN_TOT_RDF_RP_2017",
    "EARN_TOT_RDF_RP_2019",
    "EARN_TOT_RDF_RP_2021",
    "EARN_TOT_RDF_SP_1968",
    "EARN_TOT_RDF_SP_1969",
    "EARN_TOT_RDF_SP_1970",
    "EARN_TOT_RDF_SP_1971",
    "EARN_TOT_RDF_SP_1972",
    "EARN_TOT_RDF_SP_1973",
    "EARN_TOT_RDF_SP_1974",
    "EARN_TOT_RDF_SP_1975",
    "EARN_TOT_RDF_SP_1976",
    "EARN_TOT_RDF_SP_1977",
    "EARN_TOT_RDF_SP_1978",
    "EARN_TOT_RDF_SP_1979",
    "EARN_TOT_RDF_SP_1980",
    "EARN_TOT_RDF_SP_1981",
    "EARN_TOT_RDF_SP_1982",
    "EARN_TOT_RDF_SP_1983",
    "EARN_TOT_RDF_SP_1984",
    "EARN_TOT_RDF_SP_1985",
    "EARN_TOT_RDF_SP_1986",
    "EARN_TOT_RDF_SP_1987",
    "EARN_TOT_RDF_SP_1988",
    "EARN_TOT_RDF_SP_1989",
    "EARN_TOT_RDF_SP_1990",
    "EARN_TOT_RDF_SP_1991",
    "EARN_TOT_RDF_SP_1992",
    "EARN_TOT_RDF_SP_1993",
    "EARN_TOT_RDF_SP_1994",
    "EARN_TOT_RDF_SP_1995",
    "EARN_TOT_RDF_SP_1996",
    "EARN_TOT_RDF_SP_1997",
    "EARN_TOT_RDF_SP_1999",
    "EARN_TOT_RDF_SP_2001",
    "EARN_TOT_RDF_SP_2003",
    "EARN_TOT_RDF_SP_2005",
    "EARN_TOT_RDF_SP_2007",
    "EARN_TOT_RDF_SP_2009",
    "EARN_TOT_RDF_SP_2011",
    "EARN_TOT_RDF_SP_2013",
    "EARN_TOT_RDF_SP_2015",
    "EARN_TOT_RDF_SP_2017",
    "EARN_TOT_RDF_SP_2019",
    "EARN_TOT_RDF_SP_2021"
  ],
  "education": [
    "ID",
    "LINEAGE",
    "PNUM",
    "EDU_YEAR_1968",
    "EDU_YEAR_1969",
    "EDU_YEAR_1970",
    "EDU_YEAR_1971",
    "EDU_YEAR_1972",
    "EDU_YEAR_1973",
    "EDU_YEAR_1974",
    "EDU_YEAR_1975",
    "EDU_YEAR_1976",
    "EDU_YEAR_1977",
    "EDU_YEAR_1978",
    "EDU_YEAR_1979",
    "EDU_YEAR_1980",
    "EDU_YEAR_1981",
    "EDU_YEAR_1982",
    "EDU_YEAR_1983",
    "EDU_YEAR_1984",
    "EDU_YEAR_1985",
    "EDU_YEAR_1986",
    "EDU_YEAR_1987",
    "EDU_YEAR_1988",
    "EDU_YEAR_1989",
    "EDU_YEAR_1990",
    "EDU_YEAR_1991",
    "EDU_YEAR_1992",
    "EDU_YEAR_1993",
    "EDU_YEAR_1994",
    "EDU_YEAR_1995",
    "EDU_YEAR_1996",
    "EDU_YEAR_1997",
    "EDU_YEAR_1999",
    "EDU_YEAR_2001",
    "EDU_YEAR_2003",
    "EDU_YEAR_2005",
    "EDU_YEAR_2007",
    "EDU_YEAR_2009",
    "EDU_YEAR_2011",
    "EDU_YEAR_2013",
    "EDU_YEAR_2015",
    "EDU_YEAR_2017",
    "EDU_YEAR_2019",
    "EDU_YEAR_2021",
    "EDU_YEAR_RP_1968",
    "EDU_YEAR_RP_1969",
    "EDU_YEAR_RP_1970",
    "EDU_YEAR_RP_1971",
    "EDU_YEAR_RP_1972",
    "EDU_YEAR_RP_1973",
    "EDU_YEAR_RP_1974",
    "EDU_YEAR_RP_1975",
    "EDU_YEAR_RP_1976",
    "EDU_YEAR_RP_1977",
    "EDU_YEAR_RP_1978",
    "EDU_YEAR_RP_1979",
    "EDU_YEAR_RP_1980",
    "EDU_YEAR_RP_1981",
    "EDU_YEAR_RP_1982",
    "EDU_YEAR_RP_1983",
    "EDU_YEAR_RP_1984",
    "EDU_YEAR_RP_1985",
    "EDU_YEAR_RP_1986",
    "EDU_YEAR_RP_1987",
    "EDU_YEAR_RP_1988",
    "EDU_YEAR_RP_1989",
    "EDU_YEAR_RP_1990",
    "EDU_YEAR_RP_1991",
    "EDU_YEAR_RP_1992",
    "EDU_YEAR_RP_1993",
    "EDU_YEAR_RP_1994",
    "EDU_YEAR_RP_1995",
    "EDU_YEAR_RP_1996",
    "EDU_YEAR_RP_1997",
    "EDU_YEAR_RP_1999",
    "EDU_YEAR_RP_2001",
    "EDU_YEAR_RP_2003",
    "EDU_YEAR_RP_2005",
    "EDU_YEAR_RP_2007",
    "EDU_YEAR_RP_2009",
    "EDU_YEAR_RP_2011",
    "EDU_YEAR_RP_2013",
    "EDU_YEAR_RP_2015",
    "EDU_YEAR_RP_2017",
    "EDU_YEAR_RP_2019",
    "EDU_YEAR_RP_2021",
    "EDU_YEAR_SP_1968",
    "EDU_YEAR_SP_1969",
    "EDU_YEAR_SP_1970",
    "EDU_YEAR_SP_1971",
    "EDU_YEAR_SP_1972",
    "EDU_YEAR_SP_1973",
    "EDU_YEAR_SP_1974",
    "EDU_YEAR_SP_1975",
    "EDU_YEAR_SP_1976",
    "EDU_YEAR_SP_1977",
    "EDU_YEAR_SP_1978",
    "EDU_YEAR_SP_1979",
    "EDU_YEAR_SP_1980",
    "EDU_YEAR_SP_1981",
    "EDU_YEAR_SP_1982",
    "EDU_YEAR_SP_1983",
    "EDU_YEAR_SP_1984",
    "EDU_YEAR_SP_1985",
    "EDU_YEAR_SP_1986",
    "EDU_YEAR_SP_1987",
    "EDU_YEAR_SP_1988",
    "EDU_YEAR_SP_1989",
    "EDU_YEAR_SP_1990",
    "EDU_YEAR_SP_1991",
    "EDU_YEAR_SP_1992",
    "EDU_YEAR_SP_1993",
    "EDU_YEAR_SP_1994",
    "EDU_YEAR_SP_1995",
    "EDU_YEAR_SP_1996",
    "EDU_YEAR_SP_1997",
    "EDU_YEAR_SP_1999",
    "EDU_YEAR_SP_2001",
    "EDU_YEAR_SP_2003",
    "EDU_YEAR_SP_2005",
    "EDU_YEAR_SP_2007",
    "EDU_YEAR_SP_2009",
    "EDU_YEAR_SP_2011",
    "EDU_YEAR_SP_2013",
    "EDU_YEAR_SP_2015",
    "EDU_YEAR_SP_2017",
    "EDU_YEAR_SP_2019",
    "EDU_YEAR_SP_2021",
    "EDU_YEAR_MAX",
    "EDU_YEAR_MAX_RP_1968",
    "EDU_YEAR_MAX_RP_1969",
    "EDU_YEAR_MAX_RP_1970",
    "EDU_YEAR_MAX_RP_1971",
    "EDU_YEAR_MAX_RP_1972",
    "EDU_YEAR_MAX_RP_1973",
    "EDU_YEAR_MAX_RP_1974",
    "EDU_YEAR_MAX_RP_1975",
    "EDU_YEAR_MAX_RP_1976",
    "EDU_YEAR_MAX_RP_1977",
    "EDU_YEAR_MAX_RP_1978",
    "EDU_YEAR_MAX_RP_1979",
    "EDU_YEAR_MAX_RP_1980",
    "EDU_YEAR_MAX_RP_1981",
    "EDU_YEAR_MAX_RP_1982",
    "EDU_YEAR_MAX_RP_1983",
    "EDU_YEAR_MAX_RP_1984",
    "EDU_YEAR_MAX_RP_1985",
    "EDU_YEAR_MAX_RP_1986",
    "EDU_YEAR_MAX_RP_1987",
    "EDU_YEAR_MAX_RP_1988",
    "EDU_YEAR_MAX_RP_1989",
    "EDU_YEAR_MAX_RP_1990",
    "EDU_YEAR_MAX_RP_1991",
    "EDU_YEAR_MAX_RP_1992",
    "EDU_YEAR_MAX_RP_1993",
    "EDU_YEAR_MAX_RP_1994",
    "EDU_YEAR_MAX_RP_1995",
    "EDU_YEAR_MAX_RP_1996",
    "EDU_YEAR_MAX_RP_1997",
    "EDU_YEAR_MAX_RP_1999",
    "EDU_YEAR_MAX_RP_2001",
    "EDU_YEAR_MAX_RP_2003",
    "EDU_YEAR_MAX_RP_2005",
    "EDU_YEAR_MAX_RP_2007",
    "EDU_YEAR_MAX_RP_2009",
    "EDU_YEAR_MAX_RP_2011",
    "EDU_YEAR_MAX_RP_2013",
    "EDU_YEAR_MAX_RP_2015",
    "EDU_YEAR_MAX_RP_2017",
    "EDU_YEAR_MAX_RP_2019",
    "EDU_YEAR_MAX_RP_2021",
    "EDU_YEAR_MAX_SP_1968",
    "EDU_YEAR_MAX_SP_1969",
    "EDU_YEAR_MAX_SP_1970",
    "EDU_YEAR_MAX_SP_1971",
    "EDU_YEAR_MAX_SP_1972",
    "EDU_YEAR_MAX_SP_1973",
    "EDU_YEAR_MAX_SP_1974",
    "EDU_YEAR_MAX_SP_1975",
    "EDU_YEAR_MAX_SP_1976",
    "EDU_YEAR_MAX_SP_1977",
    "EDU_YEAR_MAX_SP_1978",
    "EDU_YEAR_MAX_SP_1979",
    "EDU_YEAR_MAX_SP_1980",
    "EDU_YEAR_MAX_SP_1981",
    "EDU_YEAR_MAX_SP_1982",
    "EDU_YEAR_MAX_SP_1983",
    "EDU_YEAR_MAX_SP_1984",
    "EDU_YEAR_MAX_SP_1985",
    "EDU_YEAR_MAX_SP_1986",
    "EDU_YEAR_MAX_SP_1987",
    "EDU_YEAR_MAX_SP_1988",
    "EDU_YEAR_MAX_SP_1989",
    "EDU_YEAR_MAX_SP_1990",
    "EDU_YEAR_MAX_SP_1991",
    "EDU_YEAR_MAX_SP_1992",
    "EDU_YEAR_MAX_SP_1993",
    "EDU_YEAR_MAX_SP_1994",
    "EDU_YEAR_MAX_SP_1995",
    "EDU_YEAR_MAX_SP_1996",
    "EDU_YEAR_MAX_SP_1997",
    "EDU_YEAR_MAX_SP_1999",
    "EDU_YEAR_MAX_SP_2001",
    "EDU_YEAR_MAX_SP_2003",
    "EDU_YEAR_MAX_SP_2005",
    "EDU_YEAR_MAX_SP_2007",
    "EDU_YEAR_MAX_SP_2009",
    "EDU_YEAR_MAX_SP_2011",
    "EDU_YEAR_MAX_SP_2013",
    "EDU_YEAR_MAX_SP_2015",
    "EDU_YEAR_MAX_SP_2017",
    "EDU_YEAR_MAX_SP_2019",
    "EDU_YEAR_MAX_SP_2021",
    "EDU_LEVEL_1968",
    "EDU_LEVEL_1969",
    "EDU_LEVEL_1970",
    "EDU_LEVEL_1971",
    "EDU_LEVEL_1972",
    "EDU_LEVEL_1973",
    "EDU_LEVEL_1974",
    "EDU_LEVEL_1975",
    "EDU_LEVEL_1976",
    "EDU_LEVEL_1977",
    "EDU_LEVEL_1978",
    "EDU_LEVEL_1979",
    "EDU_LEVEL_1980",
    "EDU_LEVEL_1981",
    "EDU_LEVEL_1982",
    "EDU_LEVEL_1983",
    "EDU_LEVEL_1984",
    "EDU_LEVEL_1985",
    "EDU_LEVEL_1986",
    "EDU_LEVEL_1987",
    "EDU_LEVEL_1988",
    "EDU_LEVEL_1989",
    "EDU_LEVEL_1990",
    "EDU_LEVEL_1991",
    "EDU_LEVEL_1992",
    "EDU_LEVEL_1993",
    "EDU_LEVEL_1994",
    "EDU_LEVEL_1995",
    "EDU_LEVEL_1996",
    "EDU_LEVEL_1997",
    "EDU_LEVEL_1999",
    "EDU_LEVEL_2001",
    "EDU_LEVEL_2003",
    "EDU_LEVEL_2005",
    "EDU_LEVEL_2007",
    "EDU_LEVEL_2009",
    "EDU_LEVEL_2011",
    "EDU_LEVEL_2013",
    "EDU_LEVEL_2015",
    "EDU_LEVEL_2017",
    "EDU_LEVEL_2019",
    "EDU_LEVEL_2021",
    "EDU_LEVEL_RP_1968",
    "EDU_LEVEL_RP_1969",
    "EDU_LEVEL_RP_1970",
    "EDU_LEVEL_RP_1971",
    "EDU_LEVEL_RP_1972",
    "EDU_LEVEL_RP_1973",
    "EDU_LEVEL_RP_1974",
    "EDU_LEVEL_RP_1975",
    "EDU_LEVEL_RP_1976",
    "EDU_LEVEL_RP_1977",
    "EDU_LEVEL_RP_1978",
    "EDU_LEVEL_RP_1979",
    "EDU_LEVEL_RP_1980",
    "EDU_LEVEL_RP_1981",
    "EDU_LEVEL_RP_1982",
    "EDU_LEVEL_RP_1983",
    "EDU_LEVEL_RP_1984",
    "EDU_LEVEL_RP_1985",
    "EDU_LEVEL_RP_1986",
    "EDU_LEVEL_RP_1987",
    "EDU_LEVEL_RP_1988",
    "EDU_LEVEL_RP_1989",
    "EDU_LEVEL_RP_1990",
    "EDU_LEVEL_RP_1991",
    "EDU_LEVEL_RP_1992",
    "EDU_LEVEL_RP_1993",
    "EDU_LEVEL_RP_1994",
    "EDU_LEVEL_RP_1995",
    "EDU_LEVEL_RP_1996",
    "EDU_LEVEL_RP_1997",
    "EDU_LEVEL_RP_1999",
    "EDU_LEVEL_RP_2001",
    "EDU_LEVEL_RP_2003",
    "EDU_LEVEL_RP_2005",
    "EDU_LEVEL_RP_2007",
    "EDU_LEVEL_RP_2009",
    "EDU_LEVEL_RP_2011",
    "EDU_LEVEL_RP_2013",
    "EDU_LEVEL_RP_2015",
    "EDU_LEVEL_RP_2017",
    "EDU_LEVEL_RP_2019",
    "EDU_LEVEL_RP_2021",
    "EDU_LEVEL_SP_1968",
    "EDU_LEVEL_SP_1969",
    "EDU_LEVEL_SP_1970",
    "EDU_LEVEL_SP_1971",
    "EDU_LEVEL_SP_1972",
    "EDU_LEVEL_SP_1973",
    "EDU_LEVEL_SP_1974",
    "EDU_LEVEL_SP_1975",
    "EDU_LEVEL_SP_1976",
    "EDU_LEVEL_SP_1977",
    "EDU_LEVEL_SP_1978",
    "EDU_LEVEL_SP_1979",
    "EDU_LEVEL_SP_1980",
    "EDU_LEVEL_SP_1981",
    "EDU_LEVEL_SP_1982",
    "EDU_LEVEL_SP_1983",
    "EDU_LEVEL_SP_1984",
    "EDU_LEVEL_SP_1985",
    "EDU_LEVEL_SP_1986",
    "EDU_LEVEL_SP_1987",
    "EDU_LEVEL_SP_1988",
    "EDU_LEVEL_SP_1989",
    "EDU_LEVEL_SP_1990",
    "EDU_LEVEL_SP_1991",
    "EDU_LEVEL_SP_1992",
    "EDU_LEVEL_SP_1993",
    "EDU_LEVEL_SP_1994",
    "EDU_LEVEL_SP_1995",
    "EDU_LEVEL_SP_1996",
    "EDU_LEVEL_SP_1997",
    "EDU_LEVEL_SP_1999",
    "EDU_LEVEL_SP_2001",
    "EDU_LEVEL_SP_2003",
    "EDU_LEVEL_SP_2005",
    "EDU_LEVEL_SP_2007",
    "EDU_LEVEL_SP_2009",
    "EDU_LEVEL_SP_2011",
    "EDU_LEVEL_SP_2013",
    "EDU_LEVEL_SP_2015",
    "EDU_LEVEL_SP_2017",
    "EDU_LEVEL_SP_2019",
    "EDU_LEVEL_SP_2021",
    "EDU_LEVEL_MAX",
    "EDU_LEVEL_MAX_RP_1968",
    "EDU_LEVEL_MAX_RP_1969",
    "EDU_LEVEL_MAX_RP_1970",
    "EDU_LEVEL_MAX_RP_1971",
    "EDU_LEVEL_MAX_RP_1972",
    "EDU_LEVEL_MAX_RP_1973",
    "EDU_LEVEL_MAX_RP_1974",
    "EDU_LEVEL_MAX_RP_1975",
    "EDU_LEVEL_MAX_RP_1976",
    "EDU_LEVEL_MAX_RP_1977",
    "EDU_LEVEL_MAX_RP_1978",
    "EDU_LEVEL_MAX_RP_1979",
    "EDU_LEVEL_MAX_RP_1980",
    "EDU_LEVEL_MAX_RP_1981",
    "EDU_LEVEL_MAX_RP_1982",
    "EDU_LEVEL_MAX_RP_1983",
    "EDU_LEVEL_MAX_RP_1984",
    "EDU_LEVEL_MAX_RP_1985",
    "EDU_LEVEL_MAX_RP_1986",
    "EDU_LEVEL_MAX_RP_1987",
    "EDU_LEVEL_MAX_RP_1988",
    "EDU_LEVEL_MAX_RP_1989",
    "EDU_LEVEL_MAX_RP_1990",
    "EDU_LEVEL_MAX_RP_1991",
    "EDU_LEVEL_MAX_RP_1992",
    "EDU_LEVEL_MAX_RP_1993",
    "EDU_LEVEL_MAX_RP_1994",
    "EDU_LEVEL_MAX_RP_1995",
    "EDU_LEVEL_MAX_RP_1996",
    "EDU_LEVEL_MAX_RP_1997",
    "EDU_LEVEL_MAX_RP_1999",
    "EDU_LEVEL_MAX_RP_2001",
    "EDU_LEVEL_MAX_RP_2003",
    "EDU_LEVEL_MAX_RP_2005",
    "EDU_LEVEL_MAX_RP_2007",
    "EDU_LEVEL_MAX_RP_2009",
    "EDU_LEVEL_MAX_RP_2011",
    "EDU_LEVEL_MAX_RP_2013",
    "EDU_LEVEL_MAX_RP_2015",
    "EDU_LEVEL_MAX_RP_2017",
    "EDU_LEVEL_MAX_RP_2019",
    "EDU_LEVEL_MAX_RP_2021",
    "EDU_LEVEL_MAX_SP_1968",
    "EDU_LEVEL_MAX_SP_1969",
    "EDU_LEVEL_MAX_SP_1970",
    "EDU_LEVEL_MAX_SP_1971",
    "EDU_LEVEL_MAX_SP_1972",
    "EDU_LEVEL_MAX_SP_1973",
    "EDU_LEVEL_MAX_SP_1974",
    "EDU_LEVEL_MAX_SP_1975",
    "EDU_LEVEL_MAX_SP_1976",
    "EDU_LEVEL_MAX_SP_1977",
    "EDU_LEVEL_MAX_SP_1978",
    "EDU_LEVEL_MAX_SP_1979",
    "EDU_LEVEL_MAX_SP_1980",
    "EDU_LEVEL_MAX_SP_1981",
    "EDU_LEVEL_MAX_SP_1982",
    "EDU_LEVEL_MAX_SP_1983",
    "EDU_LEVEL_MAX_SP_1984",
    "EDU_LEVEL_MAX_SP_1985",
    "EDU_LEVEL_MAX_SP_1986",
    "EDU_LEVEL_MAX_SP_1987",
    "EDU_LEVEL_MAX_SP_1988",
    "EDU_LEVEL_MAX_SP_1989",
    "EDU_LEVEL_MAX_SP_1990",
    "EDU_LEVEL_MAX_SP_1991",
    "EDU_LEVEL_MAX_SP_1992",
    "EDU_LEVEL_MAX_SP_1993",
    "EDU_LEVEL_MAX_SP_1994",
    "EDU_LEVEL_MAX_SP_1995",
    "EDU_LEVEL_MAX_SP_1996",
    "EDU_LEVEL_MAX_SP_1997",
    "EDU_LEVEL_MAX_SP_1999",
    "EDU_LEVEL_MAX_SP_2001",
    "EDU_LEVEL_MAX_SP_2003",
    "EDU_LEVEL_MAX_SP_2005",
    "EDU_LEVEL_MAX_SP_2007",
    "EDU_LEVEL_MAX_SP_2009",
    "EDU_LEVEL_MAX_SP_2011",
    "EDU_LEVEL_MAX_SP_2013",
    "EDU_LEVEL_MAX_SP_2015",
    "EDU_LEVEL_MAX_SP_2017",
    "EDU_LEVEL_MAX_SP_2019",
    "EDU_LEVEL_MAX_SP_2021",
    "EDU_GRDE_1968",
    "EDU_GRDE_1969",
    "EDU_GRDE_1970",
    "EDU_GRDE_1971",
    "EDU_GRDE_1972",
    "EDU_GRDE_1973",
    "EDU_GRDE_1974",
    "EDU_GRDE_1975",
    "EDU_GRDE_1976",
    "EDU_GRDE_1977",
    "EDU_GRDE_1978",
    "EDU_GRDE_1979",
    "EDU_GRDE_1980",
    "EDU_GRDE_1981",
    "EDU_GRDE_1982",
    "EDU_GRDE_1983",
    "EDU_GRDE_1984",
    "EDU_GRDE_1985",
    "EDU_GRDE_1986",
    "EDU_GRDE_1987",
    "EDU_GRDE_1988",
    "EDU_GRDE_1989",
    "EDU_GRDE_1990",
    "EDU_HSCH_1985",
    "EDU_HSCH_1986",
    "EDU_HSCH_1987",
    "EDU_HSCH_1988",
    "EDU_HSCH_1989",
    "EDU_HSCH_1990",
    "EDU_HSCH_1991",
    "EDU_HSCH_1992",
    "EDU_HSCH_1993",
    "EDU_HSCH_1994",
    "EDU_HSCH_1995",
    "EDU_HSCH_1996",
    "EDU_HSCH_1997",
    "EDU_HSCH_1999",
    "EDU_HSCH_2001",
    "EDU_HSCH_2003",
    "EDU_HSCH_2005",
    "EDU_HSCH_2007",
    "EDU_HSCH_2009",
    "EDU_HSCH_2011",
    "EDU_HSCH_2013",
    "EDU_HSCH_2015",
    "EDU_HSCH_2017",
    "EDU_HSCH_2019",
    "EDU_HSCH_2021",
    "EDU_COLL_ATT_1985",
    "EDU_COLL_ATT_1986",
    "EDU_COLL_ATT_1987",
    "EDU_COLL_ATT_1988",
    "EDU_COLL_ATT_1989",
    "EDU_COLL_ATT_1990",
    "EDU_COLL_ATT_1991",
    "EDU_COLL_ATT_1992",
    "EDU_COLL_ATT_1993",
    "EDU_COLL_ATT_1994",
    "EDU_COLL_ATT_1995",
    "EDU_COLL_ATT_1996",
    "EDU_COLL_ATT_1997",
    "EDU_COLL_ATT_1999",
    "EDU_COLL_ATT_2001",
    "EDU_COLL_ATT_2003",
    "EDU_COLL_ATT_2005",
    "EDU_COLL_ATT_2007",
    "EDU_COLL_ATT_2009",
    "EDU_COLL_ATT_2011",
    "EDU_COLL_ATT_2013",
    "EDU_COLL_ATT_2015",
    "EDU_COLL_ATT_2017",
    "EDU_COLL_ATT_2019",
    "EDU_COLL_ATT_2021",
    "EDU_COLL_DEG_1985",
    "EDU_COLL_DEG_1986",
    "EDU_COLL_DEG_1987",
    "EDU_COLL_DEG_1988",
    "EDU_COLL_DEG_1989",
    "EDU_COLL_DEG_1990",
    "EDU_COLL_DEG_1991",
    "EDU_COLL_DEG_1992",
    "EDU_COLL_DEG_1993",
    "EDU_COLL_DEG_1994",
    "EDU_COLL_DEG_1995",
    "EDU_COLL_DEG_1996",
    "EDU_COLL_DEG_1997",
    "EDU_COLL_DEG_1999",
    "EDU_COLL_DEG_2001",
    "EDU_COLL_DEG_2003",
    "EDU_COLL_DEG_2005",
    "EDU_COLL_DEG_2007",
    "EDU_COLL_DEG_2009",
    "EDU_COLL_DEG_2011",
    "EDU_COLL_DEG_2013",
    "EDU_COLL_DEG_2015",
    "EDU_COLL_DEG_2017",
    "EDU_COLL_DEG_2019",
    "EDU_COLL_DEG_2021",
    "EDU_COLL_GRA_1975",
    "EDU_COLL_GRA_1976",
    "EDU_COLL_GRA_1977",
    "EDU_COLL_GRA_1978",
    "EDU_COLL_GRA_1979",
    "EDU_COLL_GRA_1980",
    "EDU_COLL_GRA_1981",
    "EDU_COLL_GRA_1982",
    "EDU_COLL_GRA_1983",
    "EDU_COLL_GRA_1984",
    "EDU_COLL_GRA_1985",
    "EDU_COLL_GRA_1986",
    "EDU_COLL_GRA_1987",
    "EDU_COLL_GRA_1988",
    "EDU_COLL_GRA_1989",
    "EDU_COLL_GRA_1990",
    "EDU_COLL_GRA_1991",
    "EDU_COLL_GRA_1992",
    "EDU_COLL_GRA_1993",
    "EDU_COLL_GRA_1994",
    "EDU_COLL_GRA_1995",
    "EDU_COLL_GRA_1996",
    "EDU_COLL_GRA_1997",
    "EDU_COLL_GRA_1999",
    "EDU_COLL_GRA_2001",
    "EDU_COLL_GRA_2003",
    "EDU_COLL_GRA_2005",
    "EDU_COLL_GRA_2007",
    "EDU_COLL_GRA_2009",
    "EDU_COLL_GRA_2011",
    "EDU_COLL_GRA_2013",
    "EDU_COLL_GRA_2015",
    "EDU_COLL_GRA_2017",
    "EDU_COLL_GRA_2019",
    "EDU_COLL_GRA_2021",
    "EDU_COLL_NUM_1985",
    "EDU_COLL_NUM_1986",
    "EDU_COLL_NUM_1987",
    "EDU_COLL_NUM_1988",
    "EDU_COLL_NUM_1989",
    "EDU_COLL_NUM_1990",
    "EDU_COLL_NUM_1991",
    "EDU_COLL_NUM_1992",
    "EDU_COLL_NUM_1993",
    "EDU_COLL_NUM_1994",
    "EDU_COLL_NUM_1995",
    "EDU_COLL_NUM_1996",
    "EDU_COLL_NUM_1997",
    "EDU_COLL_NUM_1999",
    "EDU_COLL_NUM_2001",
    "EDU_COLL_NUM_2003",
    "EDU_COLL_NUM_2005",
    "EDU_COLL_NUM_2007",
    "EDU_COLL_NUM_2009",
    "EDU_COLL_NUM_2011",
    "EDU_COLL_NUM_2013",
    "EDU_COLL_NUM_2015",
    "EDU_COLL_NUM_2017",
    "EDU_COLL_NUM_2019",
    "EDU_COLL_NUM_2021",
    "EDU_ICOL_ATT_1997",
    "EDU_ICOL_ATT_1999",
    "EDU_ICOL_ATT_2001",
    "EDU_ICOL_ATT_2003",
    "EDU_ICOL_ATT_2005",
    "EDU_ICOL_ATT_2007",
    "EDU_ICOL_ATT_2009",
    "EDU_ICOL_ATT_2011",
    "EDU_ICOL_ATT_2013",
    "EDU_ICOL_ATT_2015",
    "EDU_ICOL_ATT_2017",
    "EDU_ICOL_ATT_2019",
    "EDU_ICOL_ATT_2021",
    "EDU_ICOL_DEG_1997",
    "EDU_ICOL_DEG_1999",
    "EDU_ICOL_DEG_2001",
    "EDU_ICOL_DEG_2005",
    "EDU_ICOL_DEG_2007",
    "EDU_ICOL_DEG_2009",
    "EDU_ICOL_DEG_2011",
    "EDU_ICOL_DEG_2013",
    "EDU_ICOL_DEG_2015",
    "EDU_ICOL_DEG_2017",
    "EDU_ICOL_DEG_2019",
    "EDU_ICOL_DEG_2021"
  ],
  "employment": [
    "ID",
    "LINEAGE",
    "PNUM",
    "EMP_STAT_1M_1968",
    "EMP_STAT_1M_1969",
    "EMP_STAT_1M_1970",
    "EMP_STAT_1M_1971",
    "EMP_STAT_1M_1972",
    "EMP_STAT_1M_1973",
    "EMP_STAT_1M_1974",
    "EMP_STAT_1M_1975",
    "EMP_STAT_1M_1976",
    "EMP_STAT_1M_1977",
    "EMP_STAT_1M_1978",
    "EMP_STAT_1M_1979",
    "EMP_STAT_1M_1980",
    "EMP_STAT_1M_1981",
    "EMP_STAT_1M_1982",
    "EMP_STAT_1M_1983",
    "EMP_STAT_1M_1984",
    "EMP_STAT_1M_1985",
    "EMP_STAT_1M_1986",
    "EMP_STAT_1M_1987",
    "EMP_STAT_1M_1988",
    "EMP_STAT_1M_1989",
    "EMP_STAT_1M_1990",
    "EMP_STAT_1M_1991",
    "EMP_STAT_1M_1992",
    "EMP_STAT_1M_1993",
    "EMP_STAT_1M_1994",
    "EMP_STAT_1M_1995",
    "EMP_STAT_1M_1996",
    "EMP_STAT_1M_1997",
    "EMP_STAT_1M_1999",
    "EMP_STAT_1M_2001",
    "EMP_STAT_1M_2003",
    "EMP_STAT_1M_2005",
    "EMP_STAT_1M_2007",
    "EMP_STAT_1M_2009",
    "EMP_STAT_1M_2011",
    "EMP_STAT_1M_2013",
    "EMP_STAT_1M_2015",
    "EMP_STAT_1M_2017",
    "EMP_STAT_1M_2019",
    "EMP_STAT_1M_2021",
    "EMP_STAT_2M_1994",
    "EMP_STAT_2M_1995",
    "EMP_STAT_2M_1996",
    "EMP_STAT_2M_1997",
    "EMP_STAT_2M_1999",
    "EMP_STAT_2M_2001",
    "EMP_STAT_2M_2003",
    "EMP_STAT_2M_2005",
    "EMP_STAT_2M_2007",
    "EMP_STAT_2M_2009",
    "EMP_STAT_2M_2011",
    "EMP_STAT_2M_2013",
    "EMP_STAT_2M_2015",
    "EMP_STAT_2M_2017",
    "EMP_STAT_2M_2019",
    "EMP_STAT_2M_2021",
    "EMP_STAT_3M_1994",
    "EMP_STAT_3M_1995",
    "EMP_STAT_3M_1996",
    "EMP_STAT_3M_1997",
    "EMP_STAT_3M_1999",
    "EMP_STAT_3M_2001",
    "EMP_STAT_3M_2003",
    "EMP_STAT_3M_2005",
    "EMP_STAT_3M_2007",
    "EMP_STAT_3M_2009",
    "EMP_STAT_3M_2011",
    "EMP_STAT_3M_2013",
    "EMP_STAT_3M_2015",
    "EMP_STAT_3M_2017",
    "EMP_STAT_3M_2019",
    "EMP_STAT_3M_2021",
    "EMP_STAT_1M_RP_1968",
    "EMP_STAT_1M_RP_1969",
    "EMP_STAT_1M_RP_1970",
    "EMP_STAT_1M_RP_1971",
    "EMP_STAT_1M_RP_1972",
    "EMP_STAT_1M_RP_1973",
    "EMP_STAT_1M_RP_1974",
    "EMP_STAT_1M_RP_1975",
    "EMP_STAT_1M_RP_1976",
    "EMP_STAT_1M_RP_1977",
    "EMP_STAT_1M_RP_1978",
    "EMP_STAT_1M_RP_1979",
    "EMP_STAT_1M_RP_1980",
    "EMP_STAT_1M_RP_1981",
    "EMP_STAT_1M_RP_1982",
    "EMP_STAT_1M_RP_1983",
    "EMP_STAT_1M_RP_1984",
    "EMP_STAT_1M_RP_1985",
    "EMP_STAT_1M_RP_1986",
    "EMP_STAT_1M_RP_1987",
    "EMP_STAT_1M_RP_1988",
    "EMP_STAT_1M_RP_1989",
    "EMP_STAT_1M_RP_1990",
    "EMP_STAT_1M_RP_1991",
    "EMP_STAT_1M_RP_1992",
    "EMP_STAT_1M_RP_1993",
    "EMP_STAT_1M_RP_1994",
    "EMP_STAT_1M_RP_1995",
    "EMP_STAT_1M_RP_1996",
    "EMP_STAT_1M_RP_1997",
    "EMP_STAT_1M_RP_1999",
    "EMP_STAT_1M_RP_2001",
    "EMP_STAT_1M_RP_2003",
    "EMP_STAT_1M_RP_2005",
    "EMP_STAT_1M_RP_2007",
    "EMP_STAT_1M_RP_2009",
    "EMP_STAT_1M_RP_2011",
    "EMP_STAT_1M_RP_2013",
    "EMP_STAT_1M_RP_2015",
    "EMP_STAT_1M_RP_2017",
    "EMP_STAT_1M_RP_2019",
    "EMP_STAT_1M_RP_2021",
    "EMP_STAT_2M_RP_1994",
    "EMP_STAT_2M_RP_1995",
    "EMP_STAT_2M_RP_1996",
    "EMP_STAT_2M_RP_1997",
    "EMP_STAT_2M_RP_1999",
    "EMP_STAT_2M_RP_2001",
    "EMP_STAT_2M_RP_2003",
    "EMP_STAT_2M_RP_2005",
    "EMP_STAT_2M_RP_2007",
    "EMP_STAT_2M_RP_2009",
    "EMP_STAT_2M_RP_2011",
    "EMP_STAT_2M_RP_2013",
    "EMP_STAT_2M_RP_2015",
    "EMP_STAT_2M_RP_2017",
    "EMP_STAT_2M_RP_2019",
    "EMP_STAT_2M_RP_2021",
    "EMP_STAT_3M_RP_1994",
    "EMP_STAT_3M_RP_1995",
    "EMP_STAT_3M_RP_1996",
    "EMP_STAT_3M_RP_1997",
    "EMP_STAT_3M_RP_1999",
    "EMP_STAT_3M_RP_2001",
    "EMP_STAT_3M_RP_2003",
    "EMP_STAT_3M_RP_2005",
    "EMP_STAT_3M_RP_2007",
    "EMP_STAT_3M_RP_2009",
    "EMP_STAT_3M_RP_2011",
    "EMP_STAT_3M_RP_2013",
    "EMP_STAT_3M_RP_2015",
    "EMP_STAT_3M_RP_2017",
    "EMP_STAT_3M_RP_2019",
    "EMP_STAT_3M_RP_2021",
    "EMP_STAT_1M_SP_1976",
    "EMP_STAT_1M_SP_1979",
    "EMP_STAT_1M_SP_1980",
    "EMP_STAT_1M_SP_1981",
    "EMP_STAT_1M_SP_1982",
    "EMP_STAT_1M_SP_1983",
    "EMP_STAT_1M_SP_1984",
    "EMP_STAT_1M_SP_1985",
    "EMP_STAT_1M_SP_1986",
    "EMP_STAT_1M_SP_1987",
    "EMP_STAT_1M_SP_1988",
    "EMP_STAT_1M_SP_1989",
    "EMP_STAT_1M_SP_1990",
    "EMP_STAT_1M_SP_1991",
    "EMP_STAT_1M_SP_1992",
    "EMP_STAT_1M_SP_1993",
    "EMP_STAT_1M_SP_1994",
    "EMP_STAT_1M_SP_1995",
    "EMP_STAT_1M_SP_1996",
    "EMP_STAT_1M_SP_1997",
    "EMP_STAT_1M_SP_1999",
    "EMP_STAT_1M_SP_2001",
    "EMP_STAT_1M_SP_2003",
    "EMP_STAT_1M_SP_2005",
    "EMP_STAT_1M_SP_2007",
    "EMP_STAT_1M_SP_2009",
    "EMP_STAT_1M_SP_2011",
    "EMP_STAT_1M_SP_2013",
    "EMP_STAT_1M_SP_2015",
    "EMP_STAT_1M_SP_2017",
    "EMP_STAT_1M_SP_2019",
    "EMP_STAT_1M_SP_2021",
    "EMP_STAT_2M_SP_1994",
    "EMP_STAT_2M_SP_1995",
    "EMP_STAT_2M_SP_1996",
    "EMP_STAT_2M_SP_1997",
    "EMP_STAT_2M_SP_1999",
    "EMP_STAT_2M_SP_2001",
    "EMP_STAT_2M_SP_2003",
    "EMP_STAT_2M_SP_2005",
    "EMP_STAT_2M_SP_2007",
    "EMP_STAT_2M_SP_2009",
    "EMP_STAT_2M_SP_2011",
    "EMP_STAT_2M_SP_2013",
    "EMP_STAT_2M_SP_2015",
    "EMP_STAT_2M_SP_2017",
    "EMP_STAT_2M_SP_2019",
    "EMP_STAT_2M_SP_2021",
    "EMP_STAT_3M_SP_1994",
    "EMP_STAT_3M_SP_1995",
    "EMP_STAT_3M_SP_1996",
    "EMP_STAT_3M_SP_1997",
    "EMP_STAT_3M_SP_1999",
    "EMP_STAT_3M_SP_2001",
    "EMP_STAT_3M_SP_2003",
    "EMP_STAT_3M_SP_2005",
    "EMP_STAT_3M_SP_2007",
    "EMP_STAT_3M_SP_2009",
    "EMP_STAT_3M_SP_2011",
    "EMP_STAT_3M_SP_2013",
    "EMP_STAT_3M_SP_2015",
    "EMP_STAT_3M_SP_2017",
    "EMP_STAT_3M_SP_2019",
    "EMP_STAT_3M_SP_2021",
    "EMP_WORK_1968",
    "EMP_WORK_1969",
    "EMP_WORK_1970",
    "EMP_WORK_1971",
    "EMP_WORK_1972",
    "EMP_WORK_1973",
    "EMP_WORK_1974",
    "EMP_WORK_1975",
    "EMP_WORK_1976",
    "EMP_WORK_1977",
    "EMP_WORK_1978",
    "EMP_WORK_1979",
    "EMP_WORK_1980",
    "EMP_WORK_1981",
    "EMP_WORK_1982",
    "EMP_WORK_1983",
    "EMP_WORK_1984",
    "EMP_WORK_1985",
    "EMP_WORK_1986",
    "EMP_WORK_1987",
    "EMP_WORK_1988",
    "EMP_WORK_1989",
    "EMP_WORK_1990",
    "EMP_WORK_1991",
    "EMP_WORK_1992",
    "EMP_WORK_1993",
    "EMP_WORK_1994",
    "EMP_WORK_1995",
    "EMP_WORK_1996",
    "EMP_WORK_1997",
    "EMP_WORK_1999",
    "EMP_WORK_2001",
    "EMP_WORK_2003",
    "EMP_WORK_2005",
    "EMP_WORK_2007",
    "EMP_WORK_2009",
    "EMP_WORK_2011",
    "EMP_WORK_2013",
    "EMP_WORK_2015",
    "EMP_WORK_2017",
    "EMP_WORK_2019",
    "EMP_WORK_2021",
    "EMP_WORK_RP_1968",
    "EMP_WORK_RP_1969",
    "EMP_WORK_RP_1970",
    "EMP_WORK_RP_1971",
    "EMP_WORK_RP_1972",
    "EMP_WORK_RP_1973",
    "EMP_WORK_RP_1974",
    "EMP_WORK_RP_1975",
    "EMP_WORK_RP_1976",
    "EMP_WORK_RP_1977",
    "EMP_WORK_RP_1978",
    "EMP_WORK_RP_1979",
    "EMP_WORK_RP_1980",
    "EMP_WORK_RP_1981",
    "EMP_WORK_RP_1982",
    "EMP_WORK_RP_1983",
    "EMP_WORK_RP_1984",
    "EMP_WORK_RP_1985",
    "EMP_WORK_RP_1986",
    "EMP_WORK_RP_1987",
    "EMP_WORK_RP_1988",
    "EMP_WORK_RP_1989",
    "EMP_WORK_RP_1990",
    "EMP_WORK_RP_1991",
    "EMP_WORK_RP_1992",
    "EMP_WORK_RP_1993",
    "EMP_WORK_RP_1994",
    "EMP_WORK_RP_1995",
    "EMP_WORK_RP_1996",
    "EMP_WORK_RP_1997",
    "EMP_WORK_RP_1999",
    "EMP_WORK_RP_2001",
    "EMP_WORK_RP_2003",
    "EMP_WORK_RP_2005",
    "EMP_WORK_RP_2007",
    "EMP_WORK_RP_2009",
    "EMP_WORK_RP_2011",
    "EMP_WORK_RP_2013",
    "EMP_WORK_RP_2015",
    "EMP_WORK_RP_2017",
    "EMP_WORK_RP_2019",
    "EMP_WORK_RP_2021",
    "EMP_WORK_SP_1976",
    "EMP_WORK_SP_1979",
    "EMP_WORK_SP_1980",
    "EMP_WORK_SP_1981",
    "EMP_WORK_SP_1982",
    "EMP_WORK_SP_1983",
    "EMP_WORK_SP_1984",
    "EMP_WORK_SP_1985",
    "EMP_WORK_SP_1986",
    "EMP_WORK_SP_1987",
    "EMP_WORK_SP_1988",
    "EMP_WORK_SP_1989",
    "EMP_WORK_SP_1990",
    "EMP_WORK_SP_1991",
    "EMP_WORK_SP_1992",
    "EMP_WORK_SP_1993",
    "EMP_WORK_SP_1994",
    "EMP_WORK_SP_1995",
    "EMP_WORK_SP_1996",
    "EMP_WORK_SP_1997",
    "EMP_WORK_SP_1999",
    "EMP_WORK_SP_2001",
    "EMP_WORK_SP_2003",
    "EMP_WORK_SP_2005",
    "EMP_WORK_SP_2007",
    "EMP_WORK_SP_2009",
    "EMP_WORK_SP_2011",
    "EMP_WORK_SP_2013",
    "EMP_WORK_SP_2015",
    "EMP_WORK_SP_2017",
    "EMP_WORK_SP_2019",
    "EMP_WORK_SP_2021",
    "EMP_WORK_MM_1968",
    "EMP_WORK_MM_1969",
    "EMP_WORK_MM_1970",
    "EMP_WORK_MM_1971",
    "EMP_WORK_MM_1972",
    "EMP_WORK_MM_1973",
    "EMP_WORK_MM_1974",
    "EMP_WORK_MM_1975",
    "EMP_WORK_MM_1976",
    "EMP_WORK_MM_1977",
    "EMP_WORK_MM_1978",
    "EMP_WORK_MM_1979",
    "EMP_WORK_MM_1980",
    "EMP_WORK_MM_1981",
    "EMP_WORK_MM_1982",
    "EMP_WORK_MM_1983",
    "EMP_WORK_MM_1984",
    "EMP_WORK_MM_1985",
    "EMP_WORK_MM_1986",
    "EMP_WORK_MM_1987",
    "EMP_WORK_MM_1988",
    "EMP_WORK_MM_1989",
    "EMP_WORK_MM_1990",
    "EMP_WORK_MM_1991",
    "EMP_WORK_MM_1992",
    "EMP_WORK_MM_1993",
    "EMP_WORK_MM_1994",
    "EMP_WORK_MM_1995",
    "EMP_WORK_MM_1996",
    "EMP_WORK_MM_1997",
    "EMP_WORK_MM_1999",
    "EMP_WORK_MM_2001",
    "EMP_WORK_MM_2003",
    "EMP_WORK_MM_2005",
    "EMP_WORK_MM_2007",
    "EMP_WORK_MM_2009",
    "EMP_WORK_MM_2011",
    "EMP_WORK_MM_2013",
    "EMP_WORK_MM_2015",
    "EMP_WORK_MM_2017",
    "EMP_WORK_MM_2019",
    "EMP_WORK_MM_2021",
    "EMP_WORK_MM_RP_1968",
    "EMP_WORK_MM_RP_1969",
    "EMP_WORK_MM_RP_1970",
    "EMP_WORK_MM_RP_1971",
    "EMP_WORK_MM_RP_1972",
    "EMP_WORK_MM_RP_1973",
    "EMP_WORK_MM_RP_1974",
    "EMP_WORK_MM_RP_1975",
    "EMP_WORK_MM_RP_1976",
    "EMP_WORK_MM_RP_1977",
    "EMP_WORK_MM_RP_1978",
    "EMP_WORK_MM_RP_1979",
    "EMP_WORK_MM_RP_1980",
    "EMP_WORK_MM_RP_1981",
    "EMP_WORK_MM_RP_1982",
    "EMP_WORK_MM_RP_1983",
    "EMP_WORK_MM_RP_1984",
    "EMP_WORK_MM_RP_1985",
    "EMP_WORK_MM_RP_1986",
    "EMP_WORK_MM_RP_1987",
    "EMP_WORK_MM_RP_1988",
    "EMP_WORK_MM_RP_1989",
    "EMP_WORK_MM_RP_1990",
    "EMP_WORK_MM_RP_1991",
    "EMP_WORK_MM_RP_1992",
    "EMP_WORK_MM_RP_1993",
    "EMP_WORK_MM_RP_1994",
    "EMP_WORK_MM_RP_1995",
    "EMP_WORK_MM_RP_1996",
    "EMP_WORK_MM_RP_1997",
    "EMP_WORK_MM_RP_1999",
    "EMP_WORK_MM_RP_2001",
    "EMP_WORK_MM_RP_2003",
    "EMP_WORK_MM_RP_2005",
    "EMP_WORK_MM_RP_2007",
    "EMP_WORK_MM_RP_2009",
    "EMP_WORK_MM_RP_2011",
    "EMP_WORK_MM_RP_2013",
    "EMP_WORK_MM_RP_2015",
    "EMP_WORK_MM_RP_2017",
    "EMP_WORK_MM_RP_2019",
    "EMP_WORK_MM_RP_2021",
    "EMP_WORK_MM_SP_1976",
    "EMP_WORK_MM_SP_1979",
    "EMP_WORK_MM_SP_1980",
    "EMP_WORK_MM_SP_1981",
    "EMP_WORK_MM_SP_1982",
    "EMP_WORK_MM_SP_1983",
    "EMP_WORK_MM_SP_1984",
    "EMP_WORK_MM_SP_1985",
    "EMP_WORK_MM_SP_1986",
    "EMP_WORK_MM_SP_1987",
    "EMP_WORK_MM_SP_1988",
    "EMP_WORK_MM_SP_1989",
    "EMP_WORK_MM_SP_1990",
    "EMP_WORK_MM_SP_1991",
    "EMP_WORK_MM_SP_1992",
    "EMP_WORK_MM_SP_1993",
    "EMP_WORK_MM_SP_1994",
    "EMP_WORK_MM_SP_1995",
    "EMP_WORK_MM_SP_1996",
    "EMP_WORK_MM_SP_1997",
    "EMP_WORK_MM_SP_1999",
    "EMP_WORK_MM_SP_2001",
    "EMP_WORK_MM_SP_2003",
    "EMP_WORK_MM_SP_2005",
    "EMP_WORK_MM_SP_2007",
    "EMP_WORK_MM_SP_2009",
    "EMP_WORK_MM_SP_2011",
    "EMP_WORK_MM_SP_2013",
    "EMP_WORK_MM_SP_2015",
    "EMP_WORK_MM_SP_2017",
    "EMP_WORK_MM_SP_2019",
    "EMP_WORK_MM_SP_2021"
  ],
  "expenditures_health": [
    "ID",
    "LINEAGE",
    "PNUM",
    "EXPN_HLTH_TOT_ND_1999",
    "EXPN_HLTH_TOT_ND_2001",
    "EXPN_HLTH_TOT_ND_2003",
    "EXPN_HLTH_TOT_ND_2005",
    "EXPN_HLTH_TOT_ND_2007",
    "EXPN_HLTH_TOT_ND_2009",
    "EXPN_HLTH_TOT_ND_2011",
    "EXPN_HLTH_TOT_ND_2013",
    "EXPN_HLTH_TOT_ND_2015",
    "EXPN_HLTH_TOT_ND_2017",
    "EXPN_HLTH_TOT_ND_2019",
    "EXPN_HLTH_TOT_ND_2021",
    "EXPN_HLTH_TOT_NDF_1999",
    "EXPN_HLTH_TOT_NDF_2001",
    "EXPN_HLTH_TOT_NDF_2003",
    "EXPN_HLTH_TOT_NDF_2005",
    "EXPN_HLTH_TOT_NDF_2007",
    "EXPN_HLTH_TOT_NDF_2009",
    "EXPN_HLTH_TOT_NDF_2011",
    "EXPN_HLTH_TOT_NDF_2013",
    "EXPN_HLTH_TOT_NDF_2015",
    "EXPN_HLTH_TOT_NDF_2017",
    "EXPN_HLTH_TOT_NDF_2019",
    "EXPN_HLTH_TOT_NDF_2021",
    "EXPN_HLTH_TOT_RD_1999",
    "EXPN_HLTH_TOT_RD_2001",
    "EXPN_HLTH_TOT_RD_2003",
    "EXPN_HLTH_TOT_RD_2005",
    "EXPN_HLTH_TOT_RD_2007",
    "EXPN_HLTH_TOT_RD_2009",
    "EXPN_HLTH_TOT_RD_2011",
    "EXPN_HLTH_TOT_RD_2013",
    "EXPN_HLTH_TOT_RD_2015",
    "EXPN_HLTH_TOT_RD_2017",
    "EXPN_HLTH_TOT_RD_2019",
    "EXPN_HLTH_TOT_RD_2021",
    "EXPN_HLTH_TOT_RDF_1999",
    "EXPN_HLTH_TOT_RDF_2001",
    "EXPN_HLTH_TOT_RDF_2003",
    "EXPN_HLTH_TOT_RDF_2005",
    "EXPN_HLTH_TOT_RDF_2007",
    "EXPN_HLTH_TOT_RDF_2009",
    "EXPN_HLTH_TOT_RDF_2011",
    "EXPN_HLTH_TOT_RDF_2013",
    "EXPN_HLTH_TOT_RDF_2015",
    "EXPN_HLTH_TOT_RDF_2017",
    "EXPN_HLTH_TOT_RDF_2019",
    "EXPN_HLTH_TOT_RDF_2021",
    "EXPN_HLTH_DOC_ND_1999",
    "EXPN_HLTH_DOC_ND_2001",
    "EXPN_HLTH_DOC_ND_2003",
    "EXPN_HLTH_DOC_ND_2005",
    "EXPN_HLTH_DOC_ND_2007",
    "EXPN_HLTH_DOC_ND_2009",
    "EXPN_HLTH_DOC_ND_2011",
    "EXPN_HLTH_DOC_ND_2013",
    "EXPN_HLTH_DOC_ND_2015",
    "EXPN_HLTH_DOC_ND_2017",
    "EXPN_HLTH_DOC_ND_2019",
    "EXPN_HLTH_DOC_ND_2021",
    "EXPN_HLTH_DOC_NDF_1999",
    "EXPN_HLTH_DOC_NDF_2001",
    "EXPN_HLTH_DOC_NDF_2003",
    "EXPN_HLTH_DOC_NDF_2005",
    "EXPN_HLTH_DOC_NDF_2007",
    "EXPN_HLTH_DOC_NDF_2009",
    "EXPN_HLTH_DOC_NDF_2011",
    "EXPN_HLTH_DOC_NDF_2013",
    "EXPN_HLTH_DOC_NDF_2015",
    "EXPN_HLTH_DOC_NDF_2017",
    "EXPN_HLTH_DOC_NDF_2019",
    "EXPN_HLTH_DOC_NDF_2021",
    "EXPN_HLTH_DOC_RD_1999",
    "EXPN_HLTH_DOC_RD_2001",
    "EXPN_HLTH_DOC_RD_2003",
    "EXPN_HLTH_DOC_RD_2005",
    "EXPN_HLTH_DOC_RD_2007",
    "EXPN_HLTH_DOC_RD_2009",
    "EXPN_HLTH_DOC_RD_2011",
    "EXPN_HLTH_DOC_RD_2013",
    "EXPN_HLTH_DOC_RD_2015",
    "EXPN_HLTH_DOC_RD_2017",
    "EXPN_HLTH_DOC_RD_2019",
    "EXPN_HLTH_DOC_RD_2021",
    "EXPN_HLTH_DOC_RDF_1999",
    "EXPN_HLTH_DOC_RDF_2001",
    "EXPN_HLTH_DOC_RDF_2003",
    "EXPN_HLTH_DOC_RDF_2005",
    "EXPN_HLTH_DOC_RDF_2007",
    "EXPN_HLTH_DOC_RDF_2009",
    "EXPN_HLTH_DOC_RDF_2011",
    "EXPN_HLTH_DOC_RDF_2013",
    "EXPN_HLTH_DOC_RDF_2015",
    "EXPN_HLTH_DOC_RDF_2017",
    "EXPN_HLTH_DOC_RDF_2019",
    "EXPN_HLTH_DOC_RDF_2021",
    "EXPN_HLTH_HOS_ND_1999",
    "EXPN_HLTH_HOS_ND_2001",
    "EXPN_HLTH_HOS_ND_2003",
    "EXPN_HLTH_HOS_ND_2005",
    "EXPN_HLTH_HOS_ND_2007",
    "EXPN_HLTH_HOS_ND_2009",
    "EXPN_HLTH_HOS_ND_2011",
    "EXPN_HLTH_HOS_ND_2013",
    "EXPN_HLTH_HOS_ND_2015",
    "EXPN_HLTH_HOS_ND_2017",
    "EXPN_HLTH_HOS_ND_2019",
    "EXPN_HLTH_HOS_ND_2021",
    "EXPN_HLTH_HOS_NDF_1999",
    "EXPN_HLTH_HOS_NDF_2001",
    "EXPN_HLTH_HOS_NDF_2003",
    "EXPN_HLTH_HOS_NDF_2005",
    "EXPN_HLTH_HOS_NDF_2007",
    "EXPN_HLTH_HOS_NDF_2009",
    "EXPN_HLTH_HOS_NDF_2011",
    "EXPN_HLTH_HOS_NDF_2013",
    "EXPN_HLTH_HOS_NDF_2015",
    "EXPN_HLTH_HOS_NDF_2017",
    "EXPN_HLTH_HOS_NDF_2019",
    "EXPN_HLTH_HOS_NDF_2021",
    "EXPN_HLTH_HOS_RD_1999",
    "EXPN_HLTH_HOS_RD_2001",
    "EXPN_HLTH_HOS_RD_2003",
    "EXPN_HLTH_HOS_RD_2005",
    "EXPN_HLTH_HOS_RD_2007",
    "EXPN_HLTH_HOS_RD_2009",
    "EXPN_HLTH_HOS_RD_2011",
    "EXPN_HLTH_HOS_RD_2013",
    "EXPN_HLTH_HOS_RD_2015",
    "EXPN_HLTH_HOS_RD_2017",
    "EXPN_HLTH_HOS_RD_2019",
    "EXPN_HLTH_HOS_RD_2021",
    "EXPN_HLTH_HOS_RDF_1999",
    "EXPN_HLTH_HOS_RDF_2001",
    "EXPN_HLTH_HOS_RDF_2003",
    "EXPN_HLTH_HOS_RDF_2005",
    "EXPN_HLTH_HOS_RDF_2007",
    "EXPN_HLTH_HOS_RDF_2009",
    "EXPN_HLTH_HOS_RDF_2011",
    "EXPN_HLTH_HOS_RDF_2013",
    "EXPN_HLTH_HOS_RDF_2015",
    "EXPN_HLTH_HOS_RDF_2017",
    "EXPN_HLTH_HOS_RDF_2019",
    "EXPN_HLTH_HOS_RDF_2021",
    "EXPN_HLTH_INS_ND_1999",
    "EXPN_HLTH_INS_ND_2001",
    "EXPN_HLTH_INS_ND_2003",
    "EXPN_HLTH_INS_ND_2005",
    "EXPN_HLTH_INS_ND_2007",
    "EXPN_HLTH_INS_ND_2009",
    "EXPN_HLTH_INS_ND_2011",
    "EXPN_HLTH_INS_ND_2013",
    "EXPN_HLTH_INS_ND_2015",
    "EXPN_HLTH_INS_ND_2017",
    "EXPN_HLTH_INS_ND_2019",
    "EXPN_HLTH_INS_ND_2021",
    "EXPN_HLTH_INS_NDF_1999",
    "EXPN_HLTH_INS_NDF_2001",
    "EXPN_HLTH_INS_NDF_2003",
    "EXPN_HLTH_INS_NDF_2005",
    "EXPN_HLTH_INS_NDF_2007",
    "EXPN_HLTH_INS_NDF_2009",
    "EXPN_HLTH_INS_NDF_2011",
    "EXPN_HLTH_INS_NDF_2013",
    "EXPN_HLTH_INS_NDF_2015",
    "EXPN_HLTH_INS_NDF_2017",
    "EXPN_HLTH_INS_NDF_2019",
    "EXPN_HLTH_INS_NDF_2021",
    "EXPN_HLTH_INS_RD_1999",
    "EXPN_HLTH_INS_RD_2001",
    "EXPN_HLTH_INS_RD_2003",
    "EXPN_HLTH_INS_RD_2005",
    "EXPN_HLTH_INS_RD_2007",
    "EXPN_HLTH_INS_RD_2009",
    "EXPN_HLTH_INS_RD_2011",
    "EXPN_HLTH_INS_RD_2013",
    "EXPN_HLTH_INS_RD_2015",
    "EXPN_HLTH_INS_RD_2017",
    "EXPN_HLTH_INS_RD_2019",
    "EXPN_HLTH_INS_RD_2021",
    "EXPN_HLTH_INS_RDF_1999",
    "EXPN_HLTH_INS_RDF_2001",
    "EXPN_HLTH_INS_RDF_2003",
    "EXPN_HLTH_INS_RDF_2005",
    "EXPN_HLTH_INS_RDF_2007",
    "EXPN_HLTH_INS_RDF_2009",
    "EXPN_HLTH_INS_RDF_2011",
    "EXPN_HLTH_INS_RDF_2013",
    "EXPN_HLTH_INS_RDF_2015",
    "EXPN_HLTH_INS_RDF_2017",
    "EXPN_HLTH_INS_RDF_2019",
    "EXPN_HLTH_INS_RDF_2021",
    "EXPN_HLTH_PRE_ND_1999",
    "EXPN_HLTH_PRE_ND_2001",
    "EXPN_HLTH_PRE_ND_2003",
    "EXPN_HLTH_PRE_ND_2005",
    "EXPN_HLTH_PRE_ND_2007",
    "EXPN_HLTH_PRE_ND_2009",
    "EXPN_HLTH_PRE_ND_2011",
    "EXPN_HLTH_PRE_ND_2013",
    "EXPN_HLTH_PRE_ND_2015",
    "EXPN_HLTH_PRE_ND_2017",
    "EXPN_HLTH_PRE_ND_2019",
    "EXPN_HLTH_PRE_ND_2021",
    "EXPN_HLTH_PRE_NDF_1999",
    "EXPN_HLTH_PRE_NDF_2001",
    "EXPN_HLTH_PRE_NDF_2003",
    "EXPN_HLTH_PRE_NDF_2005",
    "EXPN_HLTH_PRE_NDF_2007",
    "EXPN_HLTH_PRE_NDF_2009",
    "EXPN_HLTH_PRE_NDF_2011",
    "EXPN_HLTH_PRE_NDF_2013",
    "EXPN_HLTH_PRE_NDF_2015",
    "EXPN_HLTH_PRE_NDF_2017",
    "EXPN_HLTH_PRE_NDF_2019",
    "EXPN_HLTH_PRE_NDF_2021",
    "EXPN_HLTH_PRE_RD_1999",
    "EXPN_HLTH_PRE_RD_2001",
    "EXPN_HLTH_PRE_RD_2003",
    "EXPN_HLTH_PRE_RD_2005",
    "EXPN_HLTH_PRE_RD_2007",
    "EXPN_HLTH_PRE_RD_2009",
    "EXPN_HLTH_PRE_RD_2011",
    "EXPN_HLTH_PRE_RD_2013",
    "EXPN_HLTH_PRE_RD_2015",
    "EXPN_HLTH_PRE_RD_2017",
    "EXPN_HLTH_PRE_RD_2019",
    "EXPN_HLTH_PRE_RD_2021",
    "EXPN_HLTH_PRE_RDF_1999",
    "EXPN_HLTH_PRE_RDF_2001",
    "EXPN_HLTH_PRE_RDF_2003",
    "EXPN_HLTH_PRE_RDF_2005",
    "EXPN_HLTH_PRE_RDF_2007",
    "EXPN_HLTH_PRE_RDF_2009",
    "EXPN_HLTH_PRE_RDF_2011",
    "EXPN_HLTH_PRE_RDF_2013",
    "EXPN_HLTH_PRE_RDF_2015",
    "EXPN_HLTH_PRE_RDF_2017",
    "EXPN_HLTH_PRE_RDF_2019",
    "EXPN_HLTH_PRE_RDF_2021"
  ],
  "expenditures_household": [
    "ID",
    "LINEAGE",
    "PNUM",
    "EXPN_FOOD_TOT_ND_1999",
    "EXPN_FOOD_TOT_ND_2001",
    "EXPN_FOOD_TOT_ND_2003",
    "EXPN_FOOD_TOT_ND_2005",
    "EXPN_FOOD_TOT_ND_2007",
    "EXPN_FOOD_TOT_ND_2009",
    "EXPN_FOOD_TOT_ND_2011",
    "EXPN_FOOD_TOT_ND_2013",
    "EXPN_FOOD_TOT_ND_2015",
    "EXPN_FOOD_TOT_ND_2017",
    "EXPN_FOOD_TOT_ND_2019",
    "EXPN_FOOD_TOT_ND_2021",
    "EXPN_FOOD_TOT_NDF_1999",
    "EXPN_FOOD_TOT_NDF_2001",
    "EXPN_FOOD_TOT_NDF_2003",
    "EXPN_FOOD_TOT_NDF_2005",
    "EXPN_FOOD_TOT_NDF_2007",
    "EXPN_FOOD_TOT_NDF_2009",
    "EXPN_FOOD_TOT_NDF_2011",
    "EXPN_FOOD_TOT_NDF_2013",
    "EXPN_FOOD_TOT_NDF_2015",
    "EXPN_FOOD_TOT_NDF_2017",
    "EXPN_FOOD_TOT_NDF_2019",
    "EXPN_FOOD_TOT_NDF_2021",
    "EXPN_FOOD_TOT_RD_1999",
    "EXPN_FOOD_TOT_RD_2001",
    "EXPN_FOOD_TOT_RD_2003",
    "EXPN_FOOD_TOT_RD_2005",
    "EXPN_FOOD_TOT_RD_2007",
    "EXPN_FOOD_TOT_RD_2009",
    "EXPN_FOOD_TOT_RD_2011",
    "EXPN_FOOD_TOT_RD_2013",
    "EXPN_FOOD_TOT_RD_2015",
    "EXPN_FOOD_TOT_RD_2017",
    "EXPN_FOOD_TOT_RD_2019",
    "EXPN_FOOD_TOT_RD_2021",
    "EXPN_FOOD_TOT_RDF_1999",
    "EXPN_FOOD_TOT_RDF_2001",
    "EXPN_FOOD_TOT_RDF_2003",
    "EXPN_FOOD_TOT_RDF_2005",
    "EXPN_FOOD_TOT_RDF_2007",
    "EXPN_FOOD_TOT_RDF_2009",
    "EXPN_FOOD_TOT_RDF_2011",
    "EXPN_FOOD_TOT_RDF_2013",
    "EXPN_FOOD_TOT_RDF_2015",
    "EXPN_FOOD_TOT_RDF_2017",
    "EXPN_FOOD_TOT_RDF_2019",
    "EXPN_FOOD_TOT_RDF_2021",
    "EXPN_HOUS_TOT_ND_1999",
    "EXPN_HOUS_TOT_ND_2001",
    "EXPN_HOUS_TOT_ND_2003",
    "EXPN_HOUS_TOT_ND_2005",
    "EXPN_HOUS_TOT_ND_2007",
    "EXPN_HOUS_TOT_ND_2009",
    "EXPN_HOUS_TOT_ND_2011",
    "EXPN_HOUS_TOT_ND_2013",
    "EXPN_HOUS_TOT_ND_2015",
    "EXPN_HOUS_TOT_ND_2017",
    "EXPN_HOUS_TOT_ND_2019",
    "EXPN_HOUS_TOT_ND_2021",
    "EXPN_HOUS_TOT_NDF_1999",
    "EXPN_HOUS_TOT_NDF_2001",
    "EXPN_HOUS_TOT_NDF_2003",
    "EXPN_HOUS_TOT_NDF_2005",
    "EXPN_HOUS_TOT_NDF_2007",
    "EXPN_HOUS_TOT_NDF_2009",
    "EXPN_HOUS_TOT_NDF_2011",
    "EXPN_HOUS_TOT_NDF_2013",
    "EXPN_HOUS_TOT_NDF_2015",
    "EXPN_HOUS_TOT_NDF_2017",
    "EXPN_HOUS_TOT_NDF_2019",
    "EXPN_HOUS_TOT_NDF_2021",
    "EXPN_HOUS_TOT_RD_1999",
    "EXPN_HOUS_TOT_RD_2001",
    "EXPN_HOUS_TOT_RD_2003",
    "EXPN_HOUS_TOT_RD_2005",
    "EXPN_HOUS_TOT_RD_2007",
    "EXPN_HOUS_TOT_RD_2009",
    "EXPN_HOUS_TOT_RD_2011",
    "EXPN_HOUS_TOT_RD_2013",
    "EXPN_HOUS_TOT_RD_2015",
    "EXPN_HOUS_TOT_RD_2017",
    "EXPN_HOUS_TOT_RD_2019",
    "EXPN_HOUS_TOT_RD_2021",
    "EXPN_HOUS_TOT_RDF_1999",
    "EXPN_HOUS_TOT_RDF_2001",
    "EXPN_HOUS_TOT_RDF_2003",
    "EXPN_HOUS_TOT_RDF_2005",
    "EXPN_HOUS_TOT_RDF_2007",
    "EXPN_HOUS_TOT_RDF_2009",
    "EXPN_HOUS_TOT_RDF_2011",
    "EXPN_HOUS_TOT_RDF_2013",
    "EXPN_HOUS_TOT_RDF_2015",
    "EXPN_HOUS_TOT_RDF_2017",
    "EXPN_HOUS_TOT_RDF_2019",
    "EXPN_HOUS_TOT_RDF_2021",
    "EXPN_TRAN_TOT_ND_1999",
    "EXPN_TRAN_TOT_ND_2001",
    "EXPN_TRAN_TOT_ND_2003",
    "EXPN_TRAN_TOT_ND_2005",
    "EXPN_TRAN_TOT_ND_2007",
    "EXPN_TRAN_TOT_ND_2009",
    "EXPN_TRAN_TOT_ND_2011",
    "EXPN_TRAN_TOT_ND_2013",
    "EXPN_TRAN_TOT_ND_2015",
    "EXPN_TRAN_TOT_ND_2017",
    "EXPN_TRAN_TOT_ND_2019",
    "EXPN_TRAN_TOT_ND_2021",
    "EXPN_TRAN_TOT_NDF_1999",
    "EXPN_TRAN_TOT_NDF_2001",
    "EXPN_TRAN_TOT_NDF_2003",
    "EXPN_TRAN_TOT_NDF_2005",
    "EXPN_TRAN_TOT_NDF_2007",
    "EXPN_TRAN_TOT_NDF_2009",
    "EXPN_TRAN_TOT_NDF_2011",
    "EXPN_TRAN_TOT_NDF_2013",
    "EXPN_TRAN_TOT_NDF_2015",
    "EXPN_TRAN_TOT_NDF_2017",
    "EXPN_TRAN_TOT_NDF_2019",
    "EXPN_TRAN_TOT_NDF_2021",
    "EXPN_TRAN_TOT_RD_1999",
    "EXPN_TRAN_TOT_RD_2001",
    "EXPN_TRAN_TOT_RD_2003",
    "EXPN_TRAN_TOT_RD_2005",
    "EXPN_TRAN_TOT_RD_2007",
    "EXPN_TRAN_TOT_RD_2009",
    "EXPN_TRAN_TOT_RD_2011",
    "EXPN_TRAN_TOT_RD_2013",
    "EXPN_TRAN_TOT_RD_2015",
    "EXPN_TRAN_TOT_RD_2017",
    "EXPN_TRAN_TOT_RD_2019",
    "EXPN_TRAN_TOT_RD_2021",
    "EXPN_TRAN_TOT_RDF_1999",
    "EXPN_TRAN_TOT_RDF_2001",
    "EXPN_TRAN_TOT_RDF_2003",
    "EXPN_TRAN_TOT_RDF_2005",
    "EXPN_TRAN_TOT_RDF_2007",
    "EXPN_TRAN_TOT_RDF_2009",
    "EXPN_TRAN_TOT_RDF_2011",
    "EXPN_TRAN_TOT_RDF_2013",
    "EXPN_TRAN_TOT_RDF_2015",
    "EXPN_TRAN_TOT_RDF_2017",
    "EXPN_TRAN_TOT_RDF_2019",
    "EXPN_TRAN_TOT_RDF_2021",
    "EXPN_TRIP_TOT_ND_2005",
    "EXPN_TRIP_TOT_ND_2007",
    "EXPN_TRIP_TOT_ND_2009",
    "EXPN_TRIP_TOT_ND_2011",
    "EXPN_TRIP_TOT_ND_2013",
    "EXPN_TRIP_TOT_ND_2015",
    "EXPN_TRIP_TOT_ND_2017",
    "EXPN_TRIP_TOT_ND_2019",
    "EXPN_TRIP_TOT_ND_2021",
    "EXPN_TRIP_TOT_NDF_2005",
    "EXPN_TRIP_TOT_NDF_2007",
    "EXPN_TRIP_TOT_NDF_2009",
    "EXPN_TRIP_TOT_NDF_2011",
    "EXPN_TRIP_TOT_NDF_2013",
    "EXPN_TRIP_TOT_NDF_2015",
    "EXPN_TRIP_TOT_NDF_2017",
    "EXPN_TRIP_TOT_NDF_2019",
    "EXPN_TRIP_TOT_NDF_2021",
    "EXPN_TRIP_TOT_RD_2005",
    "EXPN_TRIP_TOT_RD_2007",
    "EXPN_TRIP_TOT_RD_2009",
    "EXPN_TRIP_TOT_RD_2011",
    "EXPN_TRIP_TOT_RD_2013",
    "EXPN_TRIP_TOT_RD_2015",
    "EXPN_TRIP_TOT_RD_2017",
    "EXPN_TRIP_TOT_RD_2019",
    "EXPN_TRIP_TOT_RD_2021",
    "EXPN_TRIP_TOT_RDF_2005",
    "EXPN_TRIP_TOT_RDF_2007",
    "EXPN_TRIP_TOT_RDF_2009",
    "EXPN_TRIP_TOT_RDF_2011",
    "EXPN_TRIP_TOT_RDF_2013",
    "EXPN_TRIP_TOT_RDF_2015",
    "EXPN_TRIP_TOT_RDF_2017",
    "EXPN_TRIP_TOT_RDF_2019",
    "EXPN_TRIP_TOT_RDF_2021"
  ],
  "expenditures_personal": [
    "ID",
    "LINEAGE",
    "PNUM",
    "EXPN_CCAR_TOT_ND_1999",
    "EXPN_CCAR_TOT_ND_2001",
    "EXPN_CCAR_TOT_ND_2003",
    "EXPN_CCAR_TOT_ND_2005",
    "EXPN_CCAR_TOT_ND_2007",
    "EXPN_CCAR_TOT_ND_2009",
    "EXPN_CCAR_TOT_ND_2011",
    "EXPN_CCAR_TOT_ND_2013",
    "EXPN_CCAR_TOT_ND_2015",
    "EXPN_CCAR_TOT_ND_2017",
    "EXPN_CCAR_TOT_ND_2019",
    "EXPN_CCAR_TOT_ND_2021",
    "EXPN_CCAR_TOT_NDF_1999",
    "EXPN_CCAR_TOT_NDF_2001",
    "EXPN_CCAR_TOT_NDF_2003",
    "EXPN_CCAR_TOT_NDF_2005",
    "EXPN_CCAR_TOT_NDF_2007",
    "EXPN_CCAR_TOT_NDF_2009",
    "EXPN_CCAR_TOT_NDF_2011",
    "EXPN_CCAR_TOT_NDF_2013",
    "EXPN_CCAR_TOT_NDF_2015",
    "EXPN_CCAR_TOT_NDF_2017",
    "EXPN_CCAR_TOT_NDF_2019",
    "EXPN_CCAR_TOT_NDF_2021",
    "EXPN_CCAR_TOT_RD_1999",
    "EXPN_CCAR_TOT_RD_2001",
    "EXPN_CCAR_TOT_RD_2003",
    "EXPN_CCAR_TOT_RD_2005",
    "EXPN_CCAR_TOT_RD_2007",
    "EXPN_CCAR_TOT_RD_2009",
    "EXPN_CCAR_TOT_RD_2011",
    "EXPN_CCAR_TOT_RD_2013",
    "EXPN_CCAR_TOT_RD_2015",
    "EXPN_CCAR_TOT_RD_2017",
    "EXPN_CCAR_TOT_RD_2019",
    "EXPN_CCAR_TOT_RD_2021",
    "EXPN_CCAR_TOT_RDF_1999",
    "EXPN_CCAR_TOT_RDF_2001",
    "EXPN_CCAR_TOT_RDF_2003",
    "EXPN_CCAR_TOT_RDF_2005",
    "EXPN_CCAR_TOT_RDF_2007",
    "EXPN_CCAR_TOT_RDF_2009",
    "EXPN_CCAR_TOT_RDF_2011",
    "EXPN_CCAR_TOT_RDF_2013",
    "EXPN_CCAR_TOT_RDF_2015",
    "EXPN_CCAR_TOT_RDF_2017",
    "EXPN_CCAR_TOT_RDF_2019",
    "EXPN_CCAR_TOT_RDF_2021",
    "EXPN_CLOT_TOT_ND_2005",
    "EXPN_CLOT_TOT_ND_2007",
    "EXPN_CLOT_TOT_ND_2009",
    "EXPN_CLOT_TOT_ND_2011",
    "EXPN_CLOT_TOT_ND_2013",
    "EXPN_CLOT_TOT_ND_2015",
    "EXPN_CLOT_TOT_ND_2017",
    "EXPN_CLOT_TOT_ND_2019",
    "EXPN_CLOT_TOT_ND_2021",
    "EXPN_CLOT_TOT_NDF_2005",
    "EXPN_CLOT_TOT_NDF_2007",
    "EXPN_CLOT_TOT_NDF_2009",
    "EXPN_CLOT_TOT_NDF_2011",
    "EXPN_CLOT_TOT_NDF_2013",
    "EXPN_CLOT_TOT_NDF_2015",
    "EXPN_CLOT_TOT_NDF_2017",
    "EXPN_CLOT_TOT_NDF_2019",
    "EXPN_CLOT_TOT_NDF_2021",
    "EXPN_CLOT_TOT_RD_2005",
    "EXPN_CLOT_TOT_RD_2007",
    "EXPN_CLOT_TOT_RD_2009",
    "EXPN_CLOT_TOT_RD_2011",
    "EXPN_CLOT_TOT_RD_2013",
    "EXPN_CLOT_TOT_RD_2015",
    "EXPN_CLOT_TOT_RD_2017",
    "EXPN_CLOT_TOT_RD_2019",
    "EXPN_CLOT_TOT_RD_2021",
    "EXPN_CLOT_TOT_RDF_2005",
    "EXPN_CLOT_TOT_RDF_2007",
    "EXPN_CLOT_TOT_RDF_2009",
    "EXPN_CLOT_TOT_RDF_2011",
    "EXPN_CLOT_TOT_RDF_2013",
    "EXPN_CLOT_TOT_RDF_2015",
    "EXPN_CLOT_TOT_RDF_2017",
    "EXPN_CLOT_TOT_RDF_2019",
    "EXPN_CLOT_TOT_RDF_2021",
    "EXPN_COMP_TOT_ND_2017",
    "EXPN_COMP_TOT_ND_2019",
    "EXPN_COMP_TOT_ND_2021",
    "EXPN_COMP_TOT_NDF_2017",
    "EXPN_COMP_TOT_NDF_2019",
    "EXPN_COMP_TOT_NDF_2021",
    "EXPN_COMP_TOT_RD_2017",
    "EXPN_COMP_TOT_RD_2019",
    "EXPN_COMP_TOT_RD_2021",
    "EXPN_COMP_TOT_RDF_2017",
    "EXPN_COMP_TOT_RDF_2019",
    "EXPN_COMP_TOT_RDF_2021",
    "EXPN_EDUC_TOT_ND_1999",
    "EXPN_EDUC_TOT_ND_2001",
    "EXPN_EDUC_TOT_ND_2003",
    "EXPN_EDUC_TOT_ND_2005",
    "EXPN_EDUC_TOT_ND_2007",
    "EXPN_EDUC_TOT_ND_2009",
    "EXPN_EDUC_TOT_ND_2011",
    "EXPN_EDUC_TOT_ND_2013",
    "EXPN_EDUC_TOT_ND_2015",
    "EXPN_EDUC_TOT_ND_2017",
    "EXPN_EDUC_TOT_ND_2019",
    "EXPN_EDUC_TOT_ND_2021",
    "EXPN_EDUC_TOT_NDF_1999",
    "EXPN_EDUC_TOT_NDF_2001",
    "EXPN_EDUC_TOT_NDF_2003",
    "EXPN_EDUC_TOT_NDF_2005",
    "EXPN_EDUC_TOT_NDF_2007",
    "EXPN_EDUC_TOT_NDF_2009",
    "EXPN_EDUC_TOT_NDF_2011",
    "EXPN_EDUC_TOT_NDF_2013",
    "EXPN_EDUC_TOT_NDF_2015",
    "EXPN_EDUC_TOT_NDF_2017",
    "EXPN_EDUC_TOT_NDF_2019",
    "EXPN_EDUC_TOT_NDF_2021",
    "EXPN_EDUC_TOT_RD_1999",
    "EXPN_EDUC_TOT_RD_2001",
    "EXPN_EDUC_TOT_RD_2003",
    "EXPN_EDUC_TOT_RD_2005",
    "EXPN_EDUC_TOT_RD_2007",
    "EXPN_EDUC_TOT_RD_2009",
    "EXPN_EDUC_TOT_RD_2011",
    "EXPN_EDUC_TOT_RD_2013",
    "EXPN_EDUC_TOT_RD_2015",
    "EXPN_EDUC_TOT_RD_2017",
    "EXPN_EDUC_TOT_RD_2019",
    "EXPN_EDUC_TOT_RD_2021",
    "EXPN_EDUC_TOT_RDF_1999",
    "EXPN_EDUC_TOT_RDF_2001",
    "EXPN_EDUC_TOT_RDF_2003",
    "EXPN_EDUC_TOT_RDF_2005",
    "EXPN_EDUC_TOT_RDF_2007",
    "EXPN_EDUC_TOT_RDF_2009",
    "EXPN_EDUC_TOT_RDF_2011",
    "EXPN_EDUC_TOT_RDF_2013",
    "EXPN_EDUC_TOT_RDF_2015",
    "EXPN_EDUC_TOT_RDF_2017",
    "EXPN_EDUC_TOT_RDF_2019",
    "EXPN_EDUC_TOT_RDF_2021",
    "EXPN_OREC_TOT_ND_2005",
    "EXPN_OREC_TOT_ND_2007",
    "EXPN_OREC_TOT_ND_2009",
    "EXPN_OREC_TOT_ND_2011",
    "EXPN_OREC_TOT_ND_2013",
    "EXPN_OREC_TOT_ND_2015",
    "EXPN_OREC_TOT_ND_2017",
    "EXPN_OREC_TOT_ND_2019",
    "EXPN_OREC_TOT_ND_2021",
    "EXPN_OREC_TOT_NDF_2005",
    "EXPN_OREC_TOT_NDF_2007",
    "EXPN_OREC_TOT_NDF_2009",
    "EXPN_OREC_TOT_NDF_2011",
    "EXPN_OREC_TOT_NDF_2013",
    "EXPN_OREC_TOT_NDF_2015",
    "EXPN_OREC_TOT_NDF_2017",
    "EXPN_OREC_TOT_NDF_2019",
    "EXPN_OREC_TOT_NDF_2021",
    "EXPN_OREC_TOT_RD_2005",
    "EXPN_OREC_TOT_RD_2007",
    "EXPN_OREC_TOT_RD_2009",
    "EXPN_OREC_TOT_RD_2011",
    "EXPN_OREC_TOT_RD_2013",
    "EXPN_OREC_TOT_RD_2015",
    "EXPN_OREC_TOT_RD_2017",
    "EXPN_OREC_TOT_RD_2019",
    "EXPN_OREC_TOT_RD_2021",
    "EXPN_OREC_TOT_RDF_2005",
    "EXPN_OREC_TOT_RDF_2007",
    "EXPN_OREC_TOT_RDF_2009",
    "EXPN_OREC_TOT_RDF_2011",
    "EXPN_OREC_TOT_RDF_2013",
    "EXPN_OREC_TOT_RDF_2015",
    "EXPN_OREC_TOT_RDF_2017",
    "EXPN_OREC_TOT_RDF_2019",
    "EXPN_OREC_TOT_RDF_2021"
  ],
  "expenditures_totals": [
    "ID",
    "LINEAGE",
    "PNUM",
    "EXPN_TOT_ND_1999",
    "EXPN_TOT_ND_2001",
    "EXPN_TOT_ND_2003",
    "EXPN_TOT_ND_2005",
    "EXPN_TOT_ND_2007",
    "EXPN_TOT_ND_2009",
    "EXPN_TOT_ND_2011",
    "EXPN_TOT_ND_2013",
    "EXPN_TOT_ND_2015",
    "EXPN_TOT_ND_2017",
    "EXPN_TOT_ND_2019",
    "EXPN_TOT_ND_2021",
    "EXPN_TOT_NDF_1999",
    "EXPN_TOT_NDF_2001",
    "EXPN_TOT_NDF_2003",
    "EXPN_TOT_NDF_2005",
    "EXPN_TOT_NDF_2007",
    "EXPN_TOT_NDF_2009",
    "EXPN_TOT_NDF_2011",
    "EXPN_TOT_NDF_2013",
    "EXPN_TOT_NDF_2015",
    "EXPN_TOT_NDF_2017",
    "EXPN_TOT_NDF_2019",
    "EXPN_TOT_NDF_2021",
    "EXPN_TOT_RD_1999",
    "EXPN_TOT_RD_2001",
    "EXPN_TOT_RD_2003",
    "EXPN_TOT_RD_2005",
    "EXPN_TOT_RD_2007",
    "EXPN_TOT_RD_2009",
    "EXPN_TOT_RD_2011",
    "EXPN_TOT_RD_2013",
    "EXPN_TOT_RD_2015",
    "EXPN_TOT_RD_2017",
    "EXPN_TOT_RD_2019",
    "EXPN_TOT_RD_2021",
    "EXPN_TOT_RDF_1999",
    "EXPN_TOT_RDF_2001",
    "EXPN_TOT_RDF_2003",
    "EXPN_TOT_RDF_2005",
    "EXPN_TOT_RDF_2007",
    "EXPN_TOT_RDF_2009",
    "EXPN_TOT_RDF_2011",
    "EXPN_TOT_RDF_2013",
    "EXPN_TOT_RDF_2015",
    "EXPN_TOT_RDF_2017",
    "EXPN_TOT_RDF_2019",
    "EXPN_TOT_RDF_2021",
    "EXPN_TOT_IRV_ND_2017",
    "EXPN_TOT_IRV_ND_2019",
    "EXPN_TOT_IRV_ND_2021",
    "EXPN_TOT_IRV_NDF_2017",
    "EXPN_TOT_IRV_NDF_2019",
    "EXPN_TOT_IRV_NDF_2021",
    "EXPN_TOT_IRV_RD_2017",
    "EXPN_TOT_IRV_RD_2019",
    "EXPN_TOT_IRV_RD_2021",
    "EXPN_TOT_IRV_RDF_2017",
    "EXPN_TOT_IRV_RDF_2019",
    "EXPN_TOT_IRV_RDF_2021"
  ],
  "family_income": [
    "ID",
    "LINEAGE",
    "PNUM",
    "FINC_TOT_ND_1968",
    "FINC_TOT_ND_1969",
    "FINC_TOT_ND_1970",
    "FINC_TOT_ND_1971",
    "FINC_TOT_ND_1972",
    "FINC_TOT_ND_1973",
    "FINC_TOT_ND_1974",
    "FINC_TOT_ND_1975",
    "FINC_TOT_ND_1976",
    "FINC_TOT_ND_1977",
    "FINC_TOT_ND_1978",
    "FINC_TOT_ND_1979",
    "FINC_TOT_ND_1980",
    "FINC_TOT_ND_1981",
    "FINC_TOT_ND_1982",
    "FINC_TOT_ND_1983",
    "FINC_TOT_ND_1984",
    "FINC_TOT_ND_1985",
    "FINC_TOT_ND_1986",
    "FINC_TOT_ND_1987",
    "FINC_TOT_ND_1988",
    "FINC_TOT_ND_1989",
    "FINC_TOT_ND_1990",
    "FINC_TOT_ND_1991",
    "FINC_TOT_ND_1992",
    "FINC_TOT_ND_1993",
    "FINC_TOT_ND_1994",
    "FINC_TOT_ND_1995",
    "FINC_TOT_ND_1996",
    "FINC_TOT_ND_1997",
    "FINC_TOT_ND_1999",
    "FINC_TOT_ND_2001",
    "FINC_TOT_ND_2003",
    "FINC_TOT_ND_2005",
    "FINC_TOT_ND_2007",
    "FINC_TOT_ND_2009",
    "FINC_TOT_ND_2011",
    "FINC_TOT_ND_2013",
    "FINC_TOT_ND_2015",
    "FINC_TOT_ND_2017",
    "FINC_TOT_ND_2019",
    "FINC_TOT_ND_2021",
    "FINC_TOT_NDF_1968",
    "FINC_TOT_NDF_1969",
    "FINC_TOT_NDF_1970",
    "FINC_TOT_NDF_1971",
    "FINC_TOT_NDF_1972",
    "FINC_TOT_NDF_1973",
    "FINC_TOT_NDF_1974",
    "FINC_TOT_NDF_1975",
    "FINC_TOT_NDF_1976",
    "FINC_TOT_NDF_1977",
    "FINC_TOT_NDF_1978",
    "FINC_TOT_NDF_1979",
    "FINC_TOT_NDF_1980",
    "FINC_TOT_NDF_1981",
    "FINC_TOT_NDF_1982",
    "FINC_TOT_NDF_1983",
    "FINC_TOT_NDF_1984",
    "FINC_TOT_NDF_1985",
    "FINC_TOT_NDF_1986",
    "FINC_TOT_NDF_1987",
    "FINC_TOT_NDF_1988",
    "FINC_TOT_NDF_1989",
    "FINC_TOT_NDF_1990",
    "FINC_TOT_NDF_1991",
    "FINC_TOT_NDF_1992",
    "FINC_TOT_NDF_1993",
    "FINC_TOT_NDF_1994",
    "FINC_TOT_NDF_1995",
    "FINC_TOT_NDF_1996",
    "FINC_TOT_NDF_1997",
    "FINC_TOT_NDF_1999",
    "FINC_TOT_NDF_2001",
    "FINC_TOT_NDF_2003",
    "FINC_TOT_NDF_2005",
    "FINC_TOT_NDF_2007",
    "FINC_TOT_NDF_2009",
    "FINC_TOT_NDF_2011",
    "FINC_TOT_NDF_2013",
    "FINC_TOT_NDF_2015",
    "FINC_TOT_NDF_2017",
    "FINC_TOT_NDF_2019",
    "FINC_TOT_NDF_2021",
    "FINC_TOT_RD_1968",
    "FINC_TOT_RD_1969",
    "FINC_TOT_RD_1970",
    "FINC_TOT_RD_1971",
    "FINC_TOT_RD_1972",
    "FINC_TOT_RD_1973",
    "FINC_TOT_RD_1974",
    "FINC_TOT_RD_1975",
    "FINC_TOT_RD_1976",
    "FINC_TOT_RD_1977",
    "FINC_TOT_RD_1978",
    "FINC_TOT_RD_1979",
    "FINC_TOT_RD_1980",
    "FINC_TOT_RD_1981",
    "FINC_TOT_RD_1982",
    "FINC_TOT_RD_1983",
    "FINC_TOT_RD_1984",
    "FINC_TOT_RD_1985",
    "FINC_TOT_RD_1986",
    "FINC_TOT_RD_1987",
    "FINC_TOT_RD_1988",
    "FINC_TOT_RD_1989",
    "FINC_TOT_RD_1990",
    "FINC_TOT_RD_1991",
    "FINC_TOT_RD_1992",
    "FINC_TOT_RD_1993",
    "FINC_TOT_RD_1994",
    "FINC_TOT_RD_1995",
    "FINC_TOT_RD_1996",
    "FINC_TOT_RD_1997",
    "FINC_TOT_RD_1999",
    "FINC_TOT_RD_2001",
    "FINC_TOT_RD_2003",
    "FINC_TOT_RD_2005",
    "FINC_TOT_RD_2007",
    "FINC_TOT_RD_2009",
    "FINC_TOT_RD_2011",
    "FINC_TOT_RD_2013",
    "FINC_TOT_RD_2015",
    "FINC_TOT_RD_2017",
    "FINC_TOT_RD_2019",
    "FINC_TOT_RD_2021",
    "FINC_TOT_RDF_1968",
    "FINC_TOT_RDF_1969",
    "FINC_TOT_RDF_1970",
    "FINC_TOT_RDF_1971",
    "FINC_TOT_RDF_1972",
    "FINC_TOT_RDF_1973",
    "FINC_TOT_RDF_1974",
    "FINC_TOT_RDF_1975",
    "FINC_TOT_RDF_1976",
    "FINC_TOT_RDF_1977",
    "FINC_TOT_RDF_1978",
    "FINC_TOT_RDF_1979",
    "FINC_TOT_RDF_1980",
    "FINC_TOT_RDF_1981",
    "FINC_TOT_RDF_1982",
    "FINC_TOT_RDF_1983",
    "FINC_TOT_RDF_1984",
    "FINC_TOT_RDF_1985",
    "FINC_TOT_RDF_1986",
    "FINC_TOT_RDF_1987",
    "FINC_TOT_RDF_1988",
    "FINC_TOT_RDF_1989",
    "FINC_TOT_RDF_1990",
    "FINC_TOT_RDF_1991",
    "FINC_TOT_RDF_1992",
    "FINC_TOT_RDF_1993",
    "FINC_TOT_RDF_1994",
    "FINC_TOT_RDF_1995",
    "FINC_TOT_RDF_1996",
    "FINC_TOT_RDF_1997",
    "FINC_TOT_RDF_1999",
    "FINC_TOT_RDF_2001",
    "FINC_TOT_RDF_2003",
    "FINC_TOT_RDF_2005",
    "FINC_TOT_RDF_2007",
    "FINC_TOT_RDF_2009",
    "FINC_TOT_RDF_2011",
    "FINC_TOT_RDF_2013",
    "FINC_TOT_RDF_2015",
    "FINC_TOT_RDF_2017",
    "FINC_TOT_RDF_2019",
    "FINC_TOT_RDF_2021"
  ],
  "family_structure": [
    "ID",
    "LINEAGE",
    "PNUM",
    "FAM_SIZE_1968",
    "FAM_SIZE_1969",
    "FAM_SIZE_1970",
    "FAM_SIZE_1971",
    "FAM_SIZE_1972",
    "FAM_SIZE_1973",
    "FAM_SIZE_1974",
    "FAM_SIZE_1975",
    "FAM_SIZE_1976",
    "FAM_SIZE_1977",
    "FAM_SIZE_1978",
    "FAM_SIZE_1979",
    "FAM_SIZE_1980",
    "FAM_SIZE_1981",
    "FAM_SIZE_1982",
    "FAM_SIZE_1983",
    "FAM_SIZE_1984",
    "FAM_SIZE_1985",
    "FAM_SIZE_1986",
    "FAM_SIZE_1987",
    "FAM_SIZE_1988",
    "FAM_SIZE_1989",
    "FAM_SIZE_1990",
    "FAM_SIZE_1991",
    "FAM_SIZE_1992",
    "FAM_SIZE_1993",
    "FAM_SIZE_1994",
    "FAM_SIZE_1995",
    "FAM_SIZE_1996",
    "FAM_SIZE_1997",
    "FAM_SIZE_1999",
    "FAM_SIZE_2001",
    "FAM_SIZE_2003",
    "FAM_SIZE_2005",
    "FAM_SIZE_2007",
    "FAM_SIZE_2009",
    "FAM_SIZE_2011",
    "FAM_SIZE_2013",
    "FAM_SIZE_2015",
    "FAM_SIZE_2017",
    "FAM_SIZE_2019",
    "FAM_SIZE_2021",
    "FAM_SIZE_CHI_1968",
    "FAM_SIZE_CHI_1969",
    "FAM_SIZE_CHI_1970",
    "FAM_SIZE_CHI_1971",
    "FAM_SIZE_CHI_1972",
    "FAM_SIZE_CHI_1973",
    "FAM_SIZE_CHI_1974",
    "FAM_SIZE_CHI_1975",
    "FAM_SIZE_CHI_1976",
    "FAM_SIZE_CHI_1977",
    "FAM_SIZE_CHI_1978",
    "FAM_SIZE_CHI_1979",
    "FAM_SIZE_CHI_1980",
    "FAM_SIZE_CHI_1981",
    "FAM_SIZE_CHI_1982",
    "FAM_SIZE_CHI_1983",
    "FAM_SIZE_CHI_1984",
    "FAM_SIZE_CHI_1985",
    "FAM_SIZE_CHI_1986",
    "FAM_SIZE_CHI_1987",
    "FAM_SIZE_CHI_1988",
    "FAM_SIZE_CHI_1989",
    "FAM_SIZE_CHI_1990",
    "FAM_SIZE_CHI_1991",
    "FAM_SIZE_CHI_1992",
    "FAM_SIZE_CHI_1993",
    "FAM_SIZE_CHI_1994",
    "FAM_SIZE_CHI_1995",
    "FAM_SIZE_CHI_1996",
    "FAM_SIZE_CHI_1997",
    "FAM_SIZE_CHI_1999",
    "FAM_SIZE_CHI_2001",
    "FAM_SIZE_CHI_2003",
    "FAM_SIZE_CHI_2005",
    "FAM_SIZE_CHI_2007",
    "FAM_SIZE_CHI_2009",
    "FAM_SIZE_CHI_2011",
    "FAM_SIZE_CHI_2013",
    "FAM_SIZE_CHI_2015",
    "FAM_SIZE_CHI_2017",
    "FAM_SIZE_CHI_2019",
    "FAM_SIZE_CHI_2021",
    "FAM_PARTNERED_1968",
    "FAM_PARTNERED_1969",
    "FAM_PARTNERED_1970",
    "FAM_PARTNERED_1971",
    "FAM_PARTNERED_1972",
    "FAM_PARTNERED_1973",
    "FAM_PARTNERED_1974",
    "FAM_PARTNERED_1975",
    "FAM_PARTNERED_1976",
    "FAM_PARTNERED_1977",
    "FAM_PARTNERED_1978",
    "FAM_PARTNERED_1979",
    "FAM_PARTNERED_1980",
    "FAM_PARTNERED_1981",
    "FAM_PARTNERED_1982",
    "FAM_PARTNERED_1983",
    "FAM_PARTNERED_1984",
    "FAM_PARTNERED_1985",
    "FAM_PARTNERED_1986",
    "FAM_PARTNERED_1987",
    "FAM_PARTNERED_1988",
    "FAM_PARTNERED_1989",
    "FAM_PARTNERED_1990",
    "FAM_PARTNERED_1991",
    "FAM_PARTNERED_1992",
    "FAM_PARTNERED_1993",
    "FAM_PARTNERED_1994",
    "FAM_PARTNERED_1995",
    "FAM_PARTNERED_1996",
    "FAM_PARTNERED_1997",
    "FAM_PARTNERED_1999",
    "FAM_PARTNERED_2001",
    "FAM_PARTNERED_2003",
    "FAM_PARTNERED_2005",
    "FAM_PARTNERED_2007",
    "FAM_PARTNERED_2009",
    "FAM_PARTNERED_2011",
    "FAM_PARTNERED_2013",
    "FAM_PARTNERED_2015",
    "FAM_PARTNERED_2017",
    "FAM_PARTNERED_2019",
    "FAM_PARTNERED_2021",
    "FAM_PARSTAT_1968",
    "FAM_PARSTAT_1969",
    "FAM_PARSTAT_1970",
    "FAM_PARSTAT_1971",
    "FAM_PARSTAT_1972",
    "FAM_PARSTAT_1973",
    "FAM_PARSTAT_1974",
    "FAM_PARSTAT_1975",
    "FAM_PARSTAT_1976",
    "FAM_PARSTAT_1977",
    "FAM_PARSTAT_1978",
    "FAM_PARSTAT_1979",
    "FAM_PARSTAT_1980",
    "FAM_PARSTAT_1981",
    "FAM_PARSTAT_1982",
    "FAM_PARSTAT_1983",
    "FAM_PARSTAT_1984",
    "FAM_PARSTAT_1985",
    "FAM_PARSTAT_1986",
    "FAM_PARSTAT_1987",
    "FAM_PARSTAT_1988",
    "FAM_PARSTAT_1989",
    "FAM_PARSTAT_1990",
    "FAM_PARSTAT_1991",
    "FAM_PARSTAT_1992",
    "FAM_PARSTAT_1993",
    "FAM_PARSTAT_1994",
    "FAM_PARSTAT_1995",
    "FAM_PARSTAT_1996",
    "FAM_PARSTAT_1997",
    "FAM_PARSTAT_1999",
    "FAM_PARSTAT_2001",
    "FAM_PARSTAT_2003",
    "FAM_PARSTAT_2005",
    "FAM_PARSTAT_2007",
    "FAM_PARSTAT_2009",
    "FAM_PARSTAT_2011",
    "FAM_PARSTAT_2013",
    "FAM_PARSTAT_2015",
    "FAM_PARSTAT_2017",
    "FAM_PARSTAT_2019",
    "FAM_PARSTAT_2021",
    "FAM_PARTYPE_1983",
    "FAM_PARTYPE_1984",
    "FAM_PARTYPE_1985",
    "FAM_PARTYPE_1986",
    "FAM_PARTYPE_1987",
    "FAM_PARTYPE_1988",
    "FAM_PARTYPE_1989",
    "FAM_PARTYPE_1990",
    "FAM_PARTYPE_1991",
    "FAM_PARTYPE_1992",
    "FAM_PARTYPE_1993",
    "FAM_PARTYPE_1994",
    "FAM_PARTYPE_1995",
    "FAM_PARTYPE_1996",
    "FAM_PARTYPE_1997",
    "FAM_PARTYPE_1999",
    "FAM_PARTYPE_2001",
    "FAM_PARTYPE_2003",
    "FAM_PARTYPE_2005",
    "FAM_PARTYPE_2007",
    "FAM_PARTYPE_2009",
    "FAM_PARTYPE_2011",
    "FAM_PARTYPE_2013",
    "FAM_PARTYPE_2015",
    "FAM_PARTYPE_2017",
    "FAM_PARTYPE_2019",
    "FAM_PARTYPE_2021",
    "FAM_MARRIED_1978",
    "FAM_MARRIED_1979",
    "FAM_MARRIED_1980",
    "FAM_MARRIED_1981",
    "FAM_MARRIED_1982",
    "FAM_MARRIED_1983",
    "FAM_MARRIED_1984",
    "FAM_MARRIED_1985",
    "FAM_MARRIED_1986",
    "FAM_MARRIED_1987",
    "FAM_MARRIED_1988",
    "FAM_MARRIED_1989",
    "FAM_MARRIED_1990",
    "FAM_MARRIED_1991",
    "FAM_MARRIED_1992",
    "FAM_MARRIED_1993",
    "FAM_MARRIED_1994",
    "FAM_MARRIED_1995",
    "FAM_MARRIED_1996",
    "FAM_MARRIED_1997",
    "FAM_MARRIED_1999",
    "FAM_MARRIED_2001",
    "FAM_MARRIED_2003",
    "FAM_MARRIED_2005",
    "FAM_MARRIED_2007",
    "FAM_MARRIED_2009",
    "FAM_MARRIED_2011",
    "FAM_MARRIED_2013",
    "FAM_MARRIED_2015",
    "FAM_MARRIED_2017",
    "FAM_MARRIED_2019",
    "FAM_MARRIED_2021",
    "FAM_MARSTAT_1977",
    "FAM_MARSTAT_1978",
    "FAM_MARSTAT_1979",
    "FAM_MARSTAT_1980",
    "FAM_MARSTAT_1981",
    "FAM_MARSTAT_1982",
    "FAM_MARSTAT_1983",
    "FAM_MARSTAT_1984",
    "FAM_MARSTAT_1985",
    "FAM_MARSTAT_1986",
    "FAM_MARSTAT_1987",
    "FAM_MARSTAT_1988",
    "FAM_MARSTAT_1989",
    "FAM_MARSTAT_1990",
    "FAM_MARSTAT_1991",
    "FAM_MARSTAT_1992",
    "FAM_MARSTAT_1993",
    "FAM_MARSTAT_1994",
    "FAM_MARSTAT_1995",
    "FAM_MARSTAT_1996",
    "FAM_MARSTAT_1997",
    "FAM_MARSTAT_1999",
    "FAM_MARSTAT_2001",
    "FAM_MARSTAT_2003",
    "FAM_MARSTAT_2005",
    "FAM_MARSTAT_2007",
    "FAM_MARSTAT_2009",
    "FAM_MARSTAT_2011",
    "FAM_MARSTAT_2013",
    "FAM_MARSTAT_2015",
    "FAM_MARSTAT_2017",
    "FAM_MARSTAT_2019",
    "FAM_MARSTAT_2021"
  ],
  "geography": [
    "ID",
    "LINEAGE",
    "PNUM",
    "GEO_REGION_1968",
    "GEO_REGION_1969",
    "GEO_REGION_1970",
    "GEO_REGION_1971",
    "GEO_REGION_1972",
    "GEO_REGION_1973",
    "GEO_REGION_1974",
    "GEO_REGION_1975",
    "GEO_REGION_1976",
    "GEO_REGION_1977",
    "GEO_REGION_1978",
    "GEO_REGION_1979",
    "GEO_REGION_1980",
    "GEO_REGION_1981",
    "GEO_REGION_1982",
    "GEO_REGION_1983",
    "GEO_REGION_1984",
    "GEO_REGION_1985",
    "GEO_REGION_1986",
    "GEO_REGION_1987",
    "GEO_REGION_1988",
    "GEO_REGION_1989",
    "GEO_REGION_1990",
    "GEO_REGION_1991",
    "GEO_REGION_1992",
    "GEO_REGION_1993",
    "GEO_REGION_1994",
    "GEO_REGION_1995",
    "GEO_REGION_1996",
    "GEO_REGION_1997",
    "GEO_REGION_1999",
    "GEO_REGION_2001",
    "GEO_REGION_2003",
    "GEO_REGION_2005",
    "GEO_REGION_2007",
    "GEO_REGION_2009",
    "GEO_REGION_2011",
    "GEO_REGION_2013",
    "GEO_REGION_2015",
    "GEO_REGION_2017",
    "GEO_REGION_2019",
    "GEO_REGION_2021",
    "GEO_STATE_1968",
    "GEO_STATE_1969",
    "GEO_STATE_1970",
    "GEO_STATE_1971",
    "GEO_STATE_1972",
    "GEO_STATE_1973",
    "GEO_STATE_1974",
    "GEO_STATE_1975",
    "GEO_STATE_1976",
    "GEO_STATE_1977",
    "GEO_STATE_1978",
    "GEO_STATE_1979",
    "GEO_STATE_1980",
    "GEO_STATE_1981",
    "GEO_STATE_1982",
    "GEO_STATE_1983",
    "GEO_STATE_1984",
    "GEO_STATE_1985",
    "GEO_STATE_1986",
    "GEO_STATE_1987",
    "GEO_STATE_1988",
    "GEO_STATE_1989",
    "GEO_STATE_1990",
    "GEO_STATE_1991",
    "GEO_STATE_1992",
    "GEO_STATE_1993",
    "GEO_STATE_1994",
    "GEO_STATE_1995",
    "GEO_STATE_1996",
    "GEO_STATE_1997",
    "GEO_STATE_1999",
    "GEO_STATE_2001",
    "GEO_STATE_2003",
    "GEO_STATE_2005",
    "GEO_STATE_2007",
    "GEO_STATE_2009",
    "GEO_STATE_2011",
    "GEO_STATE_2013",
    "GEO_STATE_2015",
    "GEO_STATE_2017",
    "GEO_STATE_2019",
    "GEO_STATE_2021",
    "GEO_METRO_2015",
    "GEO_METRO_2017",
    "GEO_METRO_2019",
    "GEO_METRO_2021"
  ],
  "health_chronic": [
    "ID",
    "LINEAGE",
    "PNUM",
    "CCON_ARTH_DIAG_ANY_1999",
    "CCON_ARTH_DIAG_ANY_2001",
    "CCON_ARTH_DIAG_ANY_2003",
    "CCON_ARTH_DIAG_ANY_2005",
    "CCON_ARTH_DIAG_ANY_2007",
    "CCON_ARTH_DIAG_ANY_2009",
    "CCON_ARTH_DIAG_ANY_2011",
    "CCON_ARTH_DIAG_ANY_2013",
    "CCON_ARTH_DIAG_ANY_2015",
    "CCON_ARTH_DIAG_ANY_2017",
    "CCON_ARTH_DIAG_ANY_2019",
    "CCON_ARTH_DIAG_ANY_2021",
    "CCON_ARTH_DIAG_AGE_2005",
    "CCON_ARTH_DIAG_AGE_2007",
    "CCON_ARTH_DIAG_AGE_2009",
    "CCON_ARTH_DIAG_AGE_2011",
    "CCON_ARTH_DIAG_AGE_2013",
    "CCON_ARTH_DIAG_AGE_2015",
    "CCON_ARTH_DIAG_AGE_2017",
    "CCON_ARTH_DIAG_AGE_2019",
    "CCON_ARTH_DIAG_AGE_2021",
    "CCON_ARTH_DEGR_1999",
    "CCON_ARTH_DEGR_2001",
    "CCON_ARTH_DEGR_2003",
    "CCON_ARTH_DEGR_2005",
    "CCON_ARTH_DEGR_2007",
    "CCON_ARTH_DEGR_2009",
    "CCON_ARTH_DEGR_2011",
    "CCON_ARTH_DEGR_2013",
    "CCON_ARTH_DEGR_2015",
    "CCON_ARTH_DEGR_2017",
    "CCON_ARTH_DEGR_2019",
    "CCON_ARTH_DEGR_2021",
    "CCON_ARTH_CARE_2011",
    "CCON_ARTH_CARE_2013",
    "CCON_ARTH_CARE_2015",
    "CCON_ARTH_CARE_2017",
    "CCON_ARTH_CARE_2019",
    "CCON_ARTH_CARE_2021",
    "CCON_ASTH_DIAG_ANY_1999",
    "CCON_ASTH_DIAG_ANY_2001",
    "CCON_ASTH_DIAG_ANY_2003",
    "CCON_ASTH_DIAG_ANY_2005",
    "CCON_ASTH_DIAG_ANY_2007",
    "CCON_ASTH_DIAG_ANY_2009",
    "CCON_ASTH_DIAG_ANY_2011",
    "CCON_ASTH_DIAG_ANY_2013",
    "CCON_ASTH_DIAG_ANY_2015",
    "CCON_ASTH_DIAG_ANY_2017",
    "CCON_ASTH_DIAG_ANY_2019",
    "CCON_ASTH_DIAG_ANY_2021",
    "CCON_ASTH_DIAG_AGE_2005",
    "CCON_ASTH_DIAG_AGE_2007",
    "CCON_ASTH_DIAG_AGE_2009",
    "CCON_ASTH_DIAG_AGE_2011",
    "CCON_ASTH_DIAG_AGE_2013",
    "CCON_ASTH_DIAG_AGE_2015",
    "CCON_ASTH_DIAG_AGE_2017",
    "CCON_ASTH_DIAG_AGE_2019",
    "CCON_ASTH_DIAG_AGE_2021",
    "CCON_ASTH_DEGR_1999",
    "CCON_ASTH_DEGR_2001",
    "CCON_ASTH_DEGR_2003",
    "CCON_ASTH_DEGR_2005",
    "CCON_ASTH_DEGR_2007",
    "CCON_ASTH_DEGR_2009",
    "CCON_ASTH_DEGR_2011",
    "CCON_ASTH_DEGR_2013",
    "CCON_ASTH_DEGR_2015",
    "CCON_ASTH_DEGR_2017",
    "CCON_ASTH_DEGR_2019",
    "CCON_ASTH_DEGR_2021",
    "CCON_ASTH_CARE_2011",
    "CCON_ASTH_CARE_2013",
    "CCON_ASTH_CARE_2015",
    "CCON_ASTH_CARE_2017",
    "CCON_ASTH_CARE_2019",
    "CCON_ASTH_CARE_2021",
    "CCON_CANC_DIAG_ANY_1999",
    "CCON_CANC_DIAG_ANY_2001",
    "CCON_CANC_DIAG_ANY_2003",
    "CCON_CANC_DIAG_ANY_2005",
    "CCON_CANC_DIAG_ANY_2007",
    "CCON_CANC_DIAG_ANY_2009",
    "CCON_CANC_DIAG_ANY_2011",
    "CCON_CANC_DIAG_ANY_2013",
    "CCON_CANC_DIAG_ANY_2015",
    "CCON_CANC_DIAG_ANY_2017",
    "CCON_CANC_DIAG_ANY_2019",
    "CCON_CANC_DIAG_ANY_2021",
    "CCON_CANC_DIAG_AGE_2005",
    "CCON_CANC_DIAG_AGE_2007",
    "CCON_CANC_DIAG_AGE_2009",
    "CCON_CANC_DIAG_AGE_2011",
    "CCON_CANC_DIAG_AGE_2013",
    "CCON_CANC_DIAG_AGE_2015",
    "CCON_CANC_DIAG_AGE_2017",
    "CCON_CANC_DIAG_AGE_2019",
    "CCON_CANC_DIAG_AGE_2021",
    "CCON_CANC_DEGR_1999",
    "CCON_CANC_DEGR_2001",
    "CCON_CANC_DEGR_2003",
    "CCON_CANC_DEGR_2005",
    "CCON_CANC_DEGR_2007",
    "CCON_CANC_DEGR_2009",
    "CCON_CANC_DEGR_2011",
    "CCON_CANC_DEGR_2013",
    "CCON_CANC_DEGR_2015",
    "CCON_CANC_DEGR_2017",
    "CCON_CANC_DEGR_2019",
    "CCON_CANC_DEGR_2021",
    "CCON_CANC_STAT_2005",
    "CCON_CANC_STAT_2007",
    "CCON_CANC_STAT_2009",
    "CCON_CANC_STAT_2011",
    "CCON_CANC_STAT_2013",
    "CCON_CANC_STAT_2015",
    "CCON_CANC_STAT_2017",
    "CCON_CANC_STAT_2019",
    "CCON_CANC_STAT_2021",
    "CCON_CANC_TYPE_1M_2005",
    "CCON_CANC_TYPE_1M_2007",
    "CCON_CANC_TYPE_1M_2009",
    "CCON_CANC_TYPE_1M_2011",
    "CCON_CANC_TYPE_1M_2013",
    "CCON_CANC_TYPE_1M_2015",
    "CCON_CANC_TYPE_1M_2017",
    "CCON_CANC_TYPE_1M_2019",
    "CCON_CANC_TYPE_1M_2021",
    "CCON_CANC_TYPE_2M_2005",
    "CCON_CANC_TYPE_2M_2007",
    "CCON_CANC_TYPE_2M_2009",
    "CCON_CANC_TYPE_2M_2011",
    "CCON_CANC_TYPE_2M_2013",
    "CCON_CANC_TYPE_2M_2015",
    "CCON_CANC_TYPE_2M_2017",
    "CCON_CANC_TYPE_2M_2019",
    "CCON_CANC_TYPE_2M_2021",
    "CCON_DIAB_DIAG_ANY_1999",
    "CCON_DIAB_DIAG_ANY_2001",
    "CCON_DIAB_DIAG_ANY_2003",
    "CCON_DIAB_DIAG_ANY_2005",
    "CCON_DIAB_DIAG_ANY_2007",
    "CCON_DIAB_DIAG_ANY_2009",
    "CCON_DIAB_DIAG_ANY_2011",
    "CCON_DIAB_DIAG_ANY_2013",
    "CCON_DIAB_DIAG_ANY_2015",
    "CCON_DIAB_DIAG_ANY_2017",
    "CCON_DIAB_DIAG_ANY_2019",
    "CCON_DIAB_DIAG_ANY_2021",
    "CCON_DIAB_DIAG_AGE_2005",
    "CCON_DIAB_DIAG_AGE_2007",
    "CCON_DIAB_DIAG_AGE_2009",
    "CCON_DIAB_DIAG_AGE_2011",
    "CCON_DIAB_DIAG_AGE_2013",
    "CCON_DIAB_DIAG_AGE_2015",
    "CCON_DIAB_DIAG_AGE_2017",
    "CCON_DIAB_DIAG_AGE_2019",
    "CCON_DIAB_DIAG_AGE_2021",
    "CCON_DIAB_DEGR_1999",
    "CCON_DIAB_DEGR_2001",
    "CCON_DIAB_DEGR_2003",
    "CCON_DIAB_DEGR_2005",
    "CCON_DIAB_DEGR_2007",
    "CCON_DIAB_DEGR_2009",
    "CCON_DIAB_DEGR_2011",
    "CCON_DIAB_DEGR_2013",
    "CCON_DIAB_DEGR_2015",
    "CCON_DIAB_DEGR_2017",
    "CCON_DIAB_DEGR_2019",
    "CCON_DIAB_DEGR_2021",
    "CCON_DIAB_CARE_2011",
    "CCON_DIAB_CARE_2013",
    "CCON_DIAB_CARE_2015",
    "CCON_DIAB_CARE_2017",
    "CCON_DIAB_CARE_2019",
    "CCON_DIAB_CARE_2021",
    "CCON_EMOP_DIAG_ANY_1999",
    "CCON_EMOP_DIAG_ANY_2001",
    "CCON_EMOP_DIAG_ANY_2003",
    "CCON_EMOP_DIAG_ANY_2005",
    "CCON_EMOP_DIAG_ANY_2007",
    "CCON_EMOP_DIAG_ANY_2009",
    "CCON_EMOP_DIAG_ANY_2011",
    "CCON_EMOP_DIAG_ANY_2013",
    "CCON_EMOP_DIAG_ANY_2015",
    "CCON_EMOP_DIAG_ANY_2017",
    "CCON_EMOP_DIAG_ANY_2019",
    "CCON_EMOP_DIAG_ANY_2021",
    "CCON_EMOP_DIAG_AGE_2005",
    "CCON_EMOP_DIAG_AGE_2007",
    "CCON_EMOP_DIAG_AGE_2009",
    "CCON_EMOP_DIAG_AGE_2011",
    "CCON_EMOP_DIAG_AGE_2013",
    "CCON_EMOP_DIAG_AGE_2015",
    "CCON_EMOP_DIAG_AGE_2017",
    "CCON_EMOP_DIAG_AGE_2019",
    "CCON_EMOP_DIAG_AGE_2021",
    "CCON_EMOP_DEGR_1999",
    "CCON_EMOP_DEGR_2001",
    "CCON_EMOP_DEGR_2003",
    "CCON_EMOP_DEGR_2005",
    "CCON_EMOP_DEGR_2007",
    "CCON_EMOP_DEGR_2009",
    "CCON_EMOP_DEGR_2011",
    "CCON_EMOP_DEGR_2013",
    "CCON_EMOP_DEGR_2015",
    "CCON_EMOP_DEGR_2017",
    "CCON_EMOP_DEGR_2019",
    "CCON_EMOP_DEGR_2021",
    "CCON_EMOP_CARE_2011",
    "CCON_EMOP_CARE_2013",
    "CCON_EMOP_CARE_2015",
    "CCON_EMOP_CARE_2017",
    "CCON_EMOP_CARE_2019",
    "CCON_EMOP_CARE_2021",
    "CCON_EMOP_TYPE_1M_2005",
    "CCON_EMOP_TYPE_1M_2007",
    "CCON_EMOP_TYPE_1M_2009",
    "CCON_EMOP_TYPE_1M_2011",
    "CCON_EMOP_TYPE_1M_2013",
    "CCON_EMOP_TYPE_1M_2015",
    "CCON_EMOP_TYPE_1M_2017",
    "CCON_EMOP_TYPE_1M_2019",
    "CCON_EMOP_TYPE_1M_2021",
    "CCON_EMOP_TYPE_2M_2005",
    "CCON_EMOP_TYPE_2M_2007",
    "CCON_EMOP_TYPE_2M_2009",
    "CCON_EMOP_TYPE_2M_2011",
    "CCON_EMOP_TYPE_2M_2013",
    "CCON_EMOP_TYPE_2M_2015",
    "CCON_EMOP_TYPE_2M_2017",
    "CCON_EMOP_TYPE_2M_2019",
    "CCON_EMOP_TYPE_2M_2021",
    "CCON_EMOP_TYPE_3M_2005",
    "CCON_EMOP_TYPE_3M_2007",
    "CCON_EMOP_TYPE_3M_2009",
    "CCON_EMOP_TYPE_3M_2011",
    "CCON_EMOP_TYPE_3M_2013",
    "CCON_EMOP_TYPE_3M_2015",
    "CCON_EMOP_TYPE_3M_2017",
    "CCON_EMOP_TYPE_3M_2019",
    "CCON_EMOP_TYPE_3M_2021",
    "CCON_HATT_DIAG_ANY_1999",
    "CCON_HATT_DIAG_ANY_2001",
    "CCON_HATT_DIAG_ANY_2003",
    "CCON_HATT_DIAG_ANY_2005",
    "CCON_HATT_DIAG_ANY_2007",
    "CCON_HATT_DIAG_ANY_2009",
    "CCON_HATT_DIAG_ANY_2011",
    "CCON_HATT_DIAG_ANY_2013",
    "CCON_HATT_DIAG_ANY_2015",
    "CCON_HATT_DIAG_ANY_2017",
    "CCON_HATT_DIAG_ANY_2019",
    "CCON_HATT_DIAG_ANY_2021",
    "CCON_HATT_DIAG_AGE_2005",
    "CCON_HATT_DIAG_AGE_2007",
    "CCON_HATT_DIAG_AGE_2009",
    "CCON_HATT_DIAG_AGE_2011",
    "CCON_HATT_DIAG_AGE_2013",
    "CCON_HATT_DIAG_AGE_2015",
    "CCON_HATT_DIAG_AGE_2017",
    "CCON_HATT_DIAG_AGE_2019",
    "CCON_HATT_DIAG_AGE_2021",
    "CCON_HATT_DIAG_2ND_2005",
    "CCON_HATT_DIAG_2ND_2007",
    "CCON_HATT_DIAG_2ND_2009",
    "CCON_HATT_DIAG_2ND_2011",
    "CCON_HATT_DIAG_2ND_2013",
    "CCON_HATT_DIAG_2ND_2015",
    "CCON_HATT_DIAG_2ND_2017",
    "CCON_HATT_DIAG_2ND_2019",
    "CCON_HATT_DIAG_2ND_2021",
    "CCON_HATT_DEGR_1999",
    "CCON_HATT_DEGR_2001",
    "CCON_HATT_DEGR_2003",
    "CCON_HATT_DEGR_2005",
    "CCON_HATT_DEGR_2007",
    "CCON_HATT_DEGR_2009",
    "CCON_HATT_DEGR_2011",
    "CCON_HATT_DEGR_2013",
    "CCON_HATT_DEGR_2015",
    "CCON_HATT_DEGR_2017",
    "CCON_HATT_DEGR_2019",
    "CCON_HATT_DEGR_2021",
    "CCON_HATT_CARE_2011",
    "CCON_HATT_CARE_2013",
    "CCON_HATT_CARE_2015",
    "CCON_HATT_CARE_2017",
    "CCON_HATT_CARE_2019",
    "CCON_HATT_CARE_2021",
    "CCON_HDIS_DIAG_ANY_1999",
    "CCON_HDIS_DIAG_ANY_2001",
    "CCON_HDIS_DIAG_ANY_2003",
    "CCON_HDIS_DIAG_ANY_2005",
    "CCON_HDIS_DIAG_ANY_2007",
    "CCON_HDIS_DIAG_ANY_2009",
    "CCON_HDIS_DIAG_ANY_2011",
    "CCON_HDIS_DIAG_ANY_2013",
    "CCON_HDIS_DIAG_ANY_2015",
    "CCON_HDIS_DIAG_ANY_2017",
    "CCON_HDIS_DIAG_ANY_2019",
    "CCON_HDIS_DIAG_ANY_2021",
    "CCON_HDIS_DIAG_AGE_2005",
    "CCON_HDIS_DIAG_AGE_2007",
    "CCON_HDIS_DIAG_AGE_2009",
    "CCON_HDIS_DIAG_AGE_2011",
    "CCON_HDIS_DIAG_AGE_2013",
    "CCON_HDIS_DIAG_AGE_2015",
    "CCON_HDIS_DIAG_AGE_2017",
    "CCON_HDIS_DIAG_AGE_2019",
    "CCON_HDIS_DIAG_AGE_2021",
    "CCON_HDIS_DEGR_1999",
    "CCON_HDIS_DEGR_2001",
    "CCON_HDIS_DEGR_2003",
    "CCON_HDIS_DEGR_2005",
    "CCON_HDIS_DEGR_2007",
    "CCON_HDIS_DEGR_2009",
    "CCON_HDIS_DEGR_2011",
    "CCON_HDIS_DEGR_2013",
    "CCON_HDIS_DEGR_2015",
    "CCON_HDIS_DEGR_2017",
    "CCON_HDIS_DEGR_2019",
    "CCON_HDIS_DEGR_2021",
    "CCON_HDIS_CARE_2011",
    "CCON_HDIS_CARE_2013",
    "CCON_HDIS_CARE_2015",
    "CCON_HDIS_CARE_2017",
    "CCON_HDIS_CARE_2019",
    "CCON_HDIS_CARE_2021",
    "CCON_HIBP_DIAG_ANY_1999",
    "CCON_HIBP_DIAG_ANY_2001",
    "CCON_HIBP_DIAG_ANY_2003",
    "CCON_HIBP_DIAG_ANY_2005",
    "CCON_HIBP_DIAG_ANY_2007",
    "CCON_HIBP_DIAG_ANY_2009",
    "CCON_HIBP_DIAG_ANY_2011",
    "CCON_HIBP_DIAG_ANY_2013",
    "CCON_HIBP_DIAG_ANY_2015",
    "CCON_HIBP_DIAG_ANY_2017",
    "CCON_HIBP_DIAG_ANY_2019",
    "CCON_HIBP_DIAG_ANY_2021",
    "CCON_HIBP_DIAG_AGE_2005",
    "CCON_HIBP_DIAG_AGE_2007",
    "CCON_HIBP_DIAG_AGE_2009",
    "CCON_HIBP_DIAG_AGE_2011",
    "CCON_HIBP_DIAG_AGE_2013",
    "CCON_HIBP_DIAG_AGE_2015",
    "CCON_HIBP_DIAG_AGE_2017",
    "CCON_HIBP_DIAG_AGE_2019",
    "CCON_HIBP_DIAG_AGE_2021",
    "CCON_HIBP_DEGR_1999",
    "CCON_HIBP_DEGR_2001",
    "CCON_HIBP_DEGR_2003",
    "CCON_HIBP_DEGR_2005",
    "CCON_HIBP_DEGR_2007",
    "CCON_HIBP_DEGR_2009",
    "CCON_HIBP_DEGR_2011",
    "CCON_HIBP_DEGR_2013",
    "CCON_HIBP_DEGR_2015",
    "CCON_HIBP_DEGR_2017",
    "CCON_HIBP_DEGR_2019",
    "CCON_HIBP_DEGR_2021",
    "CCON_HIBP_CARE_2011",
    "CCON_HIBP_CARE_2013",
    "CCON_HIBP_CARE_2015",
    "CCON_HIBP_CARE_2017",
    "CCON_HIBP_CARE_2019",
    "CCON_HIBP_CARE_2021",
    "CCON_LDIS_DIAG_ANY_1999",
    "CCON_LDIS_DIAG_ANY_2001",
    "CCON_LDIS_DIAG_ANY_2003",
    "CCON_LDIS_DIAG_ANY_2005",
    "CCON_LDIS_DIAG_ANY_2007",
    "CCON_LDIS_DIAG_ANY_2009",
    "CCON_LDIS_DIAG_ANY_2011",
    "CCON_LDIS_DIAG_ANY_2013",
    "CCON_LDIS_DIAG_ANY_2015",
    "CCON_LDIS_DIAG_ANY_2017",
    "CCON_LDIS_DIAG_ANY_2019",
    "CCON_LDIS_DIAG_ANY_2021",
    "CCON_LDIS_DIAG_AGE_2005",
    "CCON_LDIS_DIAG_AGE_2007",
    "CCON_LDIS_DIAG_AGE_2009",
    "CCON_LDIS_DIAG_AGE_2011",
    "CCON_LDIS_DIAG_AGE_2013",
    "CCON_LDIS_DIAG_AGE_2015",
    "CCON_LDIS_DIAG_AGE_2017",
    "CCON_LDIS_DIAG_AGE_2019",
    "CCON_LDIS_DIAG_AGE_2021",
    "CCON_LDIS_DEGR_1999",
    "CCON_LDIS_DEGR_2001",
    "CCON_LDIS_DEGR_2003",
    "CCON_LDIS_DEGR_2005",
    "CCON_LDIS_DEGR_2007",
    "CCON_LDIS_DEGR_2009",
    "CCON_LDIS_DEGR_2011",
    "CCON_LDIS_DEGR_2013",
    "CCON_LDIS_DEGR_2015",
    "CCON_LDIS_DEGR_2017",
    "CCON_LDIS_DEGR_2019",
    "CCON_LDIS_DEGR_2021",
    "CCON_LDIS_CARE_2011",
    "CCON_LDIS_CARE_2013",
    "CCON_LDIS_CARE_2015",
    "CCON_LDIS_CARE_2017",
    "CCON_LDIS_CARE_2019",
    "CCON_LDIS_CARE_2021",
    "CCON_LMEM_DIAG_ANY_1999",
    "CCON_LMEM_DIAG_ANY_2001",
    "CCON_LMEM_DIAG_ANY_2003",
    "CCON_LMEM_DIAG_ANY_2005",
    "CCON_LMEM_DIAG_ANY_2007",
    "CCON_LMEM_DIAG_ANY_2009",
    "CCON_LMEM_DIAG_ANY_2011",
    "CCON_LMEM_DIAG_ANY_2013",
    "CCON_LMEM_DIAG_ANY_2015",
    "CCON_LMEM_DIAG_ANY_2017",
    "CCON_LMEM_DIAG_ANY_2019",
    "CCON_LMEM_DIAG_ANY_2021",
    "CCON_LMEM_DIAG_AGE_2005",
    "CCON_LMEM_DIAG_AGE_2007",
    "CCON_LMEM_DIAG_AGE_2009",
    "CCON_LMEM_DIAG_AGE_2011",
    "CCON_LMEM_DIAG_AGE_2013",
    "CCON_LMEM_DIAG_AGE_2015",
    "CCON_LMEM_DIAG_AGE_2017",
    "CCON_LMEM_DIAG_AGE_2019",
    "CCON_LMEM_DIAG_AGE_2021",
    "CCON_LMEM_DEGR_1999",
    "CCON_LMEM_DEGR_2001",
    "CCON_LMEM_DEGR_2003",
    "CCON_LMEM_DEGR_2005",
    "CCON_LMEM_DEGR_2007",
    "CCON_LMEM_DEGR_2009",
    "CCON_LMEM_DEGR_2011",
    "CCON_LMEM_DEGR_2013",
    "CCON_LMEM_DEGR_2015",
    "CCON_LMEM_DEGR_2017",
    "CCON_LMEM_DEGR_2019",
    "CCON_LMEM_DEGR_2021",
    "CCON_LMEM_CARE_2011",
    "CCON_LMEM_CARE_2013",
    "CCON_LMEM_CARE_2015",
    "CCON_LMEM_CARE_2017",
    "CCON_LMEM_CARE_2019",
    "CCON_LMEM_CARE_2021",
    "CCON_LUNG_DIAG_ANY_1999",
    "CCON_LUNG_DIAG_ANY_2001",
    "CCON_LUNG_DIAG_ANY_2003",
    "CCON_LUNG_DIAG_ANY_2005",
    "CCON_LUNG_DIAG_ANY_2007",
    "CCON_LUNG_DIAG_ANY_2009",
    "CCON_LUNG_DIAG_ANY_2011",
    "CCON_LUNG_DIAG_ANY_2013",
    "CCON_LUNG_DIAG_ANY_2015",
    "CCON_LUNG_DIAG_ANY_2017",
    "CCON_LUNG_DIAG_ANY_2019",
    "CCON_LUNG_DIAG_ANY_2021",
    "CCON_LUNG_DIAG_AGE_2005",
    "CCON_LUNG_DIAG_AGE_2007",
    "CCON_LUNG_DIAG_AGE_2009",
    "CCON_LUNG_DIAG_AGE_2011",
    "CCON_LUNG_DIAG_AGE_2013",
    "CCON_LUNG_DIAG_AGE_2015",
    "CCON_LUNG_DIAG_AGE_2017",
    "CCON_LUNG_DIAG_AGE_2019",
    "CCON_LUNG_DIAG_AGE_2021",
    "CCON_LUNG_DEGR_1999",
    "CCON_LUNG_DEGR_2001",
    "CCON_LUNG_DEGR_2003",
    "CCON_LUNG_DEGR_2005",
    "CCON_LUNG_DEGR_2007",
    "CCON_LUNG_DEGR_2009",
    "CCON_LUNG_DEGR_2011",
    "CCON_LUNG_DEGR_2013",
    "CCON_LUNG_DEGR_2015",
    "CCON_LUNG_DEGR_2017",
    "CCON_LUNG_DEGR_2019",
    "CCON_LUNG_DEGR_2021",
    "CCON_LUNG_CARE_2011",
    "CCON_LUNG_CARE_2013",
    "CCON_LUNG_CARE_2015",
    "CCON_LUNG_CARE_2017",
    "CCON_LUNG_CARE_2019",
    "CCON_LUNG_CARE_2021",
    "CCON_STRO_DIAG_ANY_1999",
    "CCON_STRO_DIAG_ANY_2001",
    "CCON_STRO_DIAG_ANY_2003",
    "CCON_STRO_DIAG_ANY_2005",
    "CCON_STRO_DIAG_ANY_2007",
    "CCON_STRO_DIAG_ANY_2009",
    "CCON_STRO_DIAG_ANY_2011",
    "CCON_STRO_DIAG_ANY_2013",
    "CCON_STRO_DIAG_ANY_2015",
    "CCON_STRO_DIAG_ANY_2017",
    "CCON_STRO_DIAG_ANY_2019",
    "CCON_STRO_DIAG_ANY_2021",
    "CCON_STRO_DIAG_AGE_2005",
    "CCON_STRO_DIAG_AGE_2007",
    "CCON_STRO_DIAG_AGE_2009",
    "CCON_STRO_DIAG_AGE_2011",
    "CCON_STRO_DIAG_AGE_2013",
    "CCON_STRO_DIAG_AGE_2015",
    "CCON_STRO_DIAG_AGE_2017",
    "CCON_STRO_DIAG_AGE_2019",
    "CCON_STRO_DIAG_AGE_2021",
    "CCON_STRO_DIAG_2ND_2005",
    "CCON_STRO_DIAG_2ND_2007",
    "CCON_STRO_DIAG_2ND_2009",
    "CCON_STRO_DIAG_2ND_2011",
    "CCON_STRO_DIAG_2ND_2013",
    "CCON_STRO_DIAG_2ND_2015",
    "CCON_STRO_DIAG_2ND_2017",
    "CCON_STRO_DIAG_2ND_2019",
    "CCON_STRO_DIAG_2ND_2021",
    "CCON_STRO_DEGR_1999",
    "CCON_STRO_DEGR_2001",
    "CCON_STRO_DEGR_2003",
    "CCON_STRO_DEGR_2005",
    "CCON_STRO_DEGR_2007",
    "CCON_STRO_DEGR_2009",
    "CCON_STRO_DEGR_2011",
    "CCON_STRO_DEGR_2013",
    "CCON_STRO_DEGR_2015",
    "CCON_STRO_DEGR_2017",
    "CCON_STRO_DEGR_2019",
    "CCON_STRO_DEGR_2021",
    "CCON_STRO_CARE_2011",
    "CCON_STRO_CARE_2013",
    "CCON_STRO_CARE_2015",
    "CCON_STRO_CARE_2017",
    "CCON_STRO_CARE_2019",
    "CCON_STRO_CARE_2021",
    "CCON_OCON_DIAG_ANY_2005",
    "CCON_OCON_DIAG_ANY_2007",
    "CCON_OCON_DIAG_ANY_2009",
    "CCON_OCON_DIAG_ANY_2011",
    "CCON_OCON_DIAG_ANY_2013",
    "CCON_OCON_DIAG_ANY_2015",
    "CCON_OCON_DIAG_ANY_2017",
    "CCON_OCON_DIAG_ANY_2019",
    "CCON_OCON_DIAG_ANY_2021",
    "CCON_OCON_DIAG_AGE_2005",
    "CCON_OCON_DIAG_AGE_2007",
    "CCON_OCON_DIAG_AGE_2009",
    "CCON_OCON_DIAG_AGE_2011",
    "CCON_OCON_DIAG_AGE_2013",
    "CCON_OCON_DIAG_AGE_2015",
    "CCON_OCON_DIAG_AGE_2017",
    "CCON_OCON_DIAG_AGE_2019",
    "CCON_OCON_DIAG_AGE_2021",
    "CCON_OCON_DEGR_2005",
    "CCON_OCON_DEGR_2007",
    "CCON_OCON_DEGR_2009",
    "CCON_OCON_DEGR_2011",
    "CCON_OCON_DEGR_2013",
    "CCON_OCON_DEGR_2015",
    "CCON_OCON_DEGR_2017",
    "CCON_OCON_DEGR_2019",
    "CCON_OCON_DEGR_2021",
    "CCON_OCON_CARE_2011",
    "CCON_OCON_CARE_2013",
    "CCON_OCON_CARE_2015",
    "CCON_OCON_CARE_2017",
    "CCON_OCON_CARE_2019",
    "CCON_OCON_CARE_2021",
    "CCON_OCON_TYPE_2007",
    "CCON_OCON_TYPE_2009",
    "CCON_OCON_TYPE_2011",
    "CCON_OCON_TYPE_2013",
    "CCON_OCON_TYPE_2015",
    "CCON_OCON_TYPE_2017",
    "CCON_OCON_TYPE_2019",
    "CCON_OCON_TYPE_2021"
  ],
  "health_general": [
    "ID",
    "LINEAGE",
    "PNUM",
    "GHLTH_STAT_1984",
    "GHLTH_STAT_1985",
    "GHLTH_STAT_1986",
    "GHLTH_STAT_1987",
    "GHLTH_STAT_1988",
    "GHLTH_STAT_1989",
    "GHLTH_STAT_1990",
    "GHLTH_STAT_1991",
    "GHLTH_STAT_1992",
    "GHLTH_STAT_1993",
    "GHLTH_STAT_1994",
    "GHLTH_STAT_1995",
    "GHLTH_STAT_1996",
    "GHLTH_STAT_1997",
    "GHLTH_STAT_1999",
    "GHLTH_STAT_2001",
    "GHLTH_STAT_2003",
    "GHLTH_STAT_2005",
    "GHLTH_STAT_2007",
    "GHLTH_STAT_2009",
    "GHLTH_STAT_2011",
    "GHLTH_STAT_2013",
    "GHLTH_STAT_2015",
    "GHLTH_STAT_2017",
    "GHLTH_STAT_2019",
    "GHLTH_STAT_2021",
    "GHLTH_POOR_1986",
    "GHLTH_POOR_1988",
    "GHLTH_POOR_1989",
    "GHLTH_POOR_1990",
    "GHLTH_POOR_1991",
    "GHLTH_POOR_1992",
    "GHLTH_POOR_1993",
    "GHLTH_POOR_1994",
    "GHLTH_POOR_1995",
    "GHLTH_POOR_1996",
    "GHLTH_POOR_1997",
    "GHLTH_POOR_1999",
    "GHLTH_POOR_2001",
    "GHLTH_POOR_2003",
    "GHLTH_POOR_2005",
    "GHLTH_POOR_2007",
    "GHLTH_POOR_2009",
    "GHLTH_POOR_2011",
    "GHLTH_POOR_2013",
    "GHLTH_POOR_2015",
    "GHLTH_POOR_2017",
    "GHLTH_POOR_2019",
    "GHLTH_POOR_2021",
    "GHLTH_CHNG_1984",
    "GHLTH_CHNG_1985",
    "GHLTH_CHNG_1986",
    "GHLTH_CHNG_1987",
    "GHLTH_CHNG_2003",
    "GHLTH_CHNG_2005",
    "GHLTH_CHNG_2007",
    "GHLTH_CHNG_2009",
    "GHLTH_CHNG_2011",
    "GHLTH_CHNG_2013",
    "GHLTH_CHNG_2015",
    "GHLTH_CHNG_2017",
    "GHLTH_CHNG_2019",
    "GHLTH_CHNG_2021"
  ],
  "housing": [
    "ID",
    "LINEAGE",
    "PNUM",
    "HOME_STAT_1968",
    "HOME_STAT_1969",
    "HOME_STAT_1970",
    "HOME_STAT_1971",
    "HOME_STAT_1972",
    "HOME_STAT_1973",
    "HOME_STAT_1974",
    "HOME_STAT_1975",
    "HOME_STAT_1976",
    "HOME_STAT_1977",
    "HOME_STAT_1978",
    "HOME_STAT_1979",
    "HOME_STAT_1980",
    "HOME_STAT_1981",
    "HOME_STAT_1982",
    "HOME_STAT_1983",
    "HOME_STAT_1984",
    "HOME_STAT_1985",
    "HOME_STAT_1986",
    "HOME_STAT_1987",
    "HOME_STAT_1988",
    "HOME_STAT_1989",
    "HOME_STAT_1990",
    "HOME_STAT_1991",
    "HOME_STAT_1992",
    "HOME_STAT_1993",
    "HOME_STAT_1994",
    "HOME_STAT_1995",
    "HOME_STAT_1996",
    "HOME_STAT_1997",
    "HOME_STAT_1999",
    "HOME_STAT_2001",
    "HOME_STAT_2003",
    "HOME_STAT_2005",
    "HOME_STAT_2007",
    "HOME_STAT_2009",
    "HOME_STAT_2011",
    "HOME_STAT_2013",
    "HOME_STAT_2015",
    "HOME_STAT_2017",
    "HOME_STAT_2019",
    "HOME_STAT_2021",
    "HOME_OWN_VAL_ND_1968",
    "HOME_OWN_VAL_ND_1969",
    "HOME_OWN_VAL_ND_1970",
    "HOME_OWN_VAL_ND_1971",
    "HOME_OWN_VAL_ND_1972",
    "HOME_OWN_VAL_ND_1973",
    "HOME_OWN_VAL_ND_1974",
    "HOME_OWN_VAL_ND_1975",
    "HOME_OWN_VAL_ND_1976",
    "HOME_OWN_VAL_ND_1977",
    "HOME_OWN_VAL_ND_1978",
    "HOME_OWN_VAL_ND_1979",
    "HOME_OWN_VAL_ND_1980",
    "HOME_OWN_VAL_ND_1981",
    "HOME_OWN_VAL_ND_1982",
    "HOME_OWN_VAL_ND_1983",
    "HOME_OWN_VAL_ND_1984",
    "HOME_OWN_VAL_ND_1985",
    "HOME_OWN_VAL_ND_1986",
    "HOME_OWN_VAL_ND_1987",
    "HOME_OWN_VAL_ND_1988",
    "HOME_OWN_VAL_ND_1989",
    "HOME_OWN_VAL_ND_1990",
    "HOME_OWN_VAL_ND_1991",
    "HOME_OWN_VAL_ND_1992",
    "HOME_OWN_VAL_ND_1993",
    "HOME_OWN_VAL_ND_1994",
    "HOME_OWN_VAL_ND_1995",
    "HOME_OWN_VAL_ND_1996",
    "HOME_OWN_VAL_ND_1997",
    "HOME_OWN_VAL_ND_1999",
    "HOME_OWN_VAL_ND_2001",
    "HOME_OWN_VAL_ND_2003",
    "HOME_OWN_VAL_ND_2005",
    "HOME_OWN_VAL_ND_2007",
    "HOME_OWN_VAL_ND_2009",
    "HOME_OWN_VAL_ND_2011",
    "HOME_OWN_VAL_ND_2013",
    "HOME_OWN_VAL_ND_2015",
    "HOME_OWN_VAL_ND_2017",
    "HOME_OWN_VAL_ND_2019",
    "HOME_OWN_VAL_ND_2021",
    "HOME_OWN_VAL_NDF_1968",
    "HOME_OWN_VAL_NDF_1969",
    "HOME_OWN_VAL_NDF_1970",
    "HOME_OWN_VAL_NDF_1971",
    "HOME_OWN_VAL_NDF_1972",
    "HOME_OWN_VAL_NDF_1973",
    "HOME_OWN_VAL_NDF_1974",
    "HOME_OWN_VAL_NDF_1975",
    "HOME_OWN_VAL_NDF_1976",
    "HOME_OWN_VAL_NDF_1977",
    "HOME_OWN_VAL_NDF_1978",
    "HOME_OWN_VAL_NDF_1979",
    "HOME_OWN_VAL_NDF_1980",
    "HOME_OWN_VAL_NDF_1981",
    "HOME_OWN_VAL_NDF_1982",
    "HOME_OWN_VAL_NDF_1983",
    "HOME_OWN_VAL_NDF_1984",
    "HOME_OWN_VAL_NDF_1985",
    "HOME_OWN_VAL_NDF_1986",
    "HOME_OWN_VAL_NDF_1987",
    "HOME_OWN_VAL_NDF_1988",
    "HOME_OWN_VAL_NDF_1989",
    "HOME_OWN_VAL_NDF_1990",
    "HOME_OWN_VAL_NDF_1991",
    "HOME_OWN_VAL_NDF_1992",
    "HOME_OWN_VAL_NDF_1993",
    "HOME_OWN_VAL_NDF_1994",
    "HOME_OWN_VAL_NDF_1995",
    "HOME_OWN_VAL_NDF_1996",
    "HOME_OWN_VAL_NDF_1997",
    "HOME_OWN_VAL_NDF_1999",
    "HOME_OWN_VAL_NDF_2001",
    "HOME_OWN_VAL_NDF_2003",
    "HOME_OWN_VAL_NDF_2005",
    "HOME_OWN_VAL_NDF_2007",
    "HOME_OWN_VAL_NDF_2009",
    "HOME_OWN_VAL_NDF_2011",
    "HOME_OWN_VAL_NDF_2013",
    "HOME_OWN_VAL_NDF_2015",
    "HOME_OWN_VAL_NDF_2017",
    "HOME_OWN_VAL_NDF_2019",
    "HOME_OWN_VAL_NDF_2021",
    "HOME_OWN_VAL_RD_1968",
    "HOME_OWN_VAL_RD_1969",
    "HOME_OWN_VAL_RD_1970",
    "HOME_OWN_VAL_RD_1971",
    "HOME_OWN_VAL_RD_1972",
    "HOME_OWN_VAL_RD_1973",
    "HOME_OWN_VAL_RD_1974",
    "HOME_OWN_VAL_RD_1975",
    "HOME_OWN_VAL_RD_1976",
    "HOME_OWN_VAL_RD_1977",
    "HOME_OWN_VAL_RD_1978",
    "HOME_OWN_VAL_RD_1979",
    "HOME_OWN_VAL_RD_1980",
    "HOME_OWN_VAL_RD_1981",
    "HOME_OWN_VAL_RD_1982",
    "HOME_OWN_VAL_RD_1983",
    "HOME_OWN_VAL_RD_1984",
    "HOME_OWN_VAL_RD_1985",
    "HOME_OWN_VAL_RD_1986",
    "HOME_OWN_VAL_RD_1987",
    "HOME_OWN_VAL_RD_1988",
    "HOME_OWN_VAL_RD_1989",
    "HOME_OWN_VAL_RD_1990",
    "HOME_OWN_VAL_RD_1991",
    "HOME_OWN_VAL_RD_1992",
    "HOME_OWN_VAL_RD_1993",
    "HOME_OWN_VAL_RD_1994",
    "HOME_OWN_VAL_RD_1995",
    "HOME_OWN_VAL_RD_1996",
    "HOME_OWN_VAL_RD_1997",
    "HOME_OWN_VAL_RD_1999",
    "HOME_OWN_VAL_RD_2001",
    "HOME_OWN_VAL_RD_2003",
    "HOME_OWN_VAL_RD_2005",
    "HOME_OWN_VAL_RD_2007",
    "HOME_OWN_VAL_RD_2009",
    "HOME_OWN_VAL_RD_2011",
    "HOME_OWN_VAL_RD_2013",
    "HOME_OWN_VAL_RD_2015",
    "HOME_OWN_VAL_RD_2017",
    "HOME_OWN_VAL_RD_2019",
    "HOME_OWN_VAL_RD_2021",
    "HOME_OWN_VAL_RDF_1968",
    "HOME_OWN_VAL_RDF_1969",
    "HOME_OWN_VAL_RDF_1970",
    "HOME_OWN_VAL_RDF_1971",
    "HOME_OWN_VAL_RDF_1972",
    "HOME_OWN_VAL_RDF_1973",
    "HOME_OWN_VAL_RDF_1974",
    "HOME_OWN_VAL_RDF_1975",
    "HOME_OWN_VAL_RDF_1976",
    "HOME_OWN_VAL_RDF_1977",
    "HOME_OWN_VAL_RDF_1978",
    "HOME_OWN_VAL_RDF_1979",
    "HOME_OWN_VAL_RDF_1980",
    "HOME_OWN_VAL_RDF_1981",
    "HOME_OWN_VAL_RDF_1982",
    "HOME_OWN_VAL_RDF_1983",
    "HOME_OWN_VAL_RDF_1984",
    "HOME_OWN_VAL_RDF_1985",
    "HOME_OWN_VAL_RDF_1986",
    "HOME_OWN_VAL_RDF_1987",
    "HOME_OWN_VAL_RDF_1988",
    "HOME_OWN_VAL_RDF_1989",
    "HOME_OWN_VAL_RDF_1990",
    "HOME_OWN_VAL_RDF_1991",
    "HOME_OWN_VAL_RDF_1992",
    "HOME_OWN_VAL_RDF_1993",
    "HOME_OWN_VAL_RDF_1994",
    "HOME_OWN_VAL_RDF_1995",
    "HOME_OWN_VAL_RDF_1996",
    "HOME_OWN_VAL_RDF_1997",
    "HOME_OWN_VAL_RDF_1999",
    "HOME_OWN_VAL_RDF_2001",
    "HOME_OWN_VAL_RDF_2003",
    "HOME_OWN_VAL_RDF_2005",
    "HOME_OWN_VAL_RDF_2007",
    "HOME_OWN_VAL_RDF_2009",
    "HOME_OWN_VAL_RDF_2011",
    "HOME_OWN_VAL_RDF_2013",
    "HOME_OWN_VAL_RDF_2015",
    "HOME_OWN_VAL_RDF_2017",
    "HOME_OWN_VAL_RDF_2019",
    "HOME_OWN_VAL_RDF_2021",
    "HOME_OWN_MOR_ANY_1M_1968",
    "HOME_OWN_MOR_ANY_1M_1969",
    "HOME_OWN_MOR_ANY_1M_1970",
    "HOME_OWN_MOR_ANY_1M_1971",
    "HOME_OWN_MOR_ANY_1M_1972",
    "HOME_OWN_MOR_ANY_1M_1979",
    "HOME_OWN_MOR_ANY_1M_1980",
    "HOME_OWN_MOR_ANY_1M_1981",
    "HOME_OWN_MOR_ANY_1M_1983",
    "HOME_OWN_MOR_ANY_1M_1984",
    "HOME_OWN_MOR_ANY_1M_1985",
    "HOME_OWN_MOR_ANY_1M_1986",
    "HOME_OWN_MOR_ANY_1M_1987",
    "HOME_OWN_MOR_ANY_1M_1988",
    "HOME_OWN_MOR_ANY_1M_1989",
    "HOME_OWN_MOR_ANY_1M_1990",
    "HOME_OWN_MOR_ANY_1M_1991",
    "HOME_OWN_MOR_ANY_1M_1992",
    "HOME_OWN_MOR_ANY_1M_1993",
    "HOME_OWN_MOR_ANY_1M_1994",
    "HOME_OWN_MOR_ANY_1M_1995",
    "HOME_OWN_MOR_ANY_1M_1996",
    "HOME_OWN_MOR_ANY_1M_1997",
    "HOME_OWN_MOR_ANY_1M_1999",
    "HOME_OWN_MOR_ANY_1M_2001",
    "HOME_OWN_MOR_ANY_1M_2003",
    "HOME_OWN_MOR_ANY_1M_2005",
    "HOME_OWN_MOR_ANY_1M_2007",
    "HOME_OWN_MOR_ANY_1M_2009",
    "HOME_OWN_MOR_ANY_1M_2011",
    "HOME_OWN_MOR_ANY_1M_2013",
    "HOME_OWN_MOR_ANY_1M_2015",
    "HOME_OWN_MOR_ANY_1M_2017",
    "HOME_OWN_MOR_ANY_1M_2019",
    "HOME_OWN_MOR_ANY_1M_2021",
    "HOME_OWN_MOR_ANY_2M_1969",
    "HOME_OWN_MOR_ANY_2M_1970",
    "HOME_OWN_MOR_ANY_2M_1971",
    "HOME_OWN_MOR_ANY_2M_1972",
    "HOME_OWN_MOR_ANY_2M_1979",
    "HOME_OWN_MOR_ANY_2M_1980",
    "HOME_OWN_MOR_ANY_2M_1981",
    "HOME_OWN_MOR_ANY_2M_1983",
    "HOME_OWN_MOR_ANY_2M_1984",
    "HOME_OWN_MOR_ANY_2M_1985",
    "HOME_OWN_MOR_ANY_2M_1986",
    "HOME_OWN_MOR_ANY_2M_1987",
    "HOME_OWN_MOR_ANY_2M_1988",
    "HOME_OWN_MOR_ANY_2M_1989",
    "HOME_OWN_MOR_ANY_2M_1990",
    "HOME_OWN_MOR_ANY_2M_1991",
    "HOME_OWN_MOR_ANY_2M_1992",
    "HOME_OWN_MOR_ANY_2M_1993",
    "HOME_OWN_MOR_ANY_2M_1994",
    "HOME_OWN_MOR_ANY_2M_1995",
    "HOME_OWN_MOR_ANY_2M_1996",
    "HOME_OWN_MOR_ANY_2M_1997",
    "HOME_OWN_MOR_ANY_2M_1999",
    "HOME_OWN_MOR_ANY_2M_2001",
    "HOME_OWN_MOR_ANY_2M_2003",
    "HOME_OWN_MOR_ANY_2M_2005",
    "HOME_OWN_MOR_ANY_2M_2007",
    "HOME_OWN_MOR_ANY_2M_2009",
    "HOME_OWN_MOR_ANY_2M_2011",
    "HOME_OWN_MOR_ANY_2M_2013",
    "HOME_OWN_MOR_ANY_2M_2015",
    "HOME_OWN_MOR_ANY_2M_2017",
    "HOME_OWN_MOR_ANY_2M_2019",
    "HOME_OWN_MOR_ANY_2M_2021",
    "HOME_OWN_MOR_ANY_3M_1994",
    "HOME_OWN_MOR_ANY_3M_1995",
    "HOME_OWN_MOR_ANY_3M_1996",
    "HOME_OWN_MOR_ANY_3M_1997",
    "HOME_OWN_MOR_ANY_3M_1999",
    "HOME_OWN_MOR_ANY_3M_2001",
    "HOME_OWN_MOR_VAL_1M_ND_1969",
    "HOME_OWN_MOR_VAL_1M_ND_1970",
    "HOME_OWN_MOR_VAL_1M_ND_1971",
    "HOME_OWN_MOR_VAL_1M_ND_1972",
    "HOME_OWN_MOR_VAL_1M_ND_1976",
    "HOME_OWN_MOR_VAL_1M_ND_1977",
    "HOME_OWN_MOR_VAL_1M_ND_1978",
    "HOME_OWN_MOR_VAL_1M_ND_1979",
    "HOME_OWN_MOR_VAL_1M_ND_1980",
    "HOME_OWN_MOR_VAL_1M_ND_1981",
    "HOME_OWN_MOR_VAL_1M_ND_1983",
    "HOME_OWN_MOR_VAL_1M_ND_1984",
    "HOME_OWN_MOR_VAL_1M_ND_1985",
    "HOME_OWN_MOR_VAL_1M_ND_1986",
    "HOME_OWN_MOR_VAL_1M_ND_1987",
    "HOME_OWN_MOR_VAL_1M_ND_1988",
    "HOME_OWN_MOR_VAL_1M_ND_1989",
    "HOME_OWN_MOR_VAL_1M_ND_1990",
    "HOME_OWN_MOR_VAL_1M_ND_1991",
    "HOME_OWN_MOR_VAL_1M_ND_1992",
    "HOME_OWN_MOR_VAL_1M_ND_1993",
    "HOME_OWN_MOR_VAL_1M_ND_1994",
    "HOME_OWN_MOR_VAL_1M_ND_1995",
    "HOME_OWN_MOR_VAL_1M_ND_1996",
    "HOME_OWN_MOR_VAL_1M_ND_1997",
    "HOME_OWN_MOR_VAL_1M_ND_1999",
    "HOME_OWN_MOR_VAL_1M_ND_2001",
    "HOME_OWN_MOR_VAL_1M_ND_2003",
    "HOME_OWN_MOR_VAL_1M_ND_2005",
    "HOME_OWN_MOR_VAL_1M_ND_2007",
    "HOME_OWN_MOR_VAL_1M_ND_2009",
    "HOME_OWN_MOR_VAL_1M_ND_2011",
    "HOME_OWN_MOR_VAL_1M_ND_2013",
    "HOME_OWN_MOR_VAL_1M_ND_2015",
    "HOME_OWN_MOR_VAL_1M_ND_2017",
    "HOME_OWN_MOR_VAL_1M_ND_2019",
    "HOME_OWN_MOR_VAL_1M_ND_2021",
    "HOME_OWN_MOR_VAL_1M_NDF_1969",
    "HOME_OWN_MOR_VAL_1M_NDF_1970",
    "HOME_OWN_MOR_VAL_1M_NDF_1971",
    "HOME_OWN_MOR_VAL_1M_NDF_1972",
    "HOME_OWN_MOR_VAL_1M_NDF_1976",
    "HOME_OWN_MOR_VAL_1M_NDF_1977",
    "HOME_OWN_MOR_VAL_1M_NDF_1978",
    "HOME_OWN_MOR_VAL_1M_NDF_1979",
    "HOME_OWN_MOR_VAL_1M_NDF_1980",
    "HOME_OWN_MOR_VAL_1M_NDF_1981",
    "HOME_OWN_MOR_VAL_1M_NDF_1983",
    "HOME_OWN_MOR_VAL_1M_NDF_1984",
    "HOME_OWN_MOR_VAL_1M_NDF_1985",
    "HOME_OWN_MOR_VAL_1M_NDF_1986",
    "HOME_OWN_MOR_VAL_1M_NDF_1987",
    "HOME_OWN_MOR_VAL_1M_NDF_1988",
    "HOME_OWN_MOR_VAL_1M_NDF_1989",
    "HOME_OWN_MOR_VAL_1M_NDF_1990",
    "HOME_OWN_MOR_VAL_1M_NDF_1991",
    "HOME_OWN_MOR_VAL_1M_NDF_1992",
    "HOME_OWN_MOR_VAL_1M_NDF_1993",
    "HOME_OWN_MOR_VAL_1M_NDF_1994",
    "HOME_OWN_MOR_VAL_1M_NDF_1995",
    "HOME_OWN_MOR_VAL_1M_NDF_1996",
    "HOME_OWN_MOR_VAL_1M_NDF_1997",
    "HOME_OWN_MOR_VAL_1M_NDF_1999",
    "HOME_OWN_MOR_VAL_1M_NDF_2001",
    "HOME_OWN_MOR_VAL_1M_NDF_2003",
    "HOME_OWN_MOR_VAL_1M_NDF_2005",
    "HOME_OWN_MOR_VAL_1M_NDF_2007",
    "HOME_OWN_MOR_VAL_1M_NDF_2009",
    "HOME_OWN_MOR_VAL_1M_NDF_2011",
    "HOME_OWN_MOR_VAL_1M_NDF_2013",
    "HOME_OWN_MOR_VAL_1M_NDF_2015",
    "HOME_OWN_MOR_VAL_1M_NDF_2017",
    "HOME_OWN_MOR_VAL_1M_NDF_2019",
    "HOME_OWN_MOR_VAL_1M_NDF_2021",
    "HOME_OWN_MOR_VAL_1M_RD_1969",
    "HOME_OWN_MOR_VAL_1M_RD_1970",
    "HOME_OWN_MOR_VAL_1M_RD_1971",
    "HOME_OWN_MOR_VAL_1M_RD_1972",
    "HOME_OWN_MOR_VAL_1M_RD_1976",
    "HOME_OWN_MOR_VAL_1M_RD_1977",
    "HOME_OWN_MOR_VAL_1M_RD_1978",
    "HOME_OWN_MOR_VAL_1M_RD_1979",
    "HOME_OWN_MOR_VAL_1M_RD_1980",
    "HOME_OWN_MOR_VAL_1M_RD_1981",
    "HOME_OWN_MOR_VAL_1M_RD_1983",
    "HOME_OWN_MOR_VAL_1M_RD_1984",
    "HOME_OWN_MOR_VAL_1M_RD_1985",
    "HOME_OWN_MOR_VAL_1M_RD_1986",
    "HOME_OWN_MOR_VAL_1M_RD_1987",
    "HOME_OWN_MOR_VAL_1M_RD_1988",
    "HOME_OWN_MOR_VAL_1M_RD_1989",
    "HOME_OWN_MOR_VAL_1M_RD_1990",
    "HOME_OWN_MOR_VAL_1M_RD_1991",
    "HOME_OWN_MOR_VAL_1M_RD_1992",
    "HOME_OWN_MOR_VAL_1M_RD_1993",
    "HOME_OWN_MOR_VAL_1M_RD_1994",
    "HOME_OWN_MOR_VAL_1M_RD_1995",
    "HOME_OWN_MOR_VAL_1M_RD_1996",
    "HOME_OWN_MOR_VAL_1M_RD_1997",
    "HOME_OWN_MOR_VAL_1M_RD_1999",
    "HOME_OWN_MOR_VAL_1M_RD_2001",
    "HOME_OWN_MOR_VAL_1M_RD_2003",
    "HOME_OWN_MOR_VAL_1M_RD_2005",
    "HOME_OWN_MOR_VAL_1M_RD_2007",
    "HOME_OWN_MOR_VAL_1M_RD_2009",
    "HOME_OWN_MOR_VAL_1M_RD_2011",
    "HOME_OWN_MOR_VAL_1M_RD_2013",
    "HOME_OWN_MOR_VAL_1M_RD_2015",
    "HOME_OWN_MOR_VAL_1M_RD_2017",
    "HOME_OWN_MOR_VAL_1M_RD_2019",
    "HOME_OWN_MOR_VAL_1M_RD_2021",
    "HOME_OWN_MOR_VAL_1M_RDF_1969",
    "HOME_OWN_MOR_VAL_1M_RDF_1970",
    "HOME_OWN_MOR_VAL_1M_RDF_1971",
    "HOME_OWN_MOR_VAL_1M_RDF_1972",
    "HOME_OWN_MOR_VAL_1M_RDF_1976",
    "HOME_OWN_MOR_VAL_1M_RDF_1977",
    "HOME_OWN_MOR_VAL_1M_RDF_1978",
    "HOME_OWN_MOR_VAL_1M_RDF_1979",
    "HOME_OWN_MOR_VAL_1M_RDF_1980",
    "HOME_OWN_MOR_VAL_1M_RDF_1981",
    "HOME_OWN_MOR_VAL_1M_RDF_1983",
    "HOME_OWN_MOR_VAL_1M_RDF_1984",
    "HOME_OWN_MOR_VAL_1M_RDF_1985",
    "HOME_OWN_MOR_VAL_1M_RDF_1986",
    "HOME_OWN_MOR_VAL_1M_RDF_1987",
    "HOME_OWN_MOR_VAL_1M_RDF_1988",
    "HOME_OWN_MOR_VAL_1M_RDF_1989",
    "HOME_OWN_MOR_VAL_1M_RDF_1990",
    "HOME_OWN_MOR_VAL_1M_RDF_1991",
    "HOME_OWN_MOR_VAL_1M_RDF_1992",
    "HOME_OWN_MOR_VAL_1M_RDF_1993",
    "HOME_OWN_MOR_VAL_1M_RDF_1994",
    "HOME_OWN_MOR_VAL_1M_RDF_1995",
    "HOME_OWN_MOR_VAL_1M_RDF_1996",
    "HOME_OWN_MOR_VAL_1M_RDF_1997",
    "HOME_OWN_MOR_VAL_1M_RDF_1999",
    "HOME_OWN_MOR_VAL_1M_RDF_2001",
    "HOME_OWN_MOR_VAL_1M_RDF_2003",
    "HOME_OWN_MOR_VAL_1M_RDF_2005",
    "HOME_OWN_MOR_VAL_1M_RDF_2007",
    "HOME_OWN_MOR_VAL_1M_RDF_2009",
    "HOME_OWN_MOR_VAL_1M_RDF_2011",
    "HOME_OWN_MOR_VAL_1M_RDF_2013",
    "HOME_OWN_MOR_VAL_1M_RDF_2015",
    "HOME_OWN_MOR_VAL_1M_RDF_2017",
    "HOME_OWN_MOR_VAL_1M_RDF_2019",
    "HOME_OWN_MOR_VAL_1M_RDF_2021",
    "HOME_OWN_MOR_VAL_2M_ND_1994",
    "HOME_OWN_MOR_VAL_2M_ND_1995",
    "HOME_OWN_MOR_VAL_2M_ND_1996",
    "HOME_OWN_MOR_VAL_2M_ND_1997",
    "HOME_OWN_MOR_VAL_2M_ND_1999",
    "HOME_OWN_MOR_VAL_2M_ND_2001",
    "HOME_OWN_MOR_VAL_2M_ND_2003",
    "HOME_OWN_MOR_VAL_2M_ND_2005",
    "HOME_OWN_MOR_VAL_2M_ND_2007",
    "HOME_OWN_MOR_VAL_2M_ND_2009",
    "HOME_OWN_MOR_VAL_2M_ND_2011",
    "HOME_OWN_MOR_VAL_2M_ND_2013",
    "HOME_OWN_MOR_VAL_2M_ND_2015",
    "HOME_OWN_MOR_VAL_2M_ND_2017",
    "HOME_OWN_MOR_VAL_2M_ND_2019",
    "HOME_OWN_MOR_VAL_2M_ND_2021",
    "HOME_OWN_MOR_VAL_2M_NDF_1994",
    "HOME_OWN_MOR_VAL_2M_NDF_1995",
    "HOME_OWN_MOR_VAL_2M_NDF_1996",
    "HOME_OWN_MOR_VAL_2M_NDF_1997",
    "HOME_OWN_MOR_VAL_2M_NDF_1999",
    "HOME_OWN_MOR_VAL_2M_NDF_2001",
    "HOME_OWN_MOR_VAL_2M_NDF_2003",
    "HOME_OWN_MOR_VAL_2M_NDF_2005",
    "HOME_OWN_MOR_VAL_2M_NDF_2007",
    "HOME_OWN_MOR_VAL_2M_NDF_2009",
    "HOME_OWN_MOR_VAL_2M_NDF_2011",
    "HOME_OWN_MOR_VAL_2M_NDF_2013",
    "HOME_OWN_MOR_VAL_2M_NDF_2015",
    "HOME_OWN_MOR_VAL_2M_NDF_2017",
    "HOME_OWN_MOR_VAL_2M_NDF_2019",
    "HOME_OWN_MOR_VAL_2M_NDF_2021",
    "HOME_OWN_MOR_VAL_2M_RD_1994",
    "HOME_OWN_MOR_VAL_2M_RD_1995",
    "HOME_OWN_MOR_VAL_2M_RD_1996",
    "HOME_OWN_MOR_VAL_2M_RD_1997",
    "HOME_OWN_MOR_VAL_2M_RD_1999",
    "HOME_OWN_MOR_VAL_2M_RD_2001",
    "HOME_OWN_MOR_VAL_2M_RD_2003",
    "HOME_OWN_MOR_VAL_2M_RD_2005",
    "HOME_OWN_MOR_VAL_2M_RD_2007",
    "HOME_OWN_MOR_VAL_2M_RD_2009",
    "HOME_OWN_MOR_VAL_2M_RD_2011",
    "HOME_OWN_MOR_VAL_2M_RD_2013",
    "HOME_OWN_MOR_VAL_2M_RD_2015",
    "HOME_OWN_MOR_VAL_2M_RD_2017",
    "HOME_OWN_MOR_VAL_2M_RD_2019",
    "HOME_OWN_MOR_VAL_2M_RD_2021",
    "HOME_OWN_MOR_VAL_2M_RDF_1994",
    "HOME_OWN_MOR_VAL_2M_RDF_1995",
    "HOME_OWN_MOR_VAL_2M_RDF_1996",
    "HOME_OWN_MOR_VAL_2M_RDF_1997",
    "HOME_OWN_MOR_VAL_2M_RDF_1999",
    "HOME_OWN_MOR_VAL_2M_RDF_2001",
    "HOME_OWN_MOR_VAL_2M_RDF_2003",
    "HOME_OWN_MOR_VAL_2M_RDF_2005",
    "HOME_OWN_MOR_VAL_2M_RDF_2007",
    "HOME_OWN_MOR_VAL_2M_RDF_2009",
    "HOME_OWN_MOR_VAL_2M_RDF_2011",
    "HOME_OWN_MOR_VAL_2M_RDF_2013",
    "HOME_OWN_MOR_VAL_2M_RDF_2015",
    "HOME_OWN_MOR_VAL_2M_RDF_2017",
    "HOME_OWN_MOR_VAL_2M_RDF_2019",
    "HOME_OWN_MOR_VAL_2M_RDF_2021"
  ],
  "race_ethnicity": [
    "ID",
    "LINEAGE",
    "PNUM",
    "RACE_ETH_MAJ",
    "RACE_ETH_MAJ_COL",
    "RACE_ETH_MM_MAJ",
    "RACE_ETH_MM_MAJ_COL",
    "RACE_ETH_MAJ_RP_1968",
    "RACE_ETH_MAJ_RP_1969",
    "RACE_ETH_MAJ_RP_1970",
    "RACE_ETH_MAJ_RP_1971",
    "RACE_ETH_MAJ_RP_1972",
    "RACE_ETH_MAJ_RP_1973",
    "RACE_ETH_MAJ_RP_1974",
    "RACE_ETH_MAJ_RP_1975",
    "RACE_ETH_MAJ_RP_1976",
    "RACE_ETH_MAJ_RP_1977",
    "RACE_ETH_MAJ_RP_1978",
    "RACE_ETH_MAJ_RP_1979",
    "RACE_ETH_MAJ_RP_1980",
    "RACE_ETH_MAJ_RP_1981",
    "RACE_ETH_MAJ_RP_1982",
    "RACE_ETH_MAJ_RP_1983",
    "RACE_ETH_MAJ_RP_1984",
    "RACE_ETH_MAJ_RP_1985",
    "RACE_ETH_MAJ_RP_1986",
    "RACE_ETH_MAJ_RP_1987",
    "RACE_ETH_MAJ_RP_1988",
    "RACE_ETH_MAJ_RP_1989",
    "RACE_ETH_MAJ_RP_1990",
    "RACE_ETH_MAJ_RP_1991",
    "RACE_ETH_MAJ_RP_1992",
    "RACE_ETH_MAJ_RP_1993",
    "RACE_ETH_MAJ_RP_1994",
    "RACE_ETH_MAJ_RP_1995",
    "RACE_ETH_MAJ_RP_1996",
    "RACE_ETH_MAJ_RP_1997",
    "RACE_ETH_MAJ_RP_1999",
    "RACE_ETH_MAJ_RP_2001",
    "RACE_ETH_MAJ_RP_2003",
    "RACE_ETH_MAJ_RP_2005",
    "RACE_ETH_MAJ_RP_2007",
    "RACE_ETH_MAJ_RP_2009",
    "RACE_ETH_MAJ_RP_2011",
    "RACE_ETH_MAJ_RP_2013",
    "RACE_ETH_MAJ_RP_2015",
    "RACE_ETH_MAJ_RP_2017",
    "RACE_ETH_MAJ_RP_2019",
    "RACE_ETH_MAJ_RP_2021",
    "RACE_ETH_MAJ_SP_1968",
    "RACE_ETH_MAJ_SP_1969",
    "RACE_ETH_MAJ_SP_1970",
    "RACE_ETH_MAJ_SP_1971",
    "RACE_ETH_MAJ_SP_1972",
    "RACE_ETH_MAJ_SP_1973",
    "RACE_ETH_MAJ_SP_1974",
    "RACE_ETH_MAJ_SP_1975",
    "RACE_ETH_MAJ_SP_1976",
    "RACE_ETH_MAJ_SP_1977",
    "RACE_ETH_MAJ_SP_1978",
    "RACE_ETH_MAJ_SP_1979",
    "RACE_ETH_MAJ_SP_1980",
    "RACE_ETH_MAJ_SP_1981",
    "RACE_ETH_MAJ_SP_1982",
    "RACE_ETH_MAJ_SP_1983",
    "RACE_ETH_MAJ_SP_1984",
    "RACE_ETH_MAJ_SP_1985",
    "RACE_ETH_MAJ_SP_1986",
    "RACE_ETH_MAJ_SP_1987",
    "RACE_ETH_MAJ_SP_1988",
    "RACE_ETH_MAJ_SP_1989",
    "RACE_ETH_MAJ_SP_1990",
    "RACE_ETH_MAJ_SP_1991",
    "RACE_ETH_MAJ_SP_1992",
    "RACE_ETH_MAJ_SP_1993",
    "RACE_ETH_MAJ_SP_1994",
    "RACE_ETH_MAJ_SP_1995",
    "RACE_ETH_MAJ_SP_1996",
    "RACE_ETH_MAJ_SP_1997",
    "RACE_ETH_MAJ_SP_1999",
    "RACE_ETH_MAJ_SP_2001",
    "RACE_ETH_MAJ_SP_2003",
    "RACE_ETH_MAJ_SP_2005",
    "RACE_ETH_MAJ_SP_2007",
    "RACE_ETH_MAJ_SP_2009",
    "RACE_ETH_MAJ_SP_2011",
    "RACE_ETH_MAJ_SP_2013",
    "RACE_ETH_MAJ_SP_2015",
    "RACE_ETH_MAJ_SP_2017",
    "RACE_ETH_MAJ_SP_2019",
    "RACE_ETH_MAJ_SP_2021",
    "RACE_ETH_MAJ_COL_RP_1968",
    "RACE_ETH_MAJ_COL_RP_1969",
    "RACE_ETH_MAJ_COL_RP_1970",
    "RACE_ETH_MAJ_COL_RP_1971",
    "RACE_ETH_MAJ_COL_RP_1972",
    "RACE_ETH_MAJ_COL_RP_1973",
    "RACE_ETH_MAJ_COL_RP_1974",
    "RACE_ETH_MAJ_COL_RP_1975",
    "RACE_ETH_MAJ_COL_RP_1976",
    "RACE_ETH_MAJ_COL_RP_1977",
    "RACE_ETH_MAJ_COL_RP_1978",
    "RACE_ETH_MAJ_COL_RP_1979",
    "RACE_ETH_MAJ_COL_RP_1980",
    "RACE_ETH_MAJ_COL_RP_1981",
    "RACE_ETH_MAJ_COL_RP_1982",
    "RACE_ETH_MAJ_COL_RP_1983",
    "RACE_ETH_MAJ_COL_RP_1984",
    "RACE_ETH_MAJ_COL_RP_1985",
    "RACE_ETH_MAJ_COL_RP_1986",
    "RACE_ETH_MAJ_COL_RP_1987",
    "RACE_ETH_MAJ_COL_RP_1988",
    "RACE_ETH_MAJ_COL_RP_1989",
    "RACE_ETH_MAJ_COL_RP_1990",
    "RACE_ETH_MAJ_COL_RP_1991",
    "RACE_ETH_MAJ_COL_RP_1992",
    "RACE_ETH_MAJ_COL_RP_1993",
    "RACE_ETH_MAJ_COL_RP_1994",
    "RACE_ETH_MAJ_COL_RP_1995",
    "RACE_ETH_MAJ_COL_RP_1996",
    "RACE_ETH_MAJ_COL_RP_1997",
    "RACE_ETH_MAJ_COL_RP_1999",
    "RACE_ETH_MAJ_COL_RP_2001",
    "RACE_ETH_MAJ_COL_RP_2003",
    "RACE_ETH_MAJ_COL_RP_2005",
    "RACE_ETH_MAJ_COL_RP_2007",
    "RACE_ETH_MAJ_COL_RP_2009",
    "RACE_ETH_MAJ_COL_RP_2011",
    "RACE_ETH_MAJ_COL_RP_2013",
    "RACE_ETH_MAJ_COL_RP_2015",
    "RACE_ETH_MAJ_COL_RP_2017",
    "RACE_ETH_MAJ_COL_RP_2019",
    "RACE_ETH_MAJ_COL_RP_2021",
    "RACE_ETH_MAJ_COL_SP_1968",
    "RACE_ETH_MAJ_COL_SP_1969",
    "RACE_ETH_MAJ_COL_SP_1970",
    "RACE_ETH_MAJ_COL_SP_1971",
    "RACE_ETH_MAJ_COL_SP_1972",
    "RACE_ETH_MAJ_COL_SP_1973",
    "RACE_ETH_MAJ_COL_SP_1974",
    "RACE_ETH_MAJ_COL_SP_1975",
    "RACE_ETH_MAJ_COL_SP_1976",
    "RACE_ETH_MAJ_COL_SP_1977",
    "RACE_ETH_MAJ_COL_SP_1978",
    "RACE_ETH_MAJ_COL_SP_1979",
    "RACE_ETH_MAJ_COL_SP_1980",
    "RACE_ETH_MAJ_COL_SP_1981",
    "RACE_ETH_MAJ_COL_SP_1982",
    "RACE_ETH_MAJ_COL_SP_1983",
    "RACE_ETH_MAJ_COL_SP_1984",
    "RACE_ETH_MAJ_COL_SP_1985",
    "RACE_ETH_MAJ_COL_SP_1986",
    "RACE_ETH_MAJ_COL_SP_1987",
    "RACE_ETH_MAJ_COL_SP_1988",
    "RACE_ETH_MAJ_COL_SP_1989",
    "RACE_ETH_MAJ_COL_SP_1990",
    "RACE_ETH_MAJ_COL_SP_1991",
    "RACE_ETH_MAJ_COL_SP_1992",
    "RACE_ETH_MAJ_COL_SP_1993",
    "RACE_ETH_MAJ_COL_SP_1994",
    "RACE_ETH_MAJ_COL_SP_1995",
    "RACE_ETH_MAJ_COL_SP_1996",
    "RACE_ETH_MAJ_COL_SP_1997",
    "RACE_ETH_MAJ_COL_SP_1999",
    "RACE_ETH_MAJ_COL_SP_2001",
    "RACE_ETH_MAJ_COL_SP_2003",
    "RACE_ETH_MAJ_COL_SP_2005",
    "RACE_ETH_MAJ_COL_SP_2007",
    "RACE_ETH_MAJ_COL_SP_2009",
    "RACE_ETH_MAJ_COL_SP_2011",
    "RACE_ETH_MAJ_COL_SP_2013",
    "RACE_ETH_MAJ_COL_SP_2015",
    "RACE_ETH_MAJ_COL_SP_2017",
    "RACE_ETH_MAJ_COL_SP_2019",
    "RACE_ETH_MAJ_COL_SP_2021",
    "RACE_ETH_MM_MAJ_RP_1968",
    "RACE_ETH_MM_MAJ_RP_1969",
    "RACE_ETH_MM_MAJ_RP_1970",
    "RACE_ETH_MM_MAJ_RP_1971",
    "RACE_ETH_MM_MAJ_RP_1972",
    "RACE_ETH_MM_MAJ_RP_1973",
    "RACE_ETH_MM_MAJ_RP_1974",
    "RACE_ETH_MM_MAJ_RP_1975",
    "RACE_ETH_MM_MAJ_RP_1976",
    "RACE_ETH_MM_MAJ_RP_1977",
    "RACE_ETH_MM_MAJ_RP_1978",
    "RACE_ETH_MM_MAJ_RP_1979",
    "RACE_ETH_MM_MAJ_RP_1980",
    "RACE_ETH_MM_MAJ_RP_1981",
    "RACE_ETH_MM_MAJ_RP_1982",
    "RACE_ETH_MM_MAJ_RP_1983",
    "RACE_ETH_MM_MAJ_RP_1984",
    "RACE_ETH_MM_MAJ_RP_1985",
    "RACE_ETH_MM_MAJ_RP_1986",
    "RACE_ETH_MM_MAJ_RP_1987",
    "RACE_ETH_MM_MAJ_RP_1988",
    "RACE_ETH_MM_MAJ_RP_1989",
    "RACE_ETH_MM_MAJ_RP_1990",
    "RACE_ETH_MM_MAJ_RP_1991",
    "RACE_ETH_MM_MAJ_RP_1992",
    "RACE_ETH_MM_MAJ_RP_1993",
    "RACE_ETH_MM_MAJ_RP_1994",
    "RACE_ETH_MM_MAJ_RP_1995",
    "RACE_ETH_MM_MAJ_RP_1996",
    "RACE_ETH_MM_MAJ_RP_1997",
    "RACE_ETH_MM_MAJ_RP_1999",
    "RACE_ETH_MM_MAJ_RP_2001",
    "RACE_ETH_MM_MAJ_RP_2003",
    "RACE_ETH_MM_MAJ_RP_2005",
    "RACE_ETH_MM_MAJ_RP_2007",
    "RACE_ETH_MM_MAJ_RP_2009",
    "RACE_ETH_MM_MAJ_RP_2011",
    "RACE_ETH_MM_MAJ_RP_2013",
    "RACE_ETH_MM_MAJ_RP_2015",
    "RACE_ETH_MM_MAJ_RP_2017",
    "RACE_ETH_MM_MAJ_RP_2019",
    "RACE_ETH_MM_MAJ_RP_2021",
    "RACE_ETH_MM_MAJ_SP_1968",
    "RACE_ETH_MM_MAJ_SP_1969",
    "RACE_ETH_MM_MAJ_SP_1970",
    "RACE_ETH_MM_MAJ_SP_1971",
    "RACE_ETH_MM_MAJ_SP_1972",
    "RACE_ETH_MM_MAJ_SP_1973",
    "RACE_ETH_MM_MAJ_SP_1974",
    "RACE_ETH_MM_MAJ_SP_1975",
    "RACE_ETH_MM_MAJ_SP_1976",
    "RACE_ETH_MM_MAJ_SP_1977",
    "RACE_ETH_MM_MAJ_SP_1978",
    "RACE_ETH_MM_MAJ_SP_1979",
    "RACE_ETH_MM_MAJ_SP_1980",
    "RACE_ETH_MM_MAJ_SP_1981",
    "RACE_ETH_MM_MAJ_SP_1982",
    "RACE_ETH_MM_MAJ_SP_1983",
    "RACE_ETH_MM_MAJ_SP_1984",
    "RACE_ETH_MM_MAJ_SP_1985",
    "RACE_ETH_MM_MAJ_SP_1986",
    "RACE_ETH_MM_MAJ_SP_1987",
    "RACE_ETH_MM_MAJ_SP_1988",
    "RACE_ETH_MM_MAJ_SP_1989",
    "RACE_ETH_MM_MAJ_SP_1990",
    "RACE_ETH_MM_MAJ_SP_1991",
    "RACE_ETH_MM_MAJ_SP_1992",
    "RACE_ETH_MM_MAJ_SP_1993",
    "RACE_ETH_MM_MAJ_SP_1994",
    "RACE_ETH_MM_MAJ_SP_1995",
    "RACE_ETH_MM_MAJ_SP_1996",
    "RACE_ETH_MM_MAJ_SP_1997",
    "RACE_ETH_MM_MAJ_SP_1999",
    "RACE_ETH_MM_MAJ_SP_2001",
    "RACE_ETH_MM_MAJ_SP_2003",
    "RACE_ETH_MM_MAJ_SP_2005",
    "RACE_ETH_MM_MAJ_SP_2007",
    "RACE_ETH_MM_MAJ_SP_2009",
    "RACE_ETH_MM_MAJ_SP_2011",
    "RACE_ETH_MM_MAJ_SP_2013",
    "RACE_ETH_MM_MAJ_SP_2015",
    "RACE_ETH_MM_MAJ_SP_2017",
    "RACE_ETH_MM_MAJ_SP_2019",
    "RACE_ETH_MM_MAJ_SP_2021",
    "RACE_ETH_MM_MAJ_COL_RP_1968",
    "RACE_ETH_MM_MAJ_COL_RP_1969",
    "RACE_ETH_MM_MAJ_COL_RP_1970",
    "RACE_ETH_MM_MAJ_COL_RP_1971",
    "RACE_ETH_MM_MAJ_COL_RP_1972",
    "RACE_ETH_MM_MAJ_COL_RP_1973",
    "RACE_ETH_MM_MAJ_COL_RP_1974",
    "RACE_ETH_MM_MAJ_COL_RP_1975",
    "RACE_ETH_MM_MAJ_COL_RP_1976",
    "RACE_ETH_MM_MAJ_COL_RP_1977",
    "RACE_ETH_MM_MAJ_COL_RP_1978",
    "RACE_ETH_MM_MAJ_COL_RP_1979",
    "RACE_ETH_MM_MAJ_COL_RP_1980",
    "RACE_ETH_MM_MAJ_COL_RP_1981",
    "RACE_ETH_MM_MAJ_COL_RP_1982",
    "RACE_ETH_MM_MAJ_COL_RP_1983",
    "RACE_ETH_MM_MAJ_COL_RP_1984",
    "RACE_ETH_MM_MAJ_COL_RP_1985",
    "RACE_ETH_MM_MAJ_COL_RP_1986",
    "RACE_ETH_MM_MAJ_COL_RP_1987",
    "RACE_ETH_MM_MAJ_COL_RP_1988",
    "RACE_ETH_MM_MAJ_COL_RP_1989",
    "RACE_ETH_MM_MAJ_COL_RP_1990",
    "RACE_ETH_MM_MAJ_COL_RP_1991",
    "RACE_ETH_MM_MAJ_COL_RP_1992",
    "RACE_ETH_MM_MAJ_COL_RP_1993",
    "RACE_ETH_MM_MAJ_COL_RP_1994",
    "RACE_ETH_MM_MAJ_COL_RP_1995",
    "RACE_ETH_MM_MAJ_COL_RP_1996",
    "RACE_ETH_MM_MAJ_COL_RP_1997",
    "RACE_ETH_MM_MAJ_COL_RP_1999",
    "RACE_ETH_MM_MAJ_COL_RP_2001",
    "RACE_ETH_MM_MAJ_COL_RP_2003",
    "RACE_ETH_MM_MAJ_COL_RP_2005",
    "RACE_ETH_MM_MAJ_COL_RP_2007",
    "RACE_ETH_MM_MAJ_COL_RP_2009",
    "RACE_ETH_MM_MAJ_COL_RP_2011",
    "RACE_ETH_MM_MAJ_COL_RP_2013",
    "RACE_ETH_MM_MAJ_COL_RP_2015",
    "RACE_ETH_MM_MAJ_COL_RP_2017",
    "RACE_ETH_MM_MAJ_COL_RP_2019",
    "RACE_ETH_MM_MAJ_COL_RP_2021",
    "RACE_ETH_MM_MAJ_COL_SP_1968",
    "RACE_ETH_MM_MAJ_COL_SP_1969",
    "RACE_ETH_MM_MAJ_COL_SP_1970",
    "RACE_ETH_MM_MAJ_COL_SP_1971",
    "RACE_ETH_MM_MAJ_COL_SP_1972",
    "RACE_ETH_MM_MAJ_COL_SP_1973",
    "RACE_ETH_MM_MAJ_COL_SP_1974",
    "RACE_ETH_MM_MAJ_COL_SP_1975",
    "RACE_ETH_MM_MAJ_COL_SP_1976",
    "RACE_ETH_MM_MAJ_COL_SP_1977",
    "RACE_ETH_MM_MAJ_COL_SP_1978",
    "RACE_ETH_MM_MAJ_COL_SP_1979",
    "RACE_ETH_MM_MAJ_COL_SP_1980",
    "RACE_ETH_MM_MAJ_COL_SP_1981",
    "RACE_ETH_MM_MAJ_COL_SP_1982",
    "RACE_ETH_MM_MAJ_COL_SP_1983",
    "RACE_ETH_MM_MAJ_COL_SP_1984",
    "RACE_ETH_MM_MAJ_COL_SP_1985",
    "RACE_ETH_MM_MAJ_COL_SP_1986",
    "RACE_ETH_MM_MAJ_COL_SP_1987",
    "RACE_ETH_MM_MAJ_COL_SP_1988",
    "RACE_ETH_MM_MAJ_COL_SP_1989",
    "RACE_ETH_MM_MAJ_COL_SP_1990",
    "RACE_ETH_MM_MAJ_COL_SP_1991",
    "RACE_ETH_MM_MAJ_COL_SP_1992",
    "RACE_ETH_MM_MAJ_COL_SP_1993",
    "RACE_ETH_MM_MAJ_COL_SP_1994",
    "RACE_ETH_MM_MAJ_COL_SP_1995",
    "RACE_ETH_MM_MAJ_COL_SP_1996",
    "RACE_ETH_MM_MAJ_COL_SP_1997",
    "RACE_ETH_MM_MAJ_COL_SP_1999",
    "RACE_ETH_MM_MAJ_COL_SP_2001",
    "RACE_ETH_MM_MAJ_COL_SP_2003",
    "RACE_ETH_MM_MAJ_COL_SP_2005",
    "RACE_ETH_MM_MAJ_COL_SP_2007",
    "RACE_ETH_MM_MAJ_COL_SP_2009",
    "RACE_ETH_MM_MAJ_COL_SP_2011",
    "RACE_ETH_MM_MAJ_COL_SP_2013",
    "RACE_ETH_MM_MAJ_COL_SP_2015",
    "RACE_ETH_MM_MAJ_COL_SP_2017",
    "RACE_ETH_MM_MAJ_COL_SP_2019",
    "RACE_ETH_MM_MAJ_COL_SP_2021",
    "RACE_ONLY_1M_1968",
    "RACE_ONLY_1M_1969",
    "RACE_ONLY_1M_1970",
    "RACE_ONLY_1M_1971",
    "RACE_ONLY_1M_1972",
    "RACE_ONLY_1M_1973",
    "RACE_ONLY_1M_1974",
    "RACE_ONLY_1M_1975",
    "RACE_ONLY_1M_1976",
    "RACE_ONLY_1M_1977",
    "RACE_ONLY_1M_1978",
    "RACE_ONLY_1M_1979",
    "RACE_ONLY_1M_1980",
    "RACE_ONLY_1M_1981",
    "RACE_ONLY_1M_1982",
    "RACE_ONLY_1M_1983",
    "RACE_ONLY_1M_1984",
    "RACE_ONLY_1M_1985",
    "RACE_ONLY_1M_1986",
    "RACE_ONLY_1M_1987",
    "RACE_ONLY_1M_1988",
    "RACE_ONLY_1M_1989",
    "RACE_ONLY_1M_1990",
    "RACE_ONLY_1M_1991",
    "RACE_ONLY_1M_1992",
    "RACE_ONLY_1M_1993",
    "RACE_ONLY_1M_1994",
    "RACE_ONLY_1M_1995",
    "RACE_ONLY_1M_1996",
    "RACE_ONLY_1M_1997",
    "RACE_ONLY_1M_1999",
    "RACE_ONLY_1M_2001",
    "RACE_ONLY_1M_2003",
    "RACE_ONLY_1M_2005",
    "RACE_ONLY_1M_2007",
    "RACE_ONLY_1M_2009",
    "RACE_ONLY_1M_2011",
    "RACE_ONLY_1M_2013",
    "RACE_ONLY_1M_2015",
    "RACE_ONLY_1M_2017",
    "RACE_ONLY_1M_2019",
    "RACE_ONLY_1M_2021",
    "RACE_ONLY_2M_1985",
    "RACE_ONLY_2M_1986",
    "RACE_ONLY_2M_1987",
    "RACE_ONLY_2M_1988",
    "RACE_ONLY_2M_1989",
    "RACE_ONLY_2M_1990",
    "RACE_ONLY_2M_1991",
    "RACE_ONLY_2M_1992",
    "RACE_ONLY_2M_1993",
    "RACE_ONLY_2M_1994",
    "RACE_ONLY_2M_1995",
    "RACE_ONLY_2M_1996",
    "RACE_ONLY_2M_1997",
    "RACE_ONLY_2M_1999",
    "RACE_ONLY_2M_2001",
    "RACE_ONLY_2M_2003",
    "RACE_ONLY_2M_2005",
    "RACE_ONLY_2M_2007",
    "RACE_ONLY_2M_2009",
    "RACE_ONLY_2M_2011",
    "RACE_ONLY_2M_2013",
    "RACE_ONLY_2M_2015",
    "RACE_ONLY_2M_2017",
    "RACE_ONLY_2M_2019",
    "RACE_ONLY_2M_2021",
    "RACE_ONLY_3M_1994",
    "RACE_ONLY_3M_1995",
    "RACE_ONLY_3M_1996",
    "RACE_ONLY_3M_1997",
    "RACE_ONLY_3M_1999",
    "RACE_ONLY_3M_2001",
    "RACE_ONLY_3M_2003",
    "RACE_ONLY_3M_2005",
    "RACE_ONLY_3M_2007",
    "RACE_ONLY_3M_2009",
    "RACE_ONLY_3M_2011",
    "RACE_ONLY_3M_2013",
    "RACE_ONLY_3M_2015",
    "RACE_ONLY_3M_2017",
    "RACE_ONLY_3M_2019",
    "RACE_ONLY_3M_2021",
    "RACE_ONLY_4M_1997",
    "RACE_ONLY_4M_1999",
    "RACE_ONLY_4M_2005",
    "RACE_ONLY_4M_2007",
    "RACE_ONLY_4M_2009",
    "RACE_ONLY_4M_2011",
    "RACE_ONLY_4M_2013",
    "RACE_ONLY_4M_2015",
    "RACE_ONLY_4M_2017",
    "RACE_ONLY_4M_2019",
    "RACE_ONLY_4M_2021"
  ],
  "time_use": [
    "ID",
    "LINEAGE",
    "PNUM",
    "TIME_ACAR_2017",
    "TIME_ACAR_2019",
    "TIME_ACAR_2021",
    "TIME_CCAR_2017",
    "TIME_CCAR_2019",
    "TIME_CCAR_2021",
    "TIME_EDUC_2017",
    "TIME_EDUC_2019",
    "TIME_EDUC_2021",
    "TIME_HOUS_1976",
    "TIME_HOUS_1977",
    "TIME_HOUS_1978",
    "TIME_HOUS_1979",
    "TIME_HOUS_1980",
    "TIME_HOUS_1981",
    "TIME_HOUS_1983",
    "TIME_HOUS_1984",
    "TIME_HOUS_1985",
    "TIME_HOUS_1986",
    "TIME_HOUS_1987",
    "TIME_HOUS_1988",
    "TIME_HOUS_1989",
    "TIME_HOUS_1990",
    "TIME_HOUS_1991",
    "TIME_HOUS_1992",
    "TIME_HOUS_1993",
    "TIME_HOUS_1994",
    "TIME_HOUS_1995",
    "TIME_HOUS_1996",
    "TIME_HOUS_1997",
    "TIME_HOUS_1999",
    "TIME_HOUS_2001",
    "TIME_HOUS_2003",
    "TIME_HOUS_2005",
    "TIME_HOUS_2007",
    "TIME_HOUS_2009",
    "TIME_HOUS_2011",
    "TIME_HOUS_2013",
    "TIME_HOUS_2015",
    "TIME_HOUS_2017",
    "TIME_HOUS_2019",
    "TIME_HOUS_2021",
    "TIME_LEIS_2017",
    "TIME_LEIS_2019",
    "TIME_LEIS_2021",
    "TIME_PERS_2017",
    "TIME_PERS_2019",
    "TIME_PERS_2021",
    "TIME_SHOP_2017",
    "TIME_SHOP_2019",
    "TIME_SHOP_2021",
    "TIME_VOLU_2017",
    "TIME_VOLU_2019",
    "TIME_VOLU_2021",
    "TIME_WORK_2017",
    "TIME_WORK_2019",
    "TIME_WORK_2021"
  ],
  "wealth_business": [
    "ID",
    "LINEAGE",
    "PNUM",
    "WLTH_FBUS_NET_ND_1984",
    "WLTH_FBUS_NET_ND_1989",
    "WLTH_FBUS_NET_ND_1994",
    "WLTH_FBUS_NET_ND_1999",
    "WLTH_FBUS_NET_ND_2001",
    "WLTH_FBUS_NET_ND_2003",
    "WLTH_FBUS_NET_ND_2005",
    "WLTH_FBUS_NET_ND_2007",
    "WLTH_FBUS_NET_ND_2009",
    "WLTH_FBUS_NET_ND_2011",
    "WLTH_FBUS_NET_ND_2013",
    "WLTH_FBUS_NET_ND_2015",
    "WLTH_FBUS_NET_ND_2017",
    "WLTH_FBUS_NET_ND_2019",
    "WLTH_FBUS_NET_ND_2021",
    "WLTH_FBUS_NET_NDF_1984",
    "WLTH_FBUS_NET_NDF_1989",
    "WLTH_FBUS_NET_NDF_1994",
    "WLTH_FBUS_NET_NDF_1999",
    "WLTH_FBUS_NET_NDF_2001",
    "WLTH_FBUS_NET_NDF_2003",
    "WLTH_FBUS_NET_NDF_2005",
    "WLTH_FBUS_NET_NDF_2007",
    "WLTH_FBUS_NET_NDF_2009",
    "WLTH_FBUS_NET_NDF_2011",
    "WLTH_FBUS_NET_NDF_2013",
    "WLTH_FBUS_NET_NDF_2015",
    "WLTH_FBUS_NET_NDF_2017",
    "WLTH_FBUS_NET_NDF_2019",
    "WLTH_FBUS_NET_NDF_2021",
    "WLTH_FBUS_NET_RD_1984",
    "WLTH_FBUS_NET_RD_1989",
    "WLTH_FBUS_NET_RD_1994",
    "WLTH_FBUS_NET_RD_1999",
    "WLTH_FBUS_NET_RD_2001",
    "WLTH_FBUS_NET_RD_2003",
    "WLTH_FBUS_NET_RD_2005",
    "WLTH_FBUS_NET_RD_2007",
    "WLTH_FBUS_NET_RD_2009",
    "WLTH_FBUS_NET_RD_2011",
    "WLTH_FBUS_NET_RD_2013",
    "WLTH_FBUS_NET_RD_2015",
    "WLTH_FBUS_NET_RD_2017",
    "WLTH_FBUS_NET_RD_2019",
    "WLTH_FBUS_NET_RD_2021",
    "WLTH_FBUS_NET_RDF_1984",
    "WLTH_FBUS_NET_RDF_1989",
    "WLTH_FBUS_NET_RDF_1994",
    "WLTH_FBUS_NET_RDF_1999",
    "WLTH_FBUS_NET_RDF_2001",
    "WLTH_FBUS_NET_RDF_2003",
    "WLTH_FBUS_NET_RDF_2005",
    "WLTH_FBUS_NET_RDF_2007",
    "WLTH_FBUS_NET_RDF_2009",
    "WLTH_FBUS_NET_RDF_2011",
    "WLTH_FBUS_NET_RDF_2013",
    "WLTH_FBUS_NET_RDF_2015",
    "WLTH_FBUS_NET_RDF_2017",
    "WLTH_FBUS_NET_RDF_2019",
    "WLTH_FBUS_NET_RDF_2021",
    "WLTH_FBUS_ASS_ND_2013",
    "WLTH_FBUS_ASS_ND_2015",
    "WLTH_FBUS_ASS_ND_2017",
    "WLTH_FBUS_ASS_ND_2019",
    "WLTH_FBUS_ASS_ND_2021",
    "WLTH_FBUS_ASS_NDF_2013",
    "WLTH_FBUS_ASS_NDF_2015",
    "WLTH_FBUS_ASS_NDF_2017",
    "WLTH_FBUS_ASS_NDF_2019",
    "WLTH_FBUS_ASS_NDF_2021",
    "WLTH_FBUS_ASS_RD_2013",
    "WLTH_FBUS_ASS_RD_2015",
    "WLTH_FBUS_ASS_RD_2017",
    "WLTH_FBUS_ASS_RD_2019",
    "WLTH_FBUS_ASS_RD_2021",
    "WLTH_FBUS_ASS_RDF_2013",
    "WLTH_FBUS_ASS_RDF_2015",
    "WLTH_FBUS_ASS_RDF_2017",
    "WLTH_FBUS_ASS_RDF_2019",
    "WLTH_FBUS_ASS_RDF_2021",
    "WLTH_FBUS_DEB_ND_2013",
    "WLTH_FBUS_DEB_ND_2015",
    "WLTH_FBUS_DEB_ND_2017",
    "WLTH_FBUS_DEB_ND_2019",
    "WLTH_FBUS_DEB_ND_2021",
    "WLTH_FBUS_DEB_NDF_2013",
    "WLTH_FBUS_DEB_NDF_2015",
    "WLTH_FBUS_DEB_NDF_2017",
    "WLTH_FBUS_DEB_NDF_2019",
    "WLTH_FBUS_DEB_NDF_2021",
    "WLTH_FBUS_DEB_RD_2013",
    "WLTH_FBUS_DEB_RD_2015",
    "WLTH_FBUS_DEB_RD_2017",
    "WLTH_FBUS_DEB_RD_2019",
    "WLTH_FBUS_DEB_RD_2021",
    "WLTH_FBUS_DEB_RDF_2013",
    "WLTH_FBUS_DEB_RDF_2015",
    "WLTH_FBUS_DEB_RDF_2017",
    "WLTH_FBUS_DEB_RDF_2019",
    "WLTH_FBUS_DEB_RDF_2021"
  ],
  "wealth_debt": [
    "ID",
    "LINEAGE",
    "PNUM",
    "WLTH_ODEB_NET_ND_1984",
    "WLTH_ODEB_NET_ND_1989",
    "WLTH_ODEB_NET_ND_1994",
    "WLTH_ODEB_NET_ND_1999",
    "WLTH_ODEB_NET_ND_2001",
    "WLTH_ODEB_NET_ND_2003",
    "WLTH_ODEB_NET_ND_2005",
    "WLTH_ODEB_NET_ND_2007",
    "WLTH_ODEB_NET_ND_2009",
    "WLTH_ODEB_NET_ND_2011",
    "WLTH_ODEB_NET_ND_2013",
    "WLTH_ODEB_NET_ND_2015",
    "WLTH_ODEB_NET_ND_2017",
    "WLTH_ODEB_NET_ND_2019",
    "WLTH_ODEB_NET_ND_2021",
    "WLTH_ODEB_NET_NDF_1984",
    "WLTH_ODEB_NET_NDF_1989",
    "WLTH_ODEB_NET_NDF_1994",
    "WLTH_ODEB_NET_NDF_1999",
    "WLTH_ODEB_NET_NDF_2001",
    "WLTH_ODEB_NET_NDF_2003",
    "WLTH_ODEB_NET_NDF_2005",
    "WLTH_ODEB_NET_NDF_2007",
    "WLTH_ODEB_NET_NDF_2009",
    "WLTH_ODEB_NET_NDF_2011",
    "WLTH_ODEB_NET_NDF_2013",
    "WLTH_ODEB_NET_NDF_2015",
    "WLTH_ODEB_NET_NDF_2017",
    "WLTH_ODEB_NET_NDF_2019",
    "WLTH_ODEB_NET_NDF_2021",
    "WLTH_ODEB_NET_RD_1984",
    "WLTH_ODEB_NET_RD_1989",
    "WLTH_ODEB_NET_RD_1994",
    "WLTH_ODEB_NET_RD_1999",
    "WLTH_ODEB_NET_RD_2001",
    "WLTH_ODEB_NET_RD_2003",
    "WLTH_ODEB_NET_RD_2005",
    "WLTH_ODEB_NET_RD_2007",
    "WLTH_ODEB_NET_RD_2009",
    "WLTH_ODEB_NET_RD_2011",
    "WLTH_ODEB_NET_RD_2013",
    "WLTH_ODEB_NET_RD_2015",
    "WLTH_ODEB_NET_RD_2017",
    "WLTH_ODEB_NET_RD_2019",
    "WLTH_ODEB_NET_RD_2021",
    "WLTH_ODEB_NET_RDF_1984",
    "WLTH_ODEB_NET_RDF_1989",
    "WLTH_ODEB_NET_RDF_1994",
    "WLTH_ODEB_NET_RDF_1999",
    "WLTH_ODEB_NET_RDF_2001",
    "WLTH_ODEB_NET_RDF_2003",
    "WLTH_ODEB_NET_RDF_2005",
    "WLTH_ODEB_NET_RDF_2007",
    "WLTH_ODEB_NET_RDF_2009",
    "WLTH_ODEB_NET_RDF_2011",
    "WLTH_ODEB_NET_RDF_2013",
    "WLTH_ODEB_NET_RDF_2015",
    "WLTH_ODEB_NET_RDF_2017",
    "WLTH_ODEB_NET_RDF_2019",
    "WLTH_ODEB_NET_RDF_2021",
    "WLTH_ODEB_CRE_ND_2011",
    "WLTH_ODEB_CRE_ND_2013",
    "WLTH_ODEB_CRE_ND_2015",
    "WLTH_ODEB_CRE_ND_2017",
    "WLTH_ODEB_CRE_ND_2019",
    "WLTH_ODEB_CRE_ND_2021",
    "WLTH_ODEB_CRE_NDF_2011",
    "WLTH_ODEB_CRE_NDF_2013",
    "WLTH_ODEB_CRE_NDF_2015",
    "WLTH_ODEB_CRE_NDF_2017",
    "WLTH_ODEB_CRE_NDF_2019",
    "WLTH_ODEB_CRE_NDF_2021",
    "WLTH_ODEB_CRE_RD_2011",
    "WLTH_ODEB_CRE_RD_2013",
    "WLTH_ODEB_CRE_RD_2015",
    "WLTH_ODEB_CRE_RD_2017",
    "WLTH_ODEB_CRE_RD_2019",
    "WLTH_ODEB_CRE_RD_2021",
    "WLTH_ODEB_CRE_RDF_2011",
    "WLTH_ODEB_CRE_RDF_2013",
    "WLTH_ODEB_CRE_RDF_2015",
    "WLTH_ODEB_CRE_RDF_2017",
    "WLTH_ODEB_CRE_RDF_2019",
    "WLTH_ODEB_CRE_RDF_2021",
    "WLTH_ODEB_FAM_ND_2011",
    "WLTH_ODEB_FAM_ND_2013",
    "WLTH_ODEB_FAM_ND_2015",
    "WLTH_ODEB_FAM_ND_2017",
    "WLTH_ODEB_FAM_ND_2019",
    "WLTH_ODEB_FAM_ND_2021",
    "WLTH_ODEB_FAM_NDF_2011",
    "WLTH_ODEB_FAM_NDF_2013",
    "WLTH_ODEB_FAM_NDF_2015",
    "WLTH_ODEB_FAM_NDF_2017",
    "WLTH_ODEB_FAM_NDF_2019",
    "WLTH_ODEB_FAM_NDF_2021",
    "WLTH_ODEB_FAM_RD_2011",
    "WLTH_ODEB_FAM_RD_2013",
    "WLTH_ODEB_FAM_RD_2015",
    "WLTH_ODEB_FAM_RD_2017",
    "WLTH_ODEB_FAM_RD_2019",
    "WLTH_ODEB_FAM_RD_2021",
    "WLTH_ODEB_FAM_RDF_2011",
    "WLTH_ODEB_FAM_RDF_2013",
    "WLTH_ODEB_FAM_RDF_2015",
    "WLTH_ODEB_FAM_RDF_2017",
    "WLTH_ODEB_FAM_RDF_2019",
    "WLTH_ODEB_FAM_RDF_2021",
    "WLTH_ODEB_LEG_ND_2011",
    "WLTH_ODEB_LEG_ND_2013",
    "WLTH_ODEB_LEG_ND_2015",
    "WLTH_ODEB_LEG_ND_2017",
    "WLTH_ODEB_LEG_ND_2019",
    "WLTH_ODEB_LEG_ND_2021",
    "WLTH_ODEB_LEG_NDF_2011",
    "WLTH_ODEB_LEG_NDF_2013",
    "WLTH_ODEB_LEG_NDF_2015",
    "WLTH_ODEB_LEG_NDF_2017",
    "WLTH_ODEB_LEG_NDF_2019",
    "WLTH_ODEB_LEG_NDF_2021",
    "WLTH_ODEB_LEG_RD_2011",
    "WLTH_ODEB_LEG_RD_2013",
    "WLTH_ODEB_LEG_RD_2015",
    "WLTH_ODEB_LEG_RD_2017",
    "WLTH_ODEB_LEG_RD_2019",
    "WLTH_ODEB_LEG_RD_2021",
    "WLTH_ODEB_LEG_RDF_2011",
    "WLTH_ODEB_LEG_RDF_2013",
    "WLTH_ODEB_LEG_RDF_2015",
    "WLTH_ODEB_LEG_RDF_2017",
    "WLTH_ODEB_LEG_RDF_2019",
    "WLTH_ODEB_LEG_RDF_2021",
    "WLTH_ODEB_MED_ND_2011",
    "WLTH_ODEB_MED_ND_2013",
    "WLTH_ODEB_MED_ND_2015",
    "WLTH_ODEB_MED_ND_2017",
    "WLTH_ODEB_MED_ND_2019",
    "WLTH_ODEB_MED_ND_2021",
    "WLTH_ODEB_MED_NDF_2011",
    "WLTH_ODEB_MED_NDF_2013",
    "WLTH_ODEB_MED_NDF_2015",
    "WLTH_ODEB_MED_NDF_2017",
    "WLTH_ODEB_MED_NDF_2019",
    "WLTH_ODEB_MED_NDF_2021",
    "WLTH_ODEB_MED_RD_2011",
    "WLTH_ODEB_MED_RD_2013",
    "WLTH_ODEB_MED_RD_2015",
    "WLTH_ODEB_MED_RD_2017",
    "WLTH_ODEB_MED_RD_2019",
    "WLTH_ODEB_MED_RD_2021",
    "WLTH_ODEB_MED_RDF_2011",
    "WLTH_ODEB_MED_RDF_2013",
    "WLTH_ODEB_MED_RDF_2015",
    "WLTH_ODEB_MED_RDF_2017",
    "WLTH_ODEB_MED_RDF_2019",
    "WLTH_ODEB_MED_RDF_2021",
    "WLTH_ODEB_STU_ND_2011",
    "WLTH_ODEB_STU_ND_2013",
    "WLTH_ODEB_STU_ND_2015",
    "WLTH_ODEB_STU_ND_2017",
    "WLTH_ODEB_STU_ND_2019",
    "WLTH_ODEB_STU_ND_2021",
    "WLTH_ODEB_STU_NDF_2011",
    "WLTH_ODEB_STU_NDF_2013",
    "WLTH_ODEB_STU_NDF_2015",
    "WLTH_ODEB_STU_NDF_2017",
    "WLTH_ODEB_STU_NDF_2019",
    "WLTH_ODEB_STU_NDF_2021",
    "WLTH_ODEB_STU_RD_2011",
    "WLTH_ODEB_STU_RD_2013",
    "WLTH_ODEB_STU_RD_2015",
    "WLTH_ODEB_STU_RD_2017",
    "WLTH_ODEB_STU_RD_2019",
    "WLTH_ODEB_STU_RD_2021",
    "WLTH_ODEB_STU_RDF_2011",
    "WLTH_ODEB_STU_RDF_2013",
    "WLTH_ODEB_STU_RDF_2015",
    "WLTH_ODEB_STU_RDF_2017",
    "WLTH_ODEB_STU_RDF_2019",
    "WLTH_ODEB_STU_RDF_2021",
    "WLTH_ODEB_REM_ND_2013",
    "WLTH_ODEB_REM_ND_2015",
    "WLTH_ODEB_REM_ND_2017",
    "WLTH_ODEB_REM_ND_2019",
    "WLTH_ODEB_REM_ND_2021",
    "WLTH_ODEB_REM_NDF_2013",
    "WLTH_ODEB_REM_NDF_2015",
    "WLTH_ODEB_REM_NDF_2017",
    "WLTH_ODEB_REM_NDF_2019",
    "WLTH_ODEB_REM_NDF_2021",
    "WLTH_ODEB_REM_RD_2013",
    "WLTH_ODEB_REM_RD_2015",
    "WLTH_ODEB_REM_RD_2017",
    "WLTH_ODEB_REM_RD_2019",
    "WLTH_ODEB_REM_RD_2021",
    "WLTH_ODEB_REM_RDF_2013",
    "WLTH_ODEB_REM_RDF_2015",
    "WLTH_ODEB_REM_RDF_2017",
    "WLTH_ODEB_REM_RDF_2019",
    "WLTH_ODEB_REM_RDF_2021"
  ],
  "wealth_home": [
    "ID",
    "LINEAGE",
    "PNUM",
    "WLTH_HOME_NET_ND_1984",
    "WLTH_HOME_NET_ND_1989",
    "WLTH_HOME_NET_ND_1994",
    "WLTH_HOME_NET_ND_1999",
    "WLTH_HOME_NET_ND_2001",
    "WLTH_HOME_NET_ND_2003",
    "WLTH_HOME_NET_ND_2005",
    "WLTH_HOME_NET_ND_2007",
    "WLTH_HOME_NET_ND_2009",
    "WLTH_HOME_NET_ND_2011",
    "WLTH_HOME_NET_ND_2013",
    "WLTH_HOME_NET_ND_2015",
    "WLTH_HOME_NET_ND_2017",
    "WLTH_HOME_NET_ND_2019",
    "WLTH_HOME_NET_ND_2021",
    "WLTH_HOME_NET_NDF_1984",
    "WLTH_HOME_NET_NDF_1989",
    "WLTH_HOME_NET_NDF_1994",
    "WLTH_HOME_NET_NDF_1999",
    "WLTH_HOME_NET_NDF_2001",
    "WLTH_HOME_NET_NDF_2003",
    "WLTH_HOME_NET_NDF_2005",
    "WLTH_HOME_NET_NDF_2007",
    "WLTH_HOME_NET_NDF_2009",
    "WLTH_HOME_NET_NDF_2011",
    "WLTH_HOME_NET_NDF_2013",
    "WLTH_HOME_NET_NDF_2015",
    "WLTH_HOME_NET_NDF_2017",
    "WLTH_HOME_NET_NDF_2019",
    "WLTH_HOME_NET_NDF_2021",
    "WLTH_HOME_NET_RD_1984",
    "WLTH_HOME_NET_RD_1989",
    "WLTH_HOME_NET_RD_1994",
    "WLTH_HOME_NET_RD_1999",
    "WLTH_HOME_NET_RD_2001",
    "WLTH_HOME_NET_RD_2003",
    "WLTH_HOME_NET_RD_2005",
    "WLTH_HOME_NET_RD_2007",
    "WLTH_HOME_NET_RD_2009",
    "WLTH_HOME_NET_RD_2011",
    "WLTH_HOME_NET_RD_2013",
    "WLTH_HOME_NET_RD_2015",
    "WLTH_HOME_NET_RD_2017",
    "WLTH_HOME_NET_RD_2019",
    "WLTH_HOME_NET_RD_2021",
    "WLTH_HOME_NET_RDF_1984",
    "WLTH_HOME_NET_RDF_1989",
    "WLTH_HOME_NET_RDF_1994",
    "WLTH_HOME_NET_RDF_1999",
    "WLTH_HOME_NET_RDF_2001",
    "WLTH_HOME_NET_RDF_2003",
    "WLTH_HOME_NET_RDF_2005",
    "WLTH_HOME_NET_RDF_2007",
    "WLTH_HOME_NET_RDF_2009",
    "WLTH_HOME_NET_RDF_2011",
    "WLTH_HOME_NET_RDF_2013",
    "WLTH_HOME_NET_RDF_2015",
    "WLTH_HOME_NET_RDF_2017",
    "WLTH_HOME_NET_RDF_2019",
    "WLTH_HOME_NET_RDF_2021",
    "WLTH_HOME_ASS_ND_1984",
    "WLTH_HOME_ASS_ND_1989",
    "WLTH_HOME_ASS_ND_1994",
    "WLTH_HOME_ASS_ND_1999",
    "WLTH_HOME_ASS_ND_2001",
    "WLTH_HOME_ASS_ND_2003",
    "WLTH_HOME_ASS_ND_2005",
    "WLTH_HOME_ASS_ND_2007",
    "WLTH_HOME_ASS_ND_2009",
    "WLTH_HOME_ASS_ND_2011",
    "WLTH_HOME_ASS_ND_2013",
    "WLTH_HOME_ASS_ND_2015",
    "WLTH_HOME_ASS_ND_2017",
    "WLTH_HOME_ASS_ND_2019",
    "WLTH_HOME_ASS_ND_2021",
    "WLTH_HOME_ASS_NDF_1984",
    "WLTH_HOME_ASS_NDF_1989",
    "WLTH_HOME_ASS_NDF_1994",
    "WLTH_HOME_ASS_NDF_1999",
    "WLTH_HOME_ASS_NDF_2001",
    "WLTH_HOME_ASS_NDF_2003",
    "WLTH_HOME_ASS_NDF_2005",
    "WLTH_HOME_ASS_NDF_2007",
    "WLTH_HOME_ASS_NDF_2009",
    "WLTH_HOME_ASS_NDF_2011",
    "WLTH_HOME_ASS_NDF_2013",
    "WLTH_HOME_ASS_NDF_2015",
    "WLTH_HOME_ASS_NDF_2017",
    "WLTH_HOME_ASS_NDF_2019",
    "WLTH_HOME_ASS_NDF_2021",
    "WLTH_HOME_ASS_RD_1984",
    "WLTH_HOME_ASS_RD_1989",
    "WLTH_HOME_ASS_RD_1994",
    "WLTH_HOME_ASS_RD_1999",
    "WLTH_HOME_ASS_RD_2001",
    "WLTH_HOME_ASS_RD_2003",
    "WLTH_HOME_ASS_RD_2005",
    "WLTH_HOME_ASS_RD_2007",
    "WLTH_HOME_ASS_RD_2009",
    "WLTH_HOME_ASS_RD_2011",
    "WLTH_HOME_ASS_RD_2013",
    "WLTH_HOME_ASS_RD_2015",
    "WLTH_HOME_ASS_RD_2017",
    "WLTH_HOME_ASS_RD_2019",
    "WLTH_HOME_ASS_RD_2021",
    "WLTH_HOME_ASS_RDF_1984",
    "WLTH_HOME_ASS_RDF_1989",
    "WLTH_HOME_ASS_RDF_1994",
    "WLTH_HOME_ASS_RDF_1999",
    "WLTH_HOME_ASS_RDF_2001",
    "WLTH_HOME_ASS_RDF_2003",
    "WLTH_HOME_ASS_RDF_2005",
    "WLTH_HOME_ASS_RDF_2007",
    "WLTH_HOME_ASS_RDF_2009",
    "WLTH_HOME_ASS_RDF_2011",
    "WLTH_HOME_ASS_RDF_2013",
    "WLTH_HOME_ASS_RDF_2015",
    "WLTH_HOME_ASS_RDF_2017",
    "WLTH_HOME_ASS_RDF_2019",
    "WLTH_HOME_ASS_RDF_2021",
    "WLTH_HOME_DEB_ND_1984",
    "WLTH_HOME_DEB_ND_1989",
    "WLTH_HOME_DEB_ND_1994",
    "WLTH_HOME_DEB_ND_1999",
    "WLTH_HOME_DEB_ND_2001",
    "WLTH_HOME_DEB_ND_2003",
    "WLTH_HOME_DEB_ND_2005",
    "WLTH_HOME_DEB_ND_2007",
    "WLTH_HOME_DEB_ND_2009",
    "WLTH_HOME_DEB_ND_2011",
    "WLTH_HOME_DEB_ND_2013",
    "WLTH_HOME_DEB_ND_2015",
    "WLTH_HOME_DEB_ND_2017",
    "WLTH_HOME_DEB_ND_2019",
    "WLTH_HOME_DEB_ND_2021",
    "WLTH_HOME_DEB_NDF_1984",
    "WLTH_HOME_DEB_NDF_1989",
    "WLTH_HOME_DEB_NDF_1994",
    "WLTH_HOME_DEB_NDF_1999",
    "WLTH_HOME_DEB_NDF_2001",
    "WLTH_HOME_DEB_NDF_2003",
    "WLTH_HOME_DEB_NDF_2005",
    "WLTH_HOME_DEB_NDF_2007",
    "WLTH_HOME_DEB_NDF_2009",
    "WLTH_HOME_DEB_NDF_2011",
    "WLTH_HOME_DEB_NDF_2013",
    "WLTH_HOME_DEB_NDF_2015",
    "WLTH_HOME_DEB_NDF_2017",
    "WLTH_HOME_DEB_NDF_2019",
    "WLTH_HOME_DEB_NDF_2021",
    "WLTH_HOME_DEB_RD_1984",
    "WLTH_HOME_DEB_RD_1989",
    "WLTH_HOME_DEB_RD_1994",
    "WLTH_HOME_DEB_RD_1999",
    "WLTH_HOME_DEB_RD_2001",
    "WLTH_HOME_DEB_RD_2003",
    "WLTH_HOME_DEB_RD_2005",
    "WLTH_HOME_DEB_RD_2007",
    "WLTH_HOME_DEB_RD_2009",
    "WLTH_HOME_DEB_RD_2011",
    "WLTH_HOME_DEB_RD_2013",
    "WLTH_HOME_DEB_RD_2015",
    "WLTH_HOME_DEB_RD_2017",
    "WLTH_HOME_DEB_RD_2019",
    "WLTH_HOME_DEB_RD_2021",
    "WLTH_HOME_DEB_RDF_1984",
    "WLTH_HOME_DEB_RDF_1989",
    "WLTH_HOME_DEB_RDF_1994",
    "WLTH_HOME_DEB_RDF_1999",
    "WLTH_HOME_DEB_RDF_2001",
    "WLTH_HOME_DEB_RDF_2003",
    "WLTH_HOME_DEB_RDF_2005",
    "WLTH_HOME_DEB_RDF_2007",
    "WLTH_HOME_DEB_RDF_2009",
    "WLTH_HOME_DEB_RDF_2011",
    "WLTH_HOME_DEB_RDF_2013",
    "WLTH_HOME_DEB_RDF_2015",
    "WLTH_HOME_DEB_RDF_2017",
    "WLTH_HOME_DEB_RDF_2019",
    "WLTH_HOME_DEB_RDF_2021"
  ],
  "wealth_investments": [
    "ID",
    "LINEAGE",
    "PNUM",
    "WLTH_INVE_NET_ND_1984",
    "WLTH_INVE_NET_ND_1989",
    "WLTH_INVE_NET_ND_1994",
    "WLTH_INVE_NET_ND_1999",
    "WLTH_INVE_NET_ND_2001",
    "WLTH_INVE_NET_ND_2003",
    "WLTH_INVE_NET_ND_2005",
    "WLTH_INVE_NET_ND_2007",
    "WLTH_INVE_NET_ND_2009",
    "WLTH_INVE_NET_ND_2011",
    "WLTH_INVE_NET_ND_2013",
    "WLTH_INVE_NET_ND_2015",
    "WLTH_INVE_NET_ND_2017",
    "WLTH_INVE_NET_ND_2019",
    "WLTH_INVE_NET_ND_2021",
    "WLTH_INVE_NET_NDF_1984",
    "WLTH_INVE_NET_NDF_1989",
    "WLTH_INVE_NET_NDF_1994",
    "WLTH_INVE_NET_NDF_1999",
    "WLTH_INVE_NET_NDF_2001",
    "WLTH_INVE_NET_NDF_2003",
    "WLTH_INVE_NET_NDF_2005",
    "WLTH_INVE_NET_NDF_2007",
    "WLTH_INVE_NET_NDF_2009",
    "WLTH_INVE_NET_NDF_2011",
    "WLTH_INVE_NET_NDF_2013",
    "WLTH_INVE_NET_NDF_2015",
    "WLTH_INVE_NET_NDF_2017",
    "WLTH_INVE_NET_NDF_2019",
    "WLTH_INVE_NET_NDF_2021",
    "WLTH_INVE_NET_RD_1984",
    "WLTH_INVE_NET_RD_1989",
    "WLTH_INVE_NET_RD_1994",
    "WLTH_INVE_NET_RD_1999",
    "WLTH_INVE_NET_RD_2001",
    "WLTH_INVE_NET_RD_2003",
    "WLTH_INVE_NET_RD_2005",
    "WLTH_INVE_NET_RD_2007",
    "WLTH_INVE_NET_RD_2009",
    "WLTH_INVE_NET_RD_2011",
    "WLTH_INVE_NET_RD_2013",
    "WLTH_INVE_NET_RD_2015",
    "WLTH_INVE_NET_RD_2017",
    "WLTH_INVE_NET_RD_2019",
    "WLTH_INVE_NET_RD_2021",
    "WLTH_INVE_NET_RDF_1984",
    "WLTH_INVE_NET_RDF_1989",
    "WLTH_INVE_NET_RDF_1994",
    "WLTH_INVE_NET_RDF_1999",
    "WLTH_INVE_NET_RDF_2001",
    "WLTH_INVE_NET_RDF_2003",
    "WLTH_INVE_NET_RDF_2005",
    "WLTH_INVE_NET_RDF_2007",
    "WLTH_INVE_NET_RDF_2009",
    "WLTH_INVE_NET_RDF_2011",
    "WLTH_INVE_NET_RDF_2013",
    "WLTH_INVE_NET_RDF_2015",
    "WLTH_INVE_NET_RDF_2017",
    "WLTH_INVE_NET_RDF_2019",
    "WLTH_INVE_NET_RDF_2021",
    "WLTH_INVE_IRA_ND_1999",
    "WLTH_INVE_IRA_ND_2001",
    "WLTH_INVE_IRA_ND_2003",
    "WLTH_INVE_IRA_ND_2005",
    "WLTH_INVE_IRA_ND_2007",
    "WLTH_INVE_IRA_ND_2009",
    "WLTH_INVE_IRA_ND_2011",
    "WLTH_INVE_IRA_ND_2013",
    "WLTH_INVE_IRA_ND_2015",
    "WLTH_INVE_IRA_ND_2017",
    "WLTH_INVE_IRA_ND_2019",
    "WLTH_INVE_IRA_ND_2021",
    "WLTH_INVE_IRA_NDF_1999",
    "WLTH_INVE_IRA_NDF_2001",
    "WLTH_INVE_IRA_NDF_2003",
    "WLTH_INVE_IRA_NDF_2005",
    "WLTH_INVE_IRA_NDF_2007",
    "WLTH_INVE_IRA_NDF_2009",
    "WLTH_INVE_IRA_NDF_2011",
    "WLTH_INVE_IRA_NDF_2013",
    "WLTH_INVE_IRA_NDF_2015",
    "WLTH_INVE_IRA_NDF_2017",
    "WLTH_INVE_IRA_NDF_2019",
    "WLTH_INVE_IRA_NDF_2021",
    "WLTH_INVE_IRA_RD_1999",
    "WLTH_INVE_IRA_RD_2001",
    "WLTH_INVE_IRA_RD_2003",
    "WLTH_INVE_IRA_RD_2005",
    "WLTH_INVE_IRA_RD_2007",
    "WLTH_INVE_IRA_RD_2009",
    "WLTH_INVE_IRA_RD_2011",
    "WLTH_INVE_IRA_RD_2013",
    "WLTH_INVE_IRA_RD_2015",
    "WLTH_INVE_IRA_RD_2017",
    "WLTH_INVE_IRA_RD_2019",
    "WLTH_INVE_IRA_RD_2021",
    "WLTH_INVE_IRA_RDF_1999",
    "WLTH_INVE_IRA_RDF_2001",
    "WLTH_INVE_IRA_RDF_2003",
    "WLTH_INVE_IRA_RDF_2005",
    "WLTH_INVE_IRA_RDF_2007",
    "WLTH_INVE_IRA_RDF_2009",
    "WLTH_INVE_IRA_RDF_2011",
    "WLTH_INVE_IRA_RDF_2013",
    "WLTH_INVE_IRA_RDF_2015",
    "WLTH_INVE_IRA_RDF_2017",
    "WLTH_INVE_IRA_RDF_2019",
    "WLTH_INVE_IRA_RDF_2021",
    "WLTH_INVE_STK_ND_1999",
    "WLTH_INVE_STK_ND_2001",
    "WLTH_INVE_STK_ND_2003",
    "WLTH_INVE_STK_ND_2005",
    "WLTH_INVE_STK_ND_2007",
    "WLTH_INVE_STK_ND_2009",
    "WLTH_INVE_STK_ND_2011",
    "WLTH_INVE_STK_ND_2013",
    "WLTH_INVE_STK_ND_2015",
    "WLTH_INVE_STK_ND_2017",
    "WLTH_INVE_STK_ND_2019",
    "WLTH_INVE_STK_ND_2021",
    "WLTH_INVE_STK_NDF_1999",
    "WLTH_INVE_STK_NDF_2001",
    "WLTH_INVE_STK_NDF_2003",
    "WLTH_INVE_STK_NDF_2005",
    "WLTH_INVE_STK_NDF_2007",
    "WLTH_INVE_STK_NDF_2009",
    "WLTH_INVE_STK_NDF_2011",
    "WLTH_INVE_STK_NDF_2013",
    "WLTH_INVE_STK_NDF_2015",
    "WLTH_INVE_STK_NDF_2017",
    "WLTH_INVE_STK_NDF_2019",
    "WLTH_INVE_STK_NDF_2021",
    "WLTH_INVE_STK_RD_1999",
    "WLTH_INVE_STK_RD_2001",
    "WLTH_INVE_STK_RD_2003",
    "WLTH_INVE_STK_RD_2005",
    "WLTH_INVE_STK_RD_2007",
    "WLTH_INVE_STK_RD_2009",
    "WLTH_INVE_STK_RD_2011",
    "WLTH_INVE_STK_RD_2013",
    "WLTH_INVE_STK_RD_2015",
    "WLTH_INVE_STK_RD_2017",
    "WLTH_INVE_STK_RD_2019",
    "WLTH_INVE_STK_RD_2021",
    "WLTH_INVE_STK_RDF_1999",
    "WLTH_INVE_STK_RDF_2001",
    "WLTH_INVE_STK_RDF_2003",
    "WLTH_INVE_STK_RDF_2005",
    "WLTH_INVE_STK_RDF_2007",
    "WLTH_INVE_STK_RDF_2009",
    "WLTH_INVE_STK_RDF_2011",
    "WLTH_INVE_STK_RDF_2013",
    "WLTH_INVE_STK_RDF_2015",
    "WLTH_INVE_STK_RDF_2017",
    "WLTH_INVE_STK_RDF_2019",
    "WLTH_INVE_STK_RDF_2021"
  ],
  "wealth_other_assets": [
    "ID",
    "LINEAGE",
    "PNUM",
    "WLTH_OASS_NET_ND_1984",
    "WLTH_OASS_NET_ND_1989",
    "WLTH_OASS_NET_ND_1994",
    "WLTH_OASS_NET_ND_1999",
    "WLTH_OASS_NET_ND_2001",
    "WLTH_OASS_NET_ND_2003",
    "WLTH_OASS_NET_ND_2005",
    "WLTH_OASS_NET_ND_2007",
    "WLTH_OASS_NET_ND_2009",
    "WLTH_OASS_NET_ND_2011",
    "WLTH_OASS_NET_ND_2013",
    "WLTH_OASS_NET_ND_2015",
    "WLTH_OASS_NET_ND_2017",
    "WLTH_OASS_NET_ND_2019",
    "WLTH_OASS_NET_ND_2021",
    "WLTH_OASS_NET_NDF_1984",
    "WLTH_OASS_NET_NDF_1989",
    "WLTH_OASS_NET_NDF_1994",
    "WLTH_OASS_NET_NDF_1999",
    "WLTH_OASS_NET_NDF_2001",
    "WLTH_OASS_NET_NDF_2003",
    "WLTH_OASS_NET_NDF_2005",
    "WLTH_OASS_NET_NDF_2007",
    "WLTH_OASS_NET_NDF_2009",
    "WLTH_OASS_NET_NDF_2011",
    "WLTH_OASS_NET_NDF_2013",
    "WLTH_OASS_NET_NDF_2015",
    "WLTH_OASS_NET_NDF_2017",
    "WLTH_OASS_NET_NDF_2019",
    "WLTH_OASS_NET_NDF_2021",
    "WLTH_OASS_NET_RD_1984",
    "WLTH_OASS_NET_RD_1989",
    "WLTH_OASS_NET_RD_1994",
    "WLTH_OASS_NET_RD_1999",
    "WLTH_OASS_NET_RD_2001",
    "WLTH_OASS_NET_RD_2003",
    "WLTH_OASS_NET_RD_2005",
    "WLTH_OASS_NET_RD_2007",
    "WLTH_OASS_NET_RD_2009",
    "WLTH_OASS_NET_RD_2011",
    "WLTH_OASS_NET_RD_2013",
    "WLTH_OASS_NET_RD_2015",
    "WLTH_OASS_NET_RD_2017",
    "WLTH_OASS_NET_RD_2019",
    "WLTH_OASS_NET_RD_2021",
    "WLTH_OASS_NET_RDF_1984",
    "WLTH_OASS_NET_RDF_1989",
    "WLTH_OASS_NET_RDF_1994",
    "WLTH_OASS_NET_RDF_1999",
    "WLTH_OASS_NET_RDF_2001",
    "WLTH_OASS_NET_RDF_2003",
    "WLTH_OASS_NET_RDF_2005",
    "WLTH_OASS_NET_RDF_2007",
    "WLTH_OASS_NET_RDF_2009",
    "WLTH_OASS_NET_RDF_2011",
    "WLTH_OASS_NET_RDF_2013",
    "WLTH_OASS_NET_RDF_2015",
    "WLTH_OASS_NET_RDF_2017",
    "WLTH_OASS_NET_RDF_2019",
    "WLTH_OASS_NET_RDF_2021"
  ],
  "wealth_real_estate": [
    "ID",
    "LINEAGE",
    "PNUM",
    "WLTH_REAL_NET_ND_1984",
    "WLTH_REAL_NET_ND_1989",
    "WLTH_REAL_NET_ND_1994",
    "WLTH_REAL_NET_ND_1999",
    "WLTH_REAL_NET_ND_2001",
    "WLTH_REAL_NET_ND_2003",
    "WLTH_REAL_NET_ND_2005",
    "WLTH_REAL_NET_ND_2007",
    "WLTH_REAL_NET_ND_2009",
    "WLTH_REAL_NET_ND_2011",
    "WLTH_REAL_NET_ND_2013",
    "WLTH_REAL_NET_ND_2015",
    "WLTH_REAL_NET_ND_2017",
    "WLTH_REAL_NET_ND_2019",
    "WLTH_REAL_NET_ND_2021",
    "WLTH_REAL_NET_NDF_1984",
    "WLTH_REAL_NET_NDF_1989",
    "WLTH_REAL_NET_NDF_1994",
    "WLTH_REAL_NET_NDF_1999",
    "WLTH_REAL_NET_NDF_2001",
    "WLTH_REAL_NET_NDF_2003",
    "WLTH_REAL_NET_NDF_2005",
    "WLTH_REAL_NET_NDF_2007",
    "WLTH_REAL_NET_NDF_2009",
    "WLTH_REAL_NET_NDF_2011",
    "WLTH_REAL_NET_NDF_2013",
    "WLTH_REAL_NET_NDF_2015",
    "WLTH_REAL_NET_NDF_2017",
    "WLTH_REAL_NET_NDF_2019",
    "WLTH_REAL_NET_NDF_2021",
    "WLTH_REAL_NET_RD_1984",
    "WLTH_REAL_NET_RD_1989",
    "WLTH_REAL_NET_RD_1994",
    "WLTH_REAL_NET_RD_1999",
    "WLTH_REAL_NET_RD_2001",
    "WLTH_REAL_NET_RD_2003",
    "WLTH_REAL_NET_RD_2005",
    "WLTH_REAL_NET_RD_2007",
    "WLTH_REAL_NET_RD_2009",
    "WLTH_REAL_NET_RD_2011",
    "WLTH_REAL_NET_RD_2013",
    "WLTH_REAL_NET_RD_2015",
    "WLTH_REAL_NET_RD_2017",
    "WLTH_REAL_NET_RD_2019",
    "WLTH_REAL_NET_RD_2021",
    "WLTH_REAL_NET_RDF_1984",
    "WLTH_REAL_NET_RDF_1989",
    "WLTH_REAL_NET_RDF_1994",
    "WLTH_REAL_NET_RDF_1999",
    "WLTH_REAL_NET_RDF_2001",
    "WLTH_REAL_NET_RDF_2003",
    "WLTH_REAL_NET_RDF_2005",
    "WLTH_REAL_NET_RDF_2007",
    "WLTH_REAL_NET_RDF_2009",
    "WLTH_REAL_NET_RDF_2011",
    "WLTH_REAL_NET_RDF_2013",
    "WLTH_REAL_NET_RDF_2015",
    "WLTH_REAL_NET_RDF_2017",
    "WLTH_REAL_NET_RDF_2019",
    "WLTH_REAL_NET_RDF_2021",
    "WLTH_REAL_ASS_ND_2013",
    "WLTH_REAL_ASS_ND_2015",
    "WLTH_REAL_ASS_ND_2017",
    "WLTH_REAL_ASS_ND_2019",
    "WLTH_REAL_ASS_ND_2021",
    "WLTH_REAL_ASS_NDF_2013",
    "WLTH_REAL_ASS_NDF_2015",
    "WLTH_REAL_ASS_NDF_2017",
    "WLTH_REAL_ASS_NDF_2019",
    "WLTH_REAL_ASS_NDF_2021",
    "WLTH_REAL_ASS_RD_2013",
    "WLTH_REAL_ASS_RD_2015",
    "WLTH_REAL_ASS_RD_2017",
    "WLTH_REAL_ASS_RD_2019",
    "WLTH_REAL_ASS_RD_2021",
    "WLTH_REAL_ASS_RDF_2013",
    "WLTH_REAL_ASS_RDF_2015",
    "WLTH_REAL_ASS_RDF_2017",
    "WLTH_REAL_ASS_RDF_2019",
    "WLTH_REAL_ASS_RDF_2021",
    "WLTH_REAL_DEB_ND_2013",
    "WLTH_REAL_DEB_ND_2015",
    "WLTH_REAL_DEB_ND_2017",
    "WLTH_REAL_DEB_ND_2019",
    "WLTH_REAL_DEB_ND_2021",
    "WLTH_REAL_DEB_NDF_2013",
    "WLTH_REAL_DEB_NDF_2015",
    "WLTH_REAL_DEB_NDF_2017",
    "WLTH_REAL_DEB_NDF_2019",
    "WLTH_REAL_DEB_NDF_2021",
    "WLTH_REAL_DEB_RD_2013",
    "WLTH_REAL_DEB_RD_2015",
    "WLTH_REAL_DEB_RD_2017",
    "WLTH_REAL_DEB_RD_2019",
    "WLTH_REAL_DEB_RD_2021",
    "WLTH_REAL_DEB_RDF_2013",
    "WLTH_REAL_DEB_RDF_2015",
    "WLTH_REAL_DEB_RDF_2017",
    "WLTH_REAL_DEB_RDF_2019",
    "WLTH_REAL_DEB_RDF_2021"
  ],
  "wealth_savings": [
    "ID",
    "LINEAGE",
    "PNUM",
    "WLTH_SAVI_NET_ND_1984",
    "WLTH_SAVI_NET_ND_1989",
    "WLTH_SAVI_NET_ND_1994",
    "WLTH_SAVI_NET_ND_1999",
    "WLTH_SAVI_NET_ND_2001",
    "WLTH_SAVI_NET_ND_2003",
    "WLTH_SAVI_NET_ND_2005",
    "WLTH_SAVI_NET_ND_2007",
    "WLTH_SAVI_NET_ND_2009",
    "WLTH_SAVI_NET_ND_2011",
    "WLTH_SAVI_NET_ND_2013",
    "WLTH_SAVI_NET_ND_2015",
    "WLTH_SAVI_NET_ND_2017",
    "WLTH_SAVI_NET_ND_2019",
    "WLTH_SAVI_NET_ND_2021",
    "WLTH_SAVI_NET_NDF_1984",
    "WLTH_SAVI_NET_NDF_1989",
    "WLTH_SAVI_NET_NDF_1994",
    "WLTH_SAVI_NET_NDF_1999",
    "WLTH_SAVI_NET_NDF_2001",
    "WLTH_SAVI_NET_NDF_2003",
    "WLTH_SAVI_NET_NDF_2005",
    "WLTH_SAVI_NET_NDF_2007",
    "WLTH_SAVI_NET_NDF_2009",
    "WLTH_SAVI_NET_NDF_2011",
    "WLTH_SAVI_NET_NDF_2013",
    "WLTH_SAVI_NET_NDF_2015",
    "WLTH_SAVI_NET_NDF_2017",
    "WLTH_SAVI_NET_NDF_2019",
    "WLTH_SAVI_NET_NDF_2021",
    "WLTH_SAVI_NET_RD_1984",
    "WLTH_SAVI_NET_RD_1989",
    "WLTH_SAVI_NET_RD_1994",
    "WLTH_SAVI_NET_RD_1999",
    "WLTH_SAVI_NET_RD_2001",
    "WLTH_SAVI_NET_RD_2003",
    "WLTH_SAVI_NET_RD_2005",
    "WLTH_SAVI_NET_RD_2007",
    "WLTH_SAVI_NET_RD_2009",
    "WLTH_SAVI_NET_RD_2011",
    "WLTH_SAVI_NET_RD_2013",
    "WLTH_SAVI_NET_RD_2015",
    "WLTH_SAVI_NET_RD_2017",
    "WLTH_SAVI_NET_RD_2019",
    "WLTH_SAVI_NET_RD_2021",
    "WLTH_SAVI_NET_RDF_1984",
    "WLTH_SAVI_NET_RDF_1989",
    "WLTH_SAVI_NET_RDF_1994",
    "WLTH_SAVI_NET_RDF_1999",
    "WLTH_SAVI_NET_RDF_2001",
    "WLTH_SAVI_NET_RDF_2003",
    "WLTH_SAVI_NET_RDF_2005",
    "WLTH_SAVI_NET_RDF_2007",
    "WLTH_SAVI_NET_RDF_2009",
    "WLTH_SAVI_NET_RDF_2011",
    "WLTH_SAVI_NET_RDF_2013",
    "WLTH_SAVI_NET_RDF_2015",
    "WLTH_SAVI_NET_RDF_2017",
    "WLTH_SAVI_NET_RDF_2019",
    "WLTH_SAVI_NET_RDF_2021",
    "WLTH_SAVI_BNK_ND_2019",
    "WLTH_SAVI_BNK_ND_2021",
    "WLTH_SAVI_BNK_NDF_2019",
    "WLTH_SAVI_BNK_NDF_2021",
    "WLTH_SAVI_BNK_RD_2019",
    "WLTH_SAVI_BNK_RD_2021",
    "WLTH_SAVI_BNK_RDF_2019",
    "WLTH_SAVI_BNK_RDF_2021",
    "WLTH_SAVI_BND_ND_2019",
    "WLTH_SAVI_BND_ND_2021",
    "WLTH_SAVI_BND_NDF_2019",
    "WLTH_SAVI_BND_NDF_2021",
    "WLTH_SAVI_BND_RD_2019",
    "WLTH_SAVI_BND_RD_2021",
    "WLTH_SAVI_BND_RDF_2019",
    "WLTH_SAVI_BND_RDF_2021"
  ],
  "wealth_totals_assets": [
    "ID",
    "LINEAGE",
    "PNUM",
    "WLTH_TOT_ASS_ND_1984",
    "WLTH_TOT_ASS_ND_1989",
    "WLTH_TOT_ASS_ND_1994",
    "WLTH_TOT_ASS_ND_1999",
    "WLTH_TOT_ASS_ND_2001",
    "WLTH_TOT_ASS_ND_2003",
    "WLTH_TOT_ASS_ND_2005",
    "WLTH_TOT_ASS_ND_2007",
    "WLTH_TOT_ASS_ND_2009",
    "WLTH_TOT_ASS_ND_2011",
    "WLTH_TOT_ASS_ND_2013",
    "WLTH_TOT_ASS_ND_2015",
    "WLTH_TOT_ASS_ND_2017",
    "WLTH_TOT_ASS_ND_2019",
    "WLTH_TOT_ASS_ND_2021",
    "WLTH_TOT_ASS_NDF_1984",
    "WLTH_TOT_ASS_NDF_1989",
    "WLTH_TOT_ASS_NDF_1994",
    "WLTH_TOT_ASS_NDF_1999",
    "WLTH_TOT_ASS_NDF_2001",
    "WLTH_TOT_ASS_NDF_2003",
    "WLTH_TOT_ASS_NDF_2005",
    "WLTH_TOT_ASS_NDF_2007",
    "WLTH_TOT_ASS_NDF_2009",
    "WLTH_TOT_ASS_NDF_2011",
    "WLTH_TOT_ASS_NDF_2013",
    "WLTH_TOT_ASS_NDF_2015",
    "WLTH_TOT_ASS_NDF_2017",
    "WLTH_TOT_ASS_NDF_2019",
    "WLTH_TOT_ASS_NDF_2021",
    "WLTH_TOT_ASS_RD_1984",
    "WLTH_TOT_ASS_RD_1989",
    "WLTH_TOT_ASS_RD_1994",
    "WLTH_TOT_ASS_RD_1999",
    "WLTH_TOT_ASS_RD_2001",
    "WLTH_TOT_ASS_RD_2003",
    "WLTH_TOT_ASS_RD_2005",
    "WLTH_TOT_ASS_RD_2007",
    "WLTH_TOT_ASS_RD_2009",
    "WLTH_TOT_ASS_RD_2011",
    "WLTH_TOT_ASS_RD_2013",
    "WLTH_TOT_ASS_RD_2015",
    "WLTH_TOT_ASS_RD_2017",
    "WLTH_TOT_ASS_RD_2019",
    "WLTH_TOT_ASS_RD_2021",
    "WLTH_TOT_ASS_RDF_1984",
    "WLTH_TOT_ASS_RDF_1989",
    "WLTH_TOT_ASS_RDF_1994",
    "WLTH_TOT_ASS_RDF_1999",
    "WLTH_TOT_ASS_RDF_2001",
    "WLTH_TOT_ASS_RDF_2003",
    "WLTH_TOT_ASS_RDF_2005",
    "WLTH_TOT_ASS_RDF_2007",
    "WLTH_TOT_ASS_RDF_2009",
    "WLTH_TOT_ASS_RDF_2011",
    "WLTH_TOT_ASS_RDF_2013",
    "WLTH_TOT_ASS_RDF_2015",
    "WLTH_TOT_ASS_RDF_2017",
    "WLTH_TOT_ASS_RDF_2019",
    "WLTH_TOT_ASS_RDF_2021",
    "WLTH_TOT_ASS_XH_ND_1984",
    "WLTH_TOT_ASS_XH_ND_1989",
    "WLTH_TOT_ASS_XH_ND_1994",
    "WLTH_TOT_ASS_XH_ND_1999",
    "WLTH_TOT_ASS_XH_ND_2001",
    "WLTH_TOT_ASS_XH_ND_2003",
    "WLTH_TOT_ASS_XH_ND_2005",
    "WLTH_TOT_ASS_XH_ND_2007",
    "WLTH_TOT_ASS_XH_ND_2009",
    "WLTH_TOT_ASS_XH_ND_2011",
    "WLTH_TOT_ASS_XH_ND_2013",
    "WLTH_TOT_ASS_XH_ND_2015",
    "WLTH_TOT_ASS_XH_ND_2017",
    "WLTH_TOT_ASS_XH_ND_2019",
    "WLTH_TOT_ASS_XH_ND_2021",
    "WLTH_TOT_ASS_XH_NDF_1984",
    "WLTH_TOT_ASS_XH_NDF_1989",
    "WLTH_TOT_ASS_XH_NDF_1994",
    "WLTH_TOT_ASS_XH_NDF_1999",
    "WLTH_TOT_ASS_XH_NDF_2001",
    "WLTH_TOT_ASS_XH_NDF_2003",
    "WLTH_TOT_ASS_XH_NDF_2005",
    "WLTH_TOT_ASS_XH_NDF_2007",
    "WLTH_TOT_ASS_XH_NDF_2009",
    "WLTH_TOT_ASS_XH_NDF_2011",
    "WLTH_TOT_ASS_XH_NDF_2013",
    "WLTH_TOT_ASS_XH_NDF_2015",
    "WLTH_TOT_ASS_XH_NDF_2017",
    "WLTH_TOT_ASS_XH_NDF_2019",
    "WLTH_TOT_ASS_XH_NDF_2021",
    "WLTH_TOT_ASS_XH_RD_1984",
    "WLTH_TOT_ASS_XH_RD_1989",
    "WLTH_TOT_ASS_XH_RD_1994",
    "WLTH_TOT_ASS_XH_RD_1999",
    "WLTH_TOT_ASS_XH_RD_2001",
    "WLTH_TOT_ASS_XH_RD_2003",
    "WLTH_TOT_ASS_XH_RD_2005",
    "WLTH_TOT_ASS_XH_RD_2007",
    "WLTH_TOT_ASS_XH_RD_2009",
    "WLTH_TOT_ASS_XH_RD_2011",
    "WLTH_TOT_ASS_XH_RD_2013",
    "WLTH_TOT_ASS_XH_RD_2015",
    "WLTH_TOT_ASS_XH_RD_2017",
    "WLTH_TOT_ASS_XH_RD_2019",
    "WLTH_TOT_ASS_XH_RD_2021",
    "WLTH_TOT_ASS_XH_RDF_1984",
    "WLTH_TOT_ASS_XH_RDF_1989",
    "WLTH_TOT_ASS_XH_RDF_1994",
    "WLTH_TOT_ASS_XH_RDF_1999",
    "WLTH_TOT_ASS_XH_RDF_2001",
    "WLTH_TOT_ASS_XH_RDF_2003",
    "WLTH_TOT_ASS_XH_RDF_2005",
    "WLTH_TOT_ASS_XH_RDF_2007",
    "WLTH_TOT_ASS_XH_RDF_2009",
    "WLTH_TOT_ASS_XH_RDF_2011",
    "WLTH_TOT_ASS_XH_RDF_2013",
    "WLTH_TOT_ASS_XH_RDF_2015",
    "WLTH_TOT_ASS_XH_RDF_2017",
    "WLTH_TOT_ASS_XH_RDF_2019",
    "WLTH_TOT_ASS_XH_RDF_2021",
    "WLTH_TOT_ASS_XHR_ND_1984",
    "WLTH_TOT_ASS_XHR_ND_1989",
    "WLTH_TOT_ASS_XHR_ND_1994",
    "WLTH_TOT_ASS_XHR_ND_1999",
    "WLTH_TOT_ASS_XHR_ND_2001",
    "WLTH_TOT_ASS_XHR_ND_2003",
    "WLTH_TOT_ASS_XHR_ND_2005",
    "WLTH_TOT_ASS_XHR_ND_2007",
    "WLTH_TOT_ASS_XHR_ND_2009",
    "WLTH_TOT_ASS_XHR_ND_2011",
    "WLTH_TOT_ASS_XHR_ND_2013",
    "WLTH_TOT_ASS_XHR_ND_2015",
    "WLTH_TOT_ASS_XHR_ND_2017",
    "WLTH_TOT_ASS_XHR_ND_2019",
    "WLTH_TOT_ASS_XHR_ND_2021",
    "WLTH_TOT_ASS_XHR_NDF_1984",
    "WLTH_TOT_ASS_XHR_NDF_1989",
    "WLTH_TOT_ASS_XHR_NDF_1994",
    "WLTH_TOT_ASS_XHR_NDF_1999",
    "WLTH_TOT_ASS_XHR_NDF_2001",
    "WLTH_TOT_ASS_XHR_NDF_2003",
    "WLTH_TOT_ASS_XHR_NDF_2005",
    "WLTH_TOT_ASS_XHR_NDF_2007",
    "WLTH_TOT_ASS_XHR_NDF_2009",
    "WLTH_TOT_ASS_XHR_NDF_2011",
    "WLTH_TOT_ASS_XHR_NDF_2013",
    "WLTH_TOT_ASS_XHR_NDF_2015",
    "WLTH_TOT_ASS_XHR_NDF_2017",
    "WLTH_TOT_ASS_XHR_NDF_2019",
    "WLTH_TOT_ASS_XHR_NDF_2021",
    "WLTH_TOT_ASS_XHR_RD_1984",
    "WLTH_TOT_ASS_XHR_RD_1989",
    "WLTH_TOT_ASS_XHR_RD_1994",
    "WLTH_TOT_ASS_XHR_RD_1999",
    "WLTH_TOT_ASS_XHR_RD_2001",
    "WLTH_TOT_ASS_XHR_RD_2003",
    "WLTH_TOT_ASS_XHR_RD_2005",
    "WLTH_TOT_ASS_XHR_RD_2007",
    "WLTH_TOT_ASS_XHR_RD_2009",
    "WLTH_TOT_ASS_XHR_RD_2011",
    "WLTH_TOT_ASS_XHR_RD_2013",
    "WLTH_TOT_ASS_XHR_RD_2015",
    "WLTH_TOT_ASS_XHR_RD_2017",
    "WLTH_TOT_ASS_XHR_RD_2019",
    "WLTH_TOT_ASS_XHR_RD_2021",
    "WLTH_TOT_ASS_XHR_RDF_1984",
    "WLTH_TOT_ASS_XHR_RDF_1989",
    "WLTH_TOT_ASS_XHR_RDF_1994",
    "WLTH_TOT_ASS_XHR_RDF_1999",
    "WLTH_TOT_ASS_XHR_RDF_2001",
    "WLTH_TOT_ASS_XHR_RDF_2003",
    "WLTH_TOT_ASS_XHR_RDF_2005",
    "WLTH_TOT_ASS_XHR_RDF_2007",
    "WLTH_TOT_ASS_XHR_RDF_2009",
    "WLTH_TOT_ASS_XHR_RDF_2011",
    "WLTH_TOT_ASS_XHR_RDF_2013",
    "WLTH_TOT_ASS_XHR_RDF_2015",
    "WLTH_TOT_ASS_XHR_RDF_2017",
    "WLTH_TOT_ASS_XHR_RDF_2019",
    "WLTH_TOT_ASS_XHR_RDF_2021"
  ],
  "wealth_totals_debt": [
    "ID",
    "LINEAGE",
    "PNUM",
    "WLTH_TOT_DEB_ND_1984",
    "WLTH_TOT_DEB_ND_1989",
    "WLTH_TOT_DEB_ND_1994",
    "WLTH_TOT_DEB_ND_1999",
    "WLTH_TOT_DEB_ND_2001",
    "WLTH_TOT_DEB_ND_2003",
    "WLTH_TOT_DEB_ND_2005",
    "WLTH_TOT_DEB_ND_2007",
    "WLTH_TOT_DEB_ND_2009",
    "WLTH_TOT_DEB_ND_2011",
    "WLTH_TOT_DEB_ND_2013",
    "WLTH_TOT_DEB_ND_2015",
    "WLTH_TOT_DEB_ND_2017",
    "WLTH_TOT_DEB_ND_2019",
    "WLTH_TOT_DEB_ND_2021",
    "WLTH_TOT_DEB_NDF_1984",
    "WLTH_TOT_DEB_NDF_1989",
    "WLTH_TOT_DEB_NDF_1994",
    "WLTH_TOT_DEB_NDF_1999",
    "WLTH_TOT_DEB_NDF_2001",
    "WLTH_TOT_DEB_NDF_2003",
    "WLTH_TOT_DEB_NDF_2005",
    "WLTH_TOT_DEB_NDF_2007",
    "WLTH_TOT_DEB_NDF_2009",
    "WLTH_TOT_DEB_NDF_2011",
    "WLTH_TOT_DEB_NDF_2013",
    "WLTH_TOT_DEB_NDF_2015",
    "WLTH_TOT_DEB_NDF_2017",
    "WLTH_TOT_DEB_NDF_2019",
    "WLTH_TOT_DEB_NDF_2021",
    "WLTH_TOT_DEB_RD_1984",
    "WLTH_TOT_DEB_RD_1989",
    "WLTH_TOT_DEB_RD_1994",
    "WLTH_TOT_DEB_RD_1999",
    "WLTH_TOT_DEB_RD_2001",
    "WLTH_TOT_DEB_RD_2003",
    "WLTH_TOT_DEB_RD_2005",
    "WLTH_TOT_DEB_RD_2007",
    "WLTH_TOT_DEB_RD_2009",
    "WLTH_TOT_DEB_RD_2011",
    "WLTH_TOT_DEB_RD_2013",
    "WLTH_TOT_DEB_RD_2015",
    "WLTH_TOT_DEB_RD_2017",
    "WLTH_TOT_DEB_RD_2019",
    "WLTH_TOT_DEB_RD_2021",
    "WLTH_TOT_DEB_RDF_1984",
    "WLTH_TOT_DEB_RDF_1989",
    "WLTH_TOT_DEB_RDF_1994",
    "WLTH_TOT_DEB_RDF_1999",
    "WLTH_TOT_DEB_RDF_2001",
    "WLTH_TOT_DEB_RDF_2003",
    "WLTH_TOT_DEB_RDF_2005",
    "WLTH_TOT_DEB_RDF_2007",
    "WLTH_TOT_DEB_RDF_2009",
    "WLTH_TOT_DEB_RDF_2011",
    "WLTH_TOT_DEB_RDF_2013",
    "WLTH_TOT_DEB_RDF_2015",
    "WLTH_TOT_DEB_RDF_2017",
    "WLTH_TOT_DEB_RDF_2019",
    "WLTH_TOT_DEB_RDF_2021",
    "WLTH_TOT_DEB_XH_ND_1984",
    "WLTH_TOT_DEB_XH_ND_1989",
    "WLTH_TOT_DEB_XH_ND_1994",
    "WLTH_TOT_DEB_XH_ND_1999",
    "WLTH_TOT_DEB_XH_ND_2001",
    "WLTH_TOT_DEB_XH_ND_2003",
    "WLTH_TOT_DEB_XH_ND_2005",
    "WLTH_TOT_DEB_XH_ND_2007",
    "WLTH_TOT_DEB_XH_ND_2009",
    "WLTH_TOT_DEB_XH_ND_2011",
    "WLTH_TOT_DEB_XH_ND_2013",
    "WLTH_TOT_DEB_XH_ND_2015",
    "WLTH_TOT_DEB_XH_ND_2017",
    "WLTH_TOT_DEB_XH_ND_2019",
    "WLTH_TOT_DEB_XH_ND_2021",
    "WLTH_TOT_DEB_XH_NDF_1984",
    "WLTH_TOT_DEB_XH_NDF_1989",
    "WLTH_TOT_DEB_XH_NDF_1994",
    "WLTH_TOT_DEB_XH_NDF_1999",
    "WLTH_TOT_DEB_XH_NDF_2001",
    "WLTH_TOT_DEB_XH_NDF_2003",
    "WLTH_TOT_DEB_XH_NDF_2005",
    "WLTH_TOT_DEB_XH_NDF_2007",
    "WLTH_TOT_DEB_XH_NDF_2009",
    "WLTH_TOT_DEB_XH_NDF_2011",
    "WLTH_TOT_DEB_XH_NDF_2013",
    "WLTH_TOT_DEB_XH_NDF_2015",
    "WLTH_TOT_DEB_XH_NDF_2017",
    "WLTH_TOT_DEB_XH_NDF_2019",
    "WLTH_TOT_DEB_XH_NDF_2021",
    "WLTH_TOT_DEB_XH_RD_1984",
    "WLTH_TOT_DEB_XH_RD_1989",
    "WLTH_TOT_DEB_XH_RD_1994",
    "WLTH_TOT_DEB_XH_RD_1999",
    "WLTH_TOT_DEB_XH_RD_2001",
    "WLTH_TOT_DEB_XH_RD_2003",
    "WLTH_TOT_DEB_XH_RD_2005",
    "WLTH_TOT_DEB_XH_RD_2007",
    "WLTH_TOT_DEB_XH_RD_2009",
    "WLTH_TOT_DEB_XH_RD_2011",
    "WLTH_TOT_DEB_XH_RD_2013",
    "WLTH_TOT_DEB_XH_RD_2015",
    "WLTH_TOT_DEB_XH_RD_2017",
    "WLTH_TOT_DEB_XH_RD_2019",
    "WLTH_TOT_DEB_XH_RD_2021",
    "WLTH_TOT_DEB_XH_RDF_1984",
    "WLTH_TOT_DEB_XH_RDF_1989",
    "WLTH_TOT_DEB_XH_RDF_1994",
    "WLTH_TOT_DEB_XH_RDF_1999",
    "WLTH_TOT_DEB_XH_RDF_2001",
    "WLTH_TOT_DEB_XH_RDF_2003",
    "WLTH_TOT_DEB_XH_RDF_2005",
    "WLTH_TOT_DEB_XH_RDF_2007",
    "WLTH_TOT_DEB_XH_RDF_2009",
    "WLTH_TOT_DEB_XH_RDF_2011",
    "WLTH_TOT_DEB_XH_RDF_2013",
    "WLTH_TOT_DEB_XH_RDF_2015",
    "WLTH_TOT_DEB_XH_RDF_2017",
    "WLTH_TOT_DEB_XH_RDF_2019",
    "WLTH_TOT_DEB_XH_RDF_2021",
    "WLTH_TOT_DEB_XHR_ND_1984",
    "WLTH_TOT_DEB_XHR_ND_1989",
    "WLTH_TOT_DEB_XHR_ND_1994",
    "WLTH_TOT_DEB_XHR_ND_1999",
    "WLTH_TOT_DEB_XHR_ND_2001",
    "WLTH_TOT_DEB_XHR_ND_2003",
    "WLTH_TOT_DEB_XHR_ND_2005",
    "WLTH_TOT_DEB_XHR_ND_2007",
    "WLTH_TOT_DEB_XHR_ND_2009",
    "WLTH_TOT_DEB_XHR_ND_2011",
    "WLTH_TOT_DEB_XHR_ND_2013",
    "WLTH_TOT_DEB_XHR_ND_2015",
    "WLTH_TOT_DEB_XHR_ND_2017",
    "WLTH_TOT_DEB_XHR_ND_2019",
    "WLTH_TOT_DEB_XHR_ND_2021",
    "WLTH_TOT_DEB_XHR_NDF_1984",
    "WLTH_TOT_DEB_XHR_NDF_1989",
    "WLTH_TOT_DEB_XHR_NDF_1994",
    "WLTH_TOT_DEB_XHR_NDF_1999",
    "WLTH_TOT_DEB_XHR_NDF_2001",
    "WLTH_TOT_DEB_XHR_NDF_2003",
    "WLTH_TOT_DEB_XHR_NDF_2005",
    "WLTH_TOT_DEB_XHR_NDF_2007",
    "WLTH_TOT_DEB_XHR_NDF_2009",
    "WLTH_TOT_DEB_XHR_NDF_2011",
    "WLTH_TOT_DEB_XHR_NDF_2013",
    "WLTH_TOT_DEB_XHR_NDF_2015",
    "WLTH_TOT_DEB_XHR_NDF_2017",
    "WLTH_TOT_DEB_XHR_NDF_2019",
    "WLTH_TOT_DEB_XHR_NDF_2021",
    "WLTH_TOT_DEB_XHR_RD_1984",
    "WLTH_TOT_DEB_XHR_RD_1989",
    "WLTH_TOT_DEB_XHR_RD_1994",
    "WLTH_TOT_DEB_XHR_RD_1999",
    "WLTH_TOT_DEB_XHR_RD_2001",
    "WLTH_TOT_DEB_XHR_RD_2003",
    "WLTH_TOT_DEB_XHR_RD_2005",
    "WLTH_TOT_DEB_XHR_RD_2007",
    "WLTH_TOT_DEB_XHR_RD_2009",
    "WLTH_TOT_DEB_XHR_RD_2011",
    "WLTH_TOT_DEB_XHR_RD_2013",
    "WLTH_TOT_DEB_XHR_RD_2015",
    "WLTH_TOT_DEB_XHR_RD_2017",
    "WLTH_TOT_DEB_XHR_RD_2019",
    "WLTH_TOT_DEB_XHR_RD_2021",
    "WLTH_TOT_DEB_XHR_RDF_1984",
    "WLTH_TOT_DEB_XHR_RDF_1989",
    "WLTH_TOT_DEB_XHR_RDF_1994",
    "WLTH_TOT_DEB_XHR_RDF_1999",
    "WLTH_TOT_DEB_XHR_RDF_2001",
    "WLTH_TOT_DEB_XHR_RDF_2003",
    "WLTH_TOT_DEB_XHR_RDF_2005",
    "WLTH_TOT_DEB_XHR_RDF_2007",
    "WLTH_TOT_DEB_XHR_RDF_2009",
    "WLTH_TOT_DEB_XHR_RDF_2011",
    "WLTH_TOT_DEB_XHR_RDF_2013",
    "WLTH_TOT_DEB_XHR_RDF_2015",
    "WLTH_TOT_DEB_XHR_RDF_2017",
    "WLTH_TOT_DEB_XHR_RDF_2019",
    "WLTH_TOT_DEB_XHR_RDF_2021"
  ],
  "wealth_totals_net": [
    "ID",
    "LINEAGE",
    "PNUM",
    "WLTH_TOT_NET_ND_1984",
    "WLTH_TOT_NET_ND_1989",
    "WLTH_TOT_NET_ND_1994",
    "WLTH_TOT_NET_ND_1999",
    "WLTH_TOT_NET_ND_2001",
    "WLTH_TOT_NET_ND_2003",
    "WLTH_TOT_NET_ND_2005",
    "WLTH_TOT_NET_ND_2007",
    "WLTH_TOT_NET_ND_2009",
    "WLTH_TOT_NET_ND_2011",
    "WLTH_TOT_NET_ND_2013",
    "WLTH_TOT_NET_ND_2015",
    "WLTH_TOT_NET_ND_2017",
    "WLTH_TOT_NET_ND_2019",
    "WLTH_TOT_NET_ND_2021",
    "WLTH_TOT_NET_NDF_1984",
    "WLTH_TOT_NET_NDF_1989",
    "WLTH_TOT_NET_NDF_1994",
    "WLTH_TOT_NET_NDF_1999",
    "WLTH_TOT_NET_NDF_2001",
    "WLTH_TOT_NET_NDF_2003",
    "WLTH_TOT_NET_NDF_2005",
    "WLTH_TOT_NET_NDF_2007",
    "WLTH_TOT_NET_NDF_2009",
    "WLTH_TOT_NET_NDF_2011",
    "WLTH_TOT_NET_NDF_2013",
    "WLTH_TOT_NET_NDF_2015",
    "WLTH_TOT_NET_NDF_2017",
    "WLTH_TOT_NET_NDF_2019",
    "WLTH_TOT_NET_NDF_2021",
    "WLTH_TOT_NET_RD_1984",
    "WLTH_TOT_NET_RD_1989",
    "WLTH_TOT_NET_RD_1994",
    "WLTH_TOT_NET_RD_1999",
    "WLTH_TOT_NET_RD_2001",
    "WLTH_TOT_NET_RD_2003",
    "WLTH_TOT_NET_RD_2005",
    "WLTH_TOT_NET_RD_2007",
    "WLTH_TOT_NET_RD_2009",
    "WLTH_TOT_NET_RD_2011",
    "WLTH_TOT_NET_RD_2013",
    "WLTH_TOT_NET_RD_2015",
    "WLTH_TOT_NET_RD_2017",
    "WLTH_TOT_NET_RD_2019",
    "WLTH_TOT_NET_RD_2021",
    "WLTH_TOT_NET_RDF_1984",
    "WLTH_TOT_NET_RDF_1989",
    "WLTH_TOT_NET_RDF_1994",
    "WLTH_TOT_NET_RDF_1999",
    "WLTH_TOT_NET_RDF_2001",
    "WLTH_TOT_NET_RDF_2003",
    "WLTH_TOT_NET_RDF_2005",
    "WLTH_TOT_NET_RDF_2007",
    "WLTH_TOT_NET_RDF_2009",
    "WLTH_TOT_NET_RDF_2011",
    "WLTH_TOT_NET_RDF_2013",
    "WLTH_TOT_NET_RDF_2015",
    "WLTH_TOT_NET_RDF_2017",
    "WLTH_TOT_NET_RDF_2019",
    "WLTH_TOT_NET_RDF_2021",
    "WLTH_TOT_NET_XH_ND_1984",
    "WLTH_TOT_NET_XH_ND_1989",
    "WLTH_TOT_NET_XH_ND_1994",
    "WLTH_TOT_NET_XH_ND_1999",
    "WLTH_TOT_NET_XH_ND_2001",
    "WLTH_TOT_NET_XH_ND_2003",
    "WLTH_TOT_NET_XH_ND_2005",
    "WLTH_TOT_NET_XH_ND_2007",
    "WLTH_TOT_NET_XH_ND_2009",
    "WLTH_TOT_NET_XH_ND_2011",
    "WLTH_TOT_NET_XH_ND_2013",
    "WLTH_TOT_NET_XH_ND_2015",
    "WLTH_TOT_NET_XH_ND_2017",
    "WLTH_TOT_NET_XH_ND_2019",
    "WLTH_TOT_NET_XH_ND_2021",
    "WLTH_TOT_NET_XH_NDF_1984",
    "WLTH_TOT_NET_XH_NDF_1989",
    "WLTH_TOT_NET_XH_NDF_1994",
    "WLTH_TOT_NET_XH_NDF_1999",
    "WLTH_TOT_NET_XH_NDF_2001",
    "WLTH_TOT_NET_XH_NDF_2003",
    "WLTH_TOT_NET_XH_NDF_2005",
    "WLTH_TOT_NET_XH_NDF_2007",
    "WLTH_TOT_NET_XH_NDF_2009",
    "WLTH_TOT_NET_XH_NDF_2011",
    "WLTH_TOT_NET_XH_NDF_2013",
    "WLTH_TOT_NET_XH_NDF_2015",
    "WLTH_TOT_NET_XH_NDF_2017",
    "WLTH_TOT_NET_XH_NDF_2019",
    "WLTH_TOT_NET_XH_NDF_2021",
    "WLTH_TOT_NET_XH_RD_1984",
    "WLTH_TOT_NET_XH_RD_1989",
    "WLTH_TOT_NET_XH_RD_1994",
    "WLTH_TOT_NET_XH_RD_1999",
    "WLTH_TOT_NET_XH_RD_2001",
    "WLTH_TOT_NET_XH_RD_2003",
    "WLTH_TOT_NET_XH_RD_2005",
    "WLTH_TOT_NET_XH_RD_2007",
    "WLTH_TOT_NET_XH_RD_2009",
    "WLTH_TOT_NET_XH_RD_2011",
    "WLTH_TOT_NET_XH_RD_2013",
    "WLTH_TOT_NET_XH_RD_2015",
    "WLTH_TOT_NET_XH_RD_2017",
    "WLTH_TOT_NET_XH_RD_2019",
    "WLTH_TOT_NET_XH_RD_2021",
    "WLTH_TOT_NET_XH_RDF_1984",
    "WLTH_TOT_NET_XH_RDF_1989",
    "WLTH_TOT_NET_XH_RDF_1994",
    "WLTH_TOT_NET_XH_RDF_1999",
    "WLTH_TOT_NET_XH_RDF_2001",
    "WLTH_TOT_NET_XH_RDF_2003",
    "WLTH_TOT_NET_XH_RDF_2005",
    "WLTH_TOT_NET_XH_RDF_2007",
    "WLTH_TOT_NET_XH_RDF_2009",
    "WLTH_TOT_NET_XH_RDF_2011",
    "WLTH_TOT_NET_XH_RDF_2013",
    "WLTH_TOT_NET_XH_RDF_2015",
    "WLTH_TOT_NET_XH_RDF_2017",
    "WLTH_TOT_NET_XH_RDF_2019",
    "WLTH_TOT_NET_XH_RDF_2021",
    "WLTH_TOT_NET_XHR_ND_1984",
    "WLTH_TOT_NET_XHR_ND_1989",
    "WLTH_TOT_NET_XHR_ND_1994",
    "WLTH_TOT_NET_XHR_ND_1999",
    "WLTH_TOT_NET_XHR_ND_2001",
    "WLTH_TOT_NET_XHR_ND_2003",
    "WLTH_TOT_NET_XHR_ND_2005",
    "WLTH_TOT_NET_XHR_ND_2007",
    "WLTH_TOT_NET_XHR_ND_2009",
    "WLTH_TOT_NET_XHR_ND_2011",
    "WLTH_TOT_NET_XHR_ND_2013",
    "WLTH_TOT_NET_XHR_ND_2015",
    "WLTH_TOT_NET_XHR_ND_2017",
    "WLTH_TOT_NET_XHR_ND_2019",
    "WLTH_TOT_NET_XHR_ND_2021",
    "WLTH_TOT_NET_XHR_NDF_1984",
    "WLTH_TOT_NET_XHR_NDF_1989",
    "WLTH_TOT_NET_XHR_NDF_1994",
    "WLTH_TOT_NET_XHR_NDF_1999",
    "WLTH_TOT_NET_XHR_NDF_2001",
    "WLTH_TOT_NET_XHR_NDF_2003",
    "WLTH_TOT_NET_XHR_NDF_2005",
    "WLTH_TOT_NET_XHR_NDF_2007",
    "WLTH_TOT_NET_XHR_NDF_2009",
    "WLTH_TOT_NET_XHR_NDF_2011",
    "WLTH_TOT_NET_XHR_NDF_2013",
    "WLTH_TOT_NET_XHR_NDF_2015",
    "WLTH_TOT_NET_XHR_NDF_2017",
    "WLTH_TOT_NET_XHR_NDF_2019",
    "WLTH_TOT_NET_XHR_NDF_2021",
    "WLTH_TOT_NET_XHR_RD_1984",
    "WLTH_TOT_NET_XHR_RD_1989",
    "WLTH_TOT_NET_XHR_RD_1994",
    "WLTH_TOT_NET_XHR_RD_1999",
    "WLTH_TOT_NET_XHR_RD_2001",
    "WLTH_TOT_NET_XHR_RD_2003",
    "WLTH_TOT_NET_XHR_RD_2005",
    "WLTH_TOT_NET_XHR_RD_2007",
    "WLTH_TOT_NET_XHR_RD_2009",
    "WLTH_TOT_NET_XHR_RD_2011",
    "WLTH_TOT_NET_XHR_RD_2013",
    "WLTH_TOT_NET_XHR_RD_2015",
    "WLTH_TOT_NET_XHR_RD_2017",
    "WLTH_TOT_NET_XHR_RD_2019",
    "WLTH_TOT_NET_XHR_RD_2021",
    "WLTH_TOT_NET_XHR_RDF_1984",
    "WLTH_TOT_NET_XHR_RDF_1989",
    "WLTH_TOT_NET_XHR_RDF_1994",
    "WLTH_TOT_NET_XHR_RDF_1999",
    "WLTH_TOT_NET_XHR_RDF_2001",
    "WLTH_TOT_NET_XHR_RDF_2003",
    "WLTH_TOT_NET_XHR_RDF_2005",
    "WLTH_TOT_NET_XHR_RDF_2007",
    "WLTH_TOT_NET_XHR_RDF_2009",
    "WLTH_TOT_NET_XHR_RDF_2011",
    "WLTH_TOT_NET_XHR_RDF_2013",
    "WLTH_TOT_NET_XHR_RDF_2015",
    "WLTH_TOT_NET_XHR_RDF_2017",
    "WLTH_TOT_NET_XHR_RDF_2019",
    "WLTH_TOT_NET_XHR_RDF_2021"
  ],
  "wealth_vehicles": [
    "ID",
    "LINEAGE",
    "PNUM",
    "WLTH_VEHI_NET_ND_1984",
    "WLTH_VEHI_NET_ND_1989",
    "WLTH_VEHI_NET_ND_1994",
    "WLTH_VEHI_NET_ND_1999",
    "WLTH_VEHI_NET_ND_2001",
    "WLTH_VEHI_NET_ND_2003",
    "WLTH_VEHI_NET_ND_2005",
    "WLTH_VEHI_NET_ND_2007",
    "WLTH_VEHI_NET_ND_2009",
    "WLTH_VEHI_NET_ND_2011",
    "WLTH_VEHI_NET_ND_2013",
    "WLTH_VEHI_NET_ND_2015",
    "WLTH_VEHI_NET_ND_2017",
    "WLTH_VEHI_NET_ND_2019",
    "WLTH_VEHI_NET_ND_2021",
    "WLTH_VEHI_NET_NDF_1984",
    "WLTH_VEHI_NET_NDF_1989",
    "WLTH_VEHI_NET_NDF_1994",
    "WLTH_VEHI_NET_NDF_1999",
    "WLTH_VEHI_NET_NDF_2001",
    "WLTH_VEHI_NET_NDF_2003",
    "WLTH_VEHI_NET_NDF_2005",
    "WLTH_VEHI_NET_NDF_2007",
    "WLTH_VEHI_NET_NDF_2009",
    "WLTH_VEHI_NET_NDF_2011",
    "WLTH_VEHI_NET_NDF_2013",
    "WLTH_VEHI_NET_NDF_2015",
    "WLTH_VEHI_NET_NDF_2017",
    "WLTH_VEHI_NET_NDF_2019",
    "WLTH_VEHI_NET_NDF_2021",
    "WLTH_VEHI_NET_RD_1984",
    "WLTH_VEHI_NET_RD_1989",
    "WLTH_VEHI_NET_RD_1994",
    "WLTH_VEHI_NET_RD_1999",
    "WLTH_VEHI_NET_RD_2001",
    "WLTH_VEHI_NET_RD_2003",
    "WLTH_VEHI_NET_RD_2005",
    "WLTH_VEHI_NET_RD_2007",
    "WLTH_VEHI_NET_RD_2009",
    "WLTH_VEHI_NET_RD_2011",
    "WLTH_VEHI_NET_RD_2013",
    "WLTH_VEHI_NET_RD_2015",
    "WLTH_VEHI_NET_RD_2017",
    "WLTH_VEHI_NET_RD_2019",
    "WLTH_VEHI_NET_RD_2021",
    "WLTH_VEHI_NET_RDF_1984",
    "WLTH_VEHI_NET_RDF_1989",
    "WLTH_VEHI_NET_RDF_1994",
    "WLTH_VEHI_NET_RDF_1999",
    "WLTH_VEHI_NET_RDF_2001",
    "WLTH_VEHI_NET_RDF_2003",
    "WLTH_VEHI_NET_RDF_2005",
    "WLTH_VEHI_NET_RDF_2007",
    "WLTH_VEHI_NET_RDF_2009",
    "WLTH_VEHI_NET_RDF_2011",
    "WLTH_VEHI_NET_RDF_2013",
    "WLTH_VEHI_NET_RDF_2015",
    "WLTH_VEHI_NET_RDF_2017",
    "WLTH_VEHI_NET_RDF_2019",
    "WLTH_VEHI_NET_RDF_2021"
  ]
}

print(f"Loaded mapping for {len(TOPIC_COLUMNS)} topics")
total_unique = len(set(c for cols in TOPIC_COLUMNS.values() for c in cols))
print(f"Total unique columns across all topics: {total_unique:,}")


---

## Step 3: Load the .dta File

This takes 1–3 minutes depending on your machine — the wide file is ~2.4 GB and pandas reads Stata files iteratively. Coffee time.

In [ ]:
print(f"Reading {DTA_FILE} ... (this takes a few minutes)")
df = pd.read_stata(DTA_FILE, convert_categoricals=False)
print(f"\u2713 Loaded: {df.shape[0]:,} rows \u00d7 {df.shape[1]:,} columns")


---

## Step 4: Verify Expected Columns Are Present

Before splitting, sanity-check that the columns the topic mapping expects actually exist in the .dta file. If any are missing, you'll see them listed below — usually that means you have a different SHELF version than V2.

In [ ]:
dta_cols = set(df.columns)
expected = set(c for cols in TOPIC_COLUMNS.values() for c in cols)

missing = expected - dta_cols
extra = dta_cols - expected

print(f"Columns expected by the mapping: {len(expected):,}")
print(f"Columns present in the .dta:     {len(dta_cols):,}")
print(f"Missing (in mapping, not in dta): {len(missing)}")
print(f"Extra (in dta, not in mapping):   {len(extra)}")

if missing:
    print(f"\nFirst 20 missing columns:")
    for c in sorted(missing)[:20]:
        print(f"  - {c}")
    print(f"\n\u26a0 The split will skip missing columns in each topic.")
    print(f"  This is fine if a few are missing, suspicious if many are.")
else:
    print(f"\n\u2713 Every expected column is present.")


---

## Step 5: Write the 30 Topic CSVs

For each topic, slice out its columns from the loaded DataFrame and write the slice to its own CSV. Reports rows, columns, and file size for each.

In [ ]:
print(f"Writing {len(TOPIC_COLUMNS)} topic CSVs to {OUTPUT_DIR}/\n")

for topic, cols in TOPIC_COLUMNS.items():
    # Skip any columns that don't exist in this version of the .dta
    available = [c for c in cols if c in dta_cols]
    skipped = len(cols) - len(available)

    out_path = os.path.join(OUTPUT_DIR, f"{topic}.csv")
    df[available].to_csv(out_path, index=False)

    size_mb = os.path.getsize(out_path) / (1024**2)
    note = f" (skipped {skipped})" if skipped else ""
    print(f"  \u2713 {topic:30s} {len(available):>4} cols  {size_mb:>7.1f} MB{note}")

print(f"\n\u2713 Done. {len(TOPIC_COLUMNS)} files written to {OUTPUT_DIR}/")


---

## Next Steps

You now have 30 topic CSVs. To use them with the live SHELF app:

1. Clone the [psid-shelf-app repo](https://github.com/JF11579/psid-shelf-app)
2. Copy the contents of your `OUTPUT_DIR` next to `index.html`
3. Open `index.html` in a browser

The app will read your local CSVs in place of the 10-row previews baked into the public version.

## Notes on data handling

The SHELF dataset is governed by ICPSR's terms of use. You can use the CSVs you generate here for your own research and within institutions that have ICPSR access, but you should not redistribute them. The variable mapping in this notebook is structural metadata, not data — it's safe to share.

If you publish work using SHELF, cite:

> Pfeffer, F. T., Daumler, D., & Friedman, E. (2025). *PSID-SHELF, 1968–2021: The PSID's Social, Health, and Economic Longitudinal File (PSID-SHELF), Beta Release.* ICPSR. https://doi.org/10.3886/E194322V2
